# SpeakAI-Eval — Setup Environment on Kaggle
**Yêu cầu:** Thêm dataset `tnguynthnh142/speakai-models` vào Notebook.


In [ ]:
import os
TARGET_DIR = '/kaggle/working/SpeakAI-Eval'
os.makedirs(f'{TARGET_DIR}/configs', exist_ok=True)
os.makedirs(f'{TARGET_DIR}/data', exist_ok=True)
os.makedirs(f'{TARGET_DIR}/models', exist_ok=True)
os.makedirs(f'{TARGET_DIR}/infer', exist_ok=True)
os.makedirs(f'{TARGET_DIR}/classroom_lang', exist_ok=True)
os.makedirs(f'{TARGET_DIR}/speaker-diarize/speaker_diarize', exist_ok=True)
os.makedirs(f'{TARGET_DIR}/pretrained_models', exist_ok=True)
os.makedirs(f'{TARGET_DIR}/transformer_models', exist_ok=True)
os.makedirs(f'{TARGET_DIR}/hf_cache', exist_ok=True)
print('Directories created on Kaggle.')

---
### Tải Source Code (Offline)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/paths.py
"""Project root paths — import from anywhere after sys.path includes repo root."""

from __future__ import annotations

from pathlib import Path

ROOT = Path(__file__).resolve().parent
CONFIGS = ROOT / "configs"
PRONUNCIATION_CONFIG = CONFIGS / "pronunciation.yaml"

TRANSFORMER_MODELS = ROOT / "transformer_models"
SPEAKER_DIARIZE_DIR = ROOT / "speaker-diarize"
LOGS_DIR = ROOT / "logs"
WEB_UPLOADS = ROOT / "web" / "uploads"
TEACHER_REFERENCE_DIR = WEB_UPLOADS / "teacher_reference"


In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/env_config.py
"""Centralized secret and environment configuration loader for SpeakAI-Eval.

All Python scripts, backend API endpoints, and the frontend config sync script
read credentials from one source of truth:
  1. `<repo>/.env` (loaded into process environment without overriding real env vars)
  2. Process environment variables (`os.environ`)
  3. Kaggle Secrets (`kaggle_secrets.UserSecretsClient`) when running inside Kaggle

Private backend keys (e.g. `HF_TOKEN`, `SUPABASE_SERVICE_ROLE_KEY`, `GEMINI_API_KEY`) stay strictly
server-side and are NEVER exported to `web/.env.local`.
"""

from __future__ import annotations

import os
from pathlib import Path
from typing import Dict, Tuple

ROOT = Path(__file__).resolve().parent
DEFAULT_ENV_FILE = ROOT / ".env"

DEFAULT_GEMINI_ENDPOINT = "https://revidapi.com/v1/chat/completions"
DEFAULT_GEMINI_MODEL = "gemini-3.8-flash"

_ENV_LOADED = False


def _parse_env_file_fallback(path: Path, *, override: bool = False) -> None:
    """Minimal `.env` parser when `python-dotenv` is unavailable."""
    if not path.is_file():
        return
    for raw_line in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        if line.startswith("export "):
            line = line[7:].strip()
        key, value = line.split("=", 1)
        key = key.strip()
        if not key:
            continue
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in ("'", '"'):
            value = value[1:-1]
        if override or key not in os.environ:
            os.environ[key] = value


def load_project_env(env_path: str | Path | None = None, *, override: bool = False) -> bool:
    """Load environment variables from `.env` if present. Safe to call repeatedly."""
    global _ENV_LOADED
    path = Path(env_path) if env_path is not None else DEFAULT_ENV_FILE
    if _ENV_LOADED and env_path is None and not override:
        return path.is_file()

    if path.is_file():
        try:
            from dotenv import load_dotenv

            load_dotenv(dotenv_path=path, override=override)
        except ImportError:
            _parse_env_file_fallback(path, override=override)

    if env_path is None:
        _ENV_LOADED = True
    return path.is_file()


def get_secret(name: str, default: str = "", *, aliases: Tuple[str, ...] = ()) -> str:
    """Resolve a secret from `.env` / environment variables / Kaggle Secrets."""
    load_project_env()
    candidates = (name, *aliases)

    for key in candidates:
        val = (os.getenv(key) or "").strip()
        if val:
            return val

    try:
        from kaggle_secrets import UserSecretsClient

        client = UserSecretsClient()
        for key in candidates:
            try:
                val = (client.get_secret(key) or "").strip()
                if val:
                    return val
            except Exception:
                continue
    except Exception:
        pass

    return default


def get_supabase_config() -> Tuple[str, str]:
    """Return `(supabase_url, supabase_key)` with zero hardcoded fallbacks."""
    url = get_secret("SUPABASE_URL").rstrip("/")
    key = get_secret(
        "SUPABASE_ANON_KEY",
        aliases=("SUPABASE_KEY", "SUPABASE_SERVICE_ROLE_KEY"),
    )
    return url, key


def get_supabase_service_config() -> Tuple[str, str]:
    """Return `(supabase_url, supabase_service_role_key)` strictly for server-side callers."""
    url = get_secret("SUPABASE_URL").rstrip("/")
    service_key = get_secret("SUPABASE_SERVICE_ROLE_KEY")
    return url, service_key


def get_gemini_api_key() -> str:
    """Return optional server-side fallback Gemini/LLM API key."""
    return get_secret("GEMINI_API_KEY", aliases=("SPEAKAI_GEMINI_API_KEY",))


def get_hf_token() -> str:
    """Return optional HuggingFace token from centralized env/Kaggle secrets."""
    return get_secret("HF_TOKEN", aliases=("HUGGINGFACE_HUB_TOKEN", "HUGGINGFACE_TOKEN"))


def get_frontend_config() -> Dict[str, str]:
    """Return strictly public-safe settings for static frontend generation."""
    load_project_env()
    return {
        "SUPABASE_URL": get_secret("SUPABASE_URL").rstrip("/"),
        "SUPABASE_ANON_KEY": get_secret("SUPABASE_ANON_KEY", aliases=("SUPABASE_KEY",)),
        "DEFAULT_GEMINI_ENDPOINT": get_secret("DEFAULT_GEMINI_ENDPOINT", DEFAULT_GEMINI_ENDPOINT),
        "DEFAULT_GEMINI_MODEL": get_secret("DEFAULT_GEMINI_MODEL", DEFAULT_GEMINI_MODEL),
        "SPEAKAI_API_URL": get_secret("SPEAKAI_API_URL").rstrip("/"),
    }



In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/backend_api.py
"""FastAPI backend for SpeakAI pronunciation assessment.

When running on Kaggle: pipeline, extract_embedder, generate_turn_feedback,
generate_overall_summary are expected to be defined in the notebook's global
scope BEFORE this file is executed (%run or exec).

When running standalone: those globals must be defined or the relevant
endpoints will return a 503 error.
"""

from __future__ import annotations

import builtins
import hashlib
import json
import os
import re
import shutil
import subprocess
import threading
import time
import traceback
import uuid
from pathlib import Path
from typing import Optional

try:
    import numpy as np
except ImportError:  # pragma: no cover
    np = None  # type: ignore[assignment]
import requests
import uvicorn
from fastapi import BackgroundTasks, Depends, FastAPI, File, Form, Header, HTTPException, Request, UploadFile
from fastapi.encoders import jsonable_encoder
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from fastapi.staticfiles import StaticFiles

from env_config import (
    DEFAULT_GEMINI_ENDPOINT,
    DEFAULT_GEMINI_MODEL,
    get_gemini_api_key,
    get_supabase_config,
    get_supabase_service_config,
    load_project_env,
)

load_project_env()

AUDIO_TEMP_DIR = "/tmp/SpeakAI_Audio"
MAX_TASK_HISTORY = 100


# ── Cloudflare Tunnel ──────────────────────────────────────────────────
def start_cloudflare_tunnel(port: int = 8000, timeout_sec: float = 35.0) -> Optional[str]:
    """Start a Cloudflare Quick Tunnel, return the public URL, and drain stdout."""
    print("Starting Cloudflare Quick Tunnel...")
    cmd = ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"]
    try:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
    except FileNotFoundError:
        print("❌ cloudflared not found. Skipping tunnel.")
        return None

    url_holder: list[str] = []
    found_event = threading.Event()

    def _reader() -> None:
        stream = process.stdout
        if stream is None:
            found_event.set()
            return
        try:
            while True:
                line = stream.readline()
                if not line:
                    break
                if not url_holder:
                    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
                    if match:
                        url_holder.append(match.group(0))
                        found_event.set()
        except Exception:
            pass
        finally:
            found_event.set()
            try:
                stream.close()
            except Exception:
                pass

    reader_thread = threading.Thread(target=_reader, daemon=True)
    reader_thread.start()
    found_event.wait(timeout=timeout_sec)

    url = url_holder[0] if url_holder else None
    if url is None and process.poll() is None:
        try:
            process.terminate()
        except Exception:
            pass
    return url


def _update_supabase_url(public_url: str) -> None:
    """Push the tunnel URL to Supabase global_settings using Service Role Key (RLS-safe)."""
    supabase_url, service_key = get_supabase_service_config()
    if not service_key:
        supabase_url, fallback_key = get_supabase_config()
        if fallback_key:
            print("⚠️ Thiếu SUPABASE_SERVICE_ROLE_KEY — với RLS đang bật, hãy thêm SUPABASE_SERVICE_ROLE_KEY vào .env / Kaggle Secrets.")
        service_key = fallback_key

    if not supabase_url or not service_key:
        print("⚠️ Thiếu SUPABASE_URL hoặc SUPABASE_SERVICE_ROLE_KEY trong .env / Kaggle Secrets — bỏ qua cập nhật Supabase.")
        return

    try:
        r = requests.patch(
            f"{supabase_url}/rest/v1/global_settings?id=eq.1",
            headers={
                "apikey": service_key,
                "Authorization": f"Bearer {service_key}",
                "Content-Type": "application/json",
                "Prefer": "return=minimal",
            },
            json={"api_url": public_url},
            timeout=10,
        )
        if r.status_code in (200, 204):
            print("✅ Đã tự động cập nhật API URL lên Supabase!")
        else:
            print(f"❌ Lỗi cập nhật Supabase: {r.text}")
    except Exception as e:
        print(f"❌ Lỗi cập nhật Supabase: {e}")


# ── Supabase JWT Verification & RBAC Cache ─────────────────────────────
AUTH_CACHE_TTL_SEC = 60.0
_AUTH_CACHE: dict[str, tuple[float, dict]] = {}
_auth_cache_lock = threading.Lock()


def clear_auth_cache() -> None:
    """Clear verified JWT cache (useful for tests or key rotation)."""
    with _auth_cache_lock:
        _AUTH_CACHE.clear()


def _verify_supabase_jwt(token: str) -> dict:
    """Verify Supabase access token and load caller's role + status from profiles."""
    token_hash = hashlib.sha256(token.encode("utf-8")).hexdigest()
    now = time.time()
    with _auth_cache_lock:
        cached = _AUTH_CACHE.get(token_hash)
        if cached and (now - cached[0]) < AUTH_CACHE_TTL_SEC:
            return dict(cached[1])

    supabase_url, supabase_anon_key = get_supabase_config()
    svc_url, service_key = get_supabase_service_config()
    base_url = svc_url or supabase_url
    if not base_url or not (supabase_anon_key or service_key):
        raise HTTPException(
            status_code=500,
            detail="Thiếu cấu hình SUPABASE_URL hoặc SUPABASE_ANON_KEY trên backend.",
        )

    auth_apikey = supabase_anon_key or service_key
    try:
        user_resp = requests.get(
            f"{base_url}/auth/v1/user",
            headers={
                "apikey": auth_apikey,
                "Authorization": f"Bearer {token}",
            },
            timeout=10,
        )
    except Exception as exc:
        raise HTTPException(status_code=502, detail=f"Không thể kết nối Supabase Auth: {exc}")

    if user_resp.status_code != 200:
        raise HTTPException(status_code=401, detail="Phiên đăng nhập không hợp lệ hoặc đã hết hạn.")

    try:
        user_data = user_resp.json()
    except ValueError:
        user_data = {}

    uid = str(user_data.get("id") or "").strip()
    if not uid:
        raise HTTPException(status_code=401, detail="Không xác định được danh tính tài khoản.")

    prof_apikey = service_key or auth_apikey
    prof_bearer = service_key or token
    try:
        prof_resp = requests.get(
            f"{base_url}/rest/v1/profiles",
            params={
                "id": f"eq.{uid}",
                "select": "id,role,status,email,voice_embeddings",
                "limit": "1",
            },
            headers={
                "apikey": prof_apikey,
                "Authorization": f"Bearer {prof_bearer}",
            },
            timeout=10,
        )
    except Exception as exc:
        raise HTTPException(status_code=502, detail=f"Lỗi truy vấn hồ sơ phân quyền: {exc}")

    if prof_resp.status_code != 200:
        raise HTTPException(status_code=403, detail="Không thể xác minh phân quyền tài khoản.")

    try:
        rows = prof_resp.json()
    except ValueError:
        rows = []

    profile = rows[0] if isinstance(rows, list) and rows else None
    if not profile:
        raise HTTPException(status_code=403, detail="Tài khoản chưa có hồ sơ phân quyền.")

    status = str(profile.get("status") or "active").strip().lower()
    if status != "active":
        msg = (
            "Tài khoản Giáo viên đang chờ Quản trị viên phê duyệt."
            if status == "pending"
            else "Tài khoản của bạn hiện đang bị tạm khóa."
        )
        raise HTTPException(status_code=403, detail=msg)

    verified = {
        "id": uid,
        "role": str(profile.get("role") or "student").strip().lower(),
        "status": status,
        "email": profile.get("email") or user_data.get("email") or "",
        "voice_embeddings": profile.get("voice_embeddings"),
        "token": token,
    }
    with _auth_cache_lock:
        _AUTH_CACHE[token_hash] = (now, dict(verified))
    return verified


def require_active_user(authorization: Optional[str] = Header(None)) -> dict:
    """FastAPI dependency requiring an active, authenticated Supabase user."""
    if not authorization or not authorization.lower().startswith("bearer "):
        raise HTTPException(status_code=401, detail="Thiếu Authorization Bearer JWT.")
    token = authorization.split(" ", 1)[1].strip()
    if not token:
        raise HTTPException(status_code=401, detail="Token xác thực không hợp lệ.")
    return _verify_supabase_jwt(token)


# ── Resolve globals from notebook context ──────────────────────────────
def _get_global(name: str):
    """Resolve a global defined in the notebook context (builtins or globals)."""
    return getattr(builtins, name, None) or globals().get(name)


# ── Audio & Task Helpers ───────────────────────────────────────────────
def _safe_filename(filename: Optional[str]) -> str:
    base = Path(filename or "audio.webm").name
    cleaned = re.sub(r"[^A-Za-z0-9._-]", "_", base)
    return cleaned or "audio.webm"


def _convert_to_wav_16k(src_path: str | Path, dst_path: str | Path) -> None:
    """Convert uploaded audio safely to 16 kHz mono WAV via ffmpeg."""
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(src_path), "-ar", "16000", "-ac", "1", str(dst_path)],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )


def _prune_old_tasks() -> None:
    """Keep task registry bounded by evicting oldest completed/error entries."""
    with _tasks_lock:
        if len(tasks) <= MAX_TASK_HISTORY:
            return
        finished = [
            tid for tid, t in tasks.items() if t.get("status") in ("completed", "error")
        ]
        excess = len(tasks) - MAX_TASK_HISTORY
        for tid in finished[:excess]:
            tasks.pop(tid, None)


def _parse_bool_flag(val: object, *, default: bool = False) -> bool:
    s = str(val).strip().lower()
    if default:
        return s not in ("false", "0", "no", "none", "f")
    return s in ("true", "1", "yes", "t")


def _parse_embedding_json(emb_json: Optional[str]):
    emb_list = json.loads(emb_json or "[]")
    if not emb_list:
        return None
    if np is not None:
        emb = np.array(emb_list, dtype=np.float32)
        if emb.ndim == 2:
            emb = np.mean(emb, axis=0)
        emb /= np.linalg.norm(emb) + 1e-8
        return emb
    if isinstance(emb_list[0], list):
        cols = len(emb_list[0])
        vec = [sum(float(row[i]) for row in emb_list) / len(emb_list) for i in range(cols)]
    else:
        vec = [float(x) for x in emb_list]
    norm = sum(v * v for v in vec) ** 0.5 + 1e-8
    return [v / norm for v in vec]


# ── FastAPI App ────────────────────────────────────────────────────────
PUBLIC_URL = None
SERVER_START_TIME = time.time()

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

os.makedirs(AUDIO_TEMP_DIR, exist_ok=True)
app.mount("/audio", StaticFiles(directory=AUDIO_TEMP_DIR), name="audio")

tasks: dict[str, dict] = {}
_tasks_lock = threading.Lock()

# ── GPU Concurrency Control ────────────────────────────────────────────
_gpu_semaphore = threading.Semaphore(1)
_gpu_active_task: Optional[str] = None
_gpu_queue_count = 0
_classroom_embedder = None


def _resolve_classroom_analyzer():
    """Build or reuse ClassroomLanguageAnalyzer sharing existing GPU models when available."""
    custom = _get_global("classroom_analyzer")
    if custom is not None:
        return custom

    import sys
    import torch
    import infer.transcribe as transcribe_mod
    from classroom_lang.analyzer import ClassroomLanguageAnalyzer
    from paths import SPEAKER_DIARIZE_DIR

    global _classroom_embedder
    pipe = _get_global("pipeline")
    embedder = (
        _get_global("extract_embedder")
        or getattr(getattr(pipe, "diarizer", None), "embedder", None)
        or _classroom_embedder
    )
    if embedder is None:
        if SPEAKER_DIARIZE_DIR.is_dir() and str(SPEAKER_DIARIZE_DIR) not in sys.path:
            sys.path.insert(0, str(SPEAKER_DIARIZE_DIR))
        from speaker_diarize.embedding import ERes2NetEmbedder

        dev = "cuda:0" if torch.cuda.is_available() else "cpu"
        _classroom_embedder = ERes2NetEmbedder(device=dev)
        embedder = _classroom_embedder

    transcriber = transcribe_mod._transcriber
    if transcriber is None:
        base_asr = dict(getattr(pipe, "_asr_cfg", None) or {})
        base_asr.setdefault("model_name", "openai/whisper-large-v3")
        base_asr["language"] = "auto"
        base_asr["allowed_languages"] = ["en", "vi"]
        transcriber = transcribe_mod.get_transcriber(asr_config=base_asr)

    return ClassroomLanguageAnalyzer(embedder=embedder, transcriber=transcriber)


def _enqueue_assessment(
    background_tasks: BackgroundTasks,
    audio: UploadFile,
    *,
    owner_id: Optional[str] = None,
    teacher_embeddings_json: str = "[]",
    student_embeddings_json: str = "[]",
    score_teacher: bool = False,
    diarize: bool = True,
    reference_text: Optional[str] = None,
    task_type: Optional[str] = None,
) -> JSONResponse:
    task_id = str(uuid.uuid4())
    with _tasks_lock:
        tasks[task_id] = {
            "owner_id": owner_id,
            "status": "processing",
            "step": "Đang tải file âm thanh lên server...",
            "result": None,
            "llm_feedback": None,
        }
    _prune_old_tasks()

    safe_name = _safe_filename(audio.filename)
    raw_conv_path = os.path.join(AUDIO_TEMP_DIR, f"{task_id}_raw_{safe_name}")
    conv_path = os.path.join(AUDIO_TEMP_DIR, f"{task_id}_converted.wav")
    try:
        with open(raw_conv_path, "wb") as f:
            shutil.copyfileobj(audio.file, f)
        _convert_to_wav_16k(raw_conv_path, conv_path)
    except Exception:
        with _tasks_lock:
            tasks.pop(task_id, None)
        raise
    finally:
        if os.path.exists(raw_conv_path):
            try:
                os.remove(raw_conv_path)
            except OSError:
                pass

    background_tasks.add_task(
        process_assessment,
        task_id,
        conv_path,
        teacher_embeddings_json,
        student_embeddings_json,
        score_teacher,
        diarize,
        reference_text,
        task_type,
    )
    return JSONResponse({"success": True, "task_id": task_id})


@app.post("/extract_embedding")
def extract_embedding_api(
    audio: UploadFile = File(...),
    user: dict = Depends(require_active_user),
):
    raw_audio_path = None
    audio_path = None
    try:
        embedder = _get_global("extract_embedder")
        if embedder is None:
            return JSONResponse(
                {"success": False, "error": "extract_embedder not initialized"},
                status_code=503,
            )

        temp_id = str(uuid.uuid4())
        safe_name = _safe_filename(audio.filename)
        raw_audio_path = os.path.join(AUDIO_TEMP_DIR, f"{temp_id}_raw_{safe_name}")
        audio_path = os.path.join(AUDIO_TEMP_DIR, f"{temp_id}_converted.wav")

        with open(raw_audio_path, "wb") as f:
            shutil.copyfileobj(audio.file, f)
        _convert_to_wav_16k(raw_audio_path, audio_path)

        from speaker_diarize.audio_io import load_audio
        from speaker_diarize.denoise import denoise_with_deepfilternet, level_audio_to_target

        waveform, sr = load_audio(audio_path)
        waveform, sr = denoise_with_deepfilternet(waveform, sr)
        waveform = level_audio_to_target(waveform, sr)
        emb = embedder.embed(waveform, sr)

        return JSONResponse({"success": True, "embedding": emb.tolist()})
    except Exception as e:
        traceback.print_exc()
        return JSONResponse({"success": False, "error": str(e)}, status_code=500)
    finally:
        for p in (raw_audio_path, audio_path):
            if p and os.path.exists(p):
                try:
                    os.remove(p)
                except OSError:
                    pass


@app.post("/assess_start")
def assess_start_api(
    background_tasks: BackgroundTasks,
    audio: UploadFile = File(...),
    teacher_embeddings_json: str = Form("[]"),
    student_embeddings_json: str = Form("[]"),
    score_teacher: bool = Form(False),
    diarize: str = Form("true"),
    reference_text: Optional[str] = Form(None),
    task_type: Optional[str] = Form(None),
    user: dict = Depends(require_active_user),
):
    is_diarize = _parse_bool_flag(diarize, default=True)
    if is_diarize and user.get("role") not in ("teacher", "admin"):
        return JSONResponse(
            {
                "success": False,
                "error": "Chỉ Giáo viên hoặc Quản trị viên mới được phép chấm điểm hội thoại 2 người (Diarization).",
            },
            status_code=403,
        )
    try:
        return _enqueue_assessment(
            background_tasks,
            audio,
            owner_id=user["id"],
            teacher_embeddings_json=teacher_embeddings_json,
            student_embeddings_json=student_embeddings_json,
            score_teacher=_parse_bool_flag(score_teacher, default=False),
            diarize=is_diarize,
            reference_text=reference_text,
            task_type=task_type,
        )
    except Exception as e:
        return JSONResponse({"success": False, "error": str(e)}, status_code=500)


@app.post("/assess_practice")
def assess_practice_api(
    background_tasks: BackgroundTasks,
    audio: UploadFile = File(...),
    reference_text: Optional[str] = Form(None),
    task_type: Optional[str] = Form(None),
    user: dict = Depends(require_active_user),
):
    try:
        return _enqueue_assessment(
            background_tasks,
            audio,
            owner_id=user["id"],
            diarize=False,
            score_teacher=False,
            reference_text=reference_text,
            task_type=task_type,
        )
    except Exception as e:
        return JSONResponse({"success": False, "error": str(e)}, status_code=500)


def _convert_paths_to_urls(node, public_url: str) -> None:
    audio_prefix = f"{AUDIO_TEMP_DIR.rstrip('/')}/"
    if isinstance(node, dict):
        for k, v in list(node.items()):
            if isinstance(v, (os.PathLike, Path)):
                v = str(v)
                node[k] = v
            if k in ("audio", "full_audio") and isinstance(v, str) and v.startswith(audio_prefix):
                rel_path = v[len(audio_prefix):]
                node[k] = f"{public_url}/audio/{rel_path}"
            else:
                _convert_paths_to_urls(v, public_url)
    elif isinstance(node, list):
        for i, item in enumerate(node):
            if isinstance(item, (os.PathLike, Path)):
                node[i] = str(item)
            _convert_paths_to_urls(node[i], public_url)


def process_assessment(
    task_id: str,
    conv_path: str,
    teacher_embeddings_json: str,
    student_embeddings_json: str,
    score_teacher: bool,
    diarize: bool = True,
    reference_text: Optional[str] = None,
    task_type: Optional[str] = None,
):
    global _gpu_active_task, _gpu_queue_count
    task_start_time = time.time()

    _gpu_queue_count += 1
    queue_pos = _gpu_queue_count
    if not _gpu_semaphore.acquire(blocking=False):
        tasks[task_id]["step"] = f"Đang chờ GPU (vị trí {queue_pos} trong hàng đợi)..."
        print(f"[Queue] Task {task_id[:8]} đang chờ GPU (vị trí {queue_pos})...")
        _gpu_semaphore.acquire()
    _gpu_active_task = task_id
    _gpu_queue_count = max(0, _gpu_queue_count - 1)
    gpu_wait_time = time.time() - task_start_time
    if gpu_wait_time > 0.5:
        print(f"[Queue] Task {task_id[:8]} đã chờ GPU {gpu_wait_time:.1f}s")

    try:
        pipe = _get_global("pipeline")
        if pipe is None:
            tasks[task_id]["status"] = "error"
            tasks[task_id]["error"] = "pipeline not initialized"
            return

        if not diarize:
            if task_type == "read_aloud" and reference_text:
                tasks[task_id]["step"] = "Đang chấm điểm Read Aloud trực tiếp bằng văn bản mẫu (Không dùng Whisper ASR)..."
            else:
                tasks[task_id]["step"] = "Đang phân tích phát âm trực tiếp (Single Speaker, không Diarization)..."
            raw_result = pipe.assess_single_speaker(
                conv_path,
                reference_text=reference_text,
                task_type=task_type,
            )
        else:
            tasks[task_id]["step"] = "Đang phân tích embeddings..."
            teacher_emb = _parse_embedding_json(teacher_embeddings_json)
            student_emb = _parse_embedding_json(student_embeddings_json)

            tasks[task_id]["step"] = "Đang tách lời (Diarization) & Phân tích phát âm (2 models)..."
            raw_result = pipe.assess_conversation(
                conv_path,
                teacher_embedding=teacher_emb,
                student_embedding=student_emb,
                score_teacher=score_teacher,
            )

        diar = raw_result.get("diarization", {})
        if raw_result.get("teacher") and diar.get("teacher"):
            raw_result["teacher"]["full_audio"] = str(diar["teacher"])
        if raw_result.get("student") and diar.get("student"):
            raw_result["student"]["full_audio"] = str(diar["student"])

        if PUBLIC_URL:
            _convert_paths_to_urls(raw_result, PUBLIC_URL)

        if diarize and os.path.exists(conv_path):
            os.remove(conv_path)

        total_time = time.time() - task_start_time
        tasks[task_id]["result"] = raw_result
        tasks[task_id]["llm_feedback"] = None
        tasks[task_id]["step"] = f"Hoàn tất phân tích âm học ({total_time:.1f}s)."
        tasks[task_id]["processing_time_sec"] = round(total_time, 2)
        tasks[task_id]["status"] = "completed"
        print(f"[Perf] Task {task_id[:8]} hoàn tất trong {total_time:.1f}s")
    except Exception as e:
        print(f"API Error in task {task_id}: {e}")
        traceback.print_exc()
        tasks[task_id]["status"] = "error"
        tasks[task_id]["error"] = str(e)
    finally:
        _gpu_active_task = None
        _gpu_semaphore.release()
        try:
            import torch

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass


@app.get("/assess_status/{task_id}")
def assess_status(task_id: str, user: dict = Depends(require_active_user)):
    with _tasks_lock:
        task = tasks.get(task_id)
    if task is None:
        return JSONResponse({"success": False, "error": "Task not found"}, status_code=404)
    owner_id = task.get("owner_id")
    if owner_id and owner_id != user["id"] and user.get("role") != "admin":
        return JSONResponse({"success": False, "error": "Bạn không có quyền xem tiến trình bài chấm của người khác."}, status_code=403)
    payload = {k: v for k, v in task.items() if k != "owner_id"}
    return JSONResponse({"success": True, "data": jsonable_encoder(payload)})


@app.post("/classroom_analyze_start")
def classroom_analyze_start_api(
    background_tasks: BackgroundTasks,
    audio: UploadFile = File(...),
    teacher_embeddings_json: str = Form("[]"),
    teacher_id: Optional[str] = Form(None),
    teacher_threshold: float = Form(0.40),
    vad_threshold_db: float = Form(-40.0),
    user: dict = Depends(require_active_user),
):
    """Analyze a full classroom recording: Teacher-only diarization + EN/VI word ratio (speech only)."""
    role = user.get("role")
    if role not in ("admin", "teacher"):
        return JSONResponse(
            {"success": False, "error": "Chỉ Giáo viên hoặc Quản trị viên mới được phép phân tích lớp học."},
            status_code=403,
        )

    if role == "teacher" and teacher_id and teacher_id.strip() != str(user["id"]):
        return JSONResponse(
            {"success": False, "error": "Giáo viên chỉ được phép phân tích lớp học bằng chính giọng của mình."},
            status_code=403,
        )

    raw_emb_source = teacher_embeddings_json
    if (not raw_emb_source or raw_emb_source.strip() in ("", "[]", "null")) and role == "teacher" and user.get("voice_embeddings"):
        raw_emb_source = json.dumps(user["voice_embeddings"])

    try:
        teacher_emb = _parse_embedding_json(raw_emb_source)
    except (json.JSONDecodeError, TypeError, ValueError) as exc:
        return JSONResponse(
            {"success": False, "error": f"teacher_embeddings_json không hợp lệ: {exc}"},
            status_code=400,
        )

    if teacher_emb is None:
        return JSONResponse(
            {
                "success": False,
                "error": "Thiếu mẫu giọng của Giáo viên (voice_embeddings). Vui lòng chọn giáo viên đã đăng ký giọng.",
            },
            status_code=400,
        )

    t_thresh = max(0.05, min(0.95, float(teacher_threshold)))
    v_thresh = max(-80.0, min(-5.0, float(vad_threshold_db)))

    task_id = str(uuid.uuid4())
    with _tasks_lock:
        tasks[task_id] = {
            "owner_id": user["id"],
            "status": "processing",
            "task_kind": "classroom_language",
            "step": "Đang tải file audio buổi học lên server...",
            "result": None,
        }
    _prune_old_tasks()

    safe_name = _safe_filename(audio.filename)
    raw_conv_path = os.path.join(AUDIO_TEMP_DIR, f"{task_id}_cls_raw_{safe_name}")
    conv_path = os.path.join(AUDIO_TEMP_DIR, f"{task_id}_cls_converted.wav")
    try:
        with open(raw_conv_path, "wb") as f:
            shutil.copyfileobj(audio.file, f)
        _convert_to_wav_16k(raw_conv_path, conv_path)
    except Exception as e:
        with _tasks_lock:
            tasks.pop(task_id, None)
        if os.path.exists(conv_path):
            try:
                os.remove(conv_path)
            except OSError:
                pass
        return JSONResponse({"success": False, "error": str(e)}, status_code=500)
    finally:
        if os.path.exists(raw_conv_path):
            try:
                os.remove(raw_conv_path)
            except OSError:
                pass

    background_tasks.add_task(
        process_classroom_analysis,
        task_id,
        conv_path,
        teacher_emb,
        t_thresh,
        v_thresh,
    )
    return JSONResponse({"success": True, "task_id": task_id})


def process_classroom_analysis(
    task_id: str,
    conv_path: str,
    teacher_emb: np.ndarray,
    teacher_threshold: float,
    vad_threshold_db: float,
):
    global _gpu_active_task, _gpu_queue_count
    task_start_time = time.time()

    _gpu_queue_count += 1
    queue_pos = _gpu_queue_count
    if not _gpu_semaphore.acquire(blocking=False):
        tasks[task_id]["step"] = f"Đang chờ GPU (vị trí {queue_pos} trong hàng đợi)..."
        _gpu_semaphore.acquire()
    _gpu_active_task = task_id
    _gpu_queue_count = max(0, _gpu_queue_count - 1)

    try:
        analyzer = _resolve_classroom_analyzer()
        result = analyzer.analyze(
            conv_path,
            teacher_emb,
            teacher_threshold=teacher_threshold,
            vad_threshold_db=vad_threshold_db,
            progress_callback=lambda msg: tasks[task_id].update({"step": msg}),
        )
        total_time = time.time() - task_start_time
        tasks[task_id]["result"] = result
        tasks[task_id]["step"] = f"Hoàn tất phân tích ngôn ngữ buổi học ({total_time:.1f}s)."
        tasks[task_id]["processing_time_sec"] = round(total_time, 2)
        tasks[task_id]["status"] = "completed"
        print(f"[Classroom] Task {task_id[:8]} hoàn tất trong {total_time:.1f}s")
    except Exception as e:
        print(f"Classroom API Error in task {task_id}: {e}")
        traceback.print_exc()
        tasks[task_id]["status"] = "error"
        tasks[task_id]["error"] = str(e)
    finally:
        if os.path.exists(conv_path):
            try:
                os.remove(conv_path)
            except OSError:
                pass
        _gpu_active_task = None
        _gpu_semaphore.release()
        try:
            import torch

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass


@app.get("/classroom_status/{task_id}")
def classroom_status(task_id: str, user: dict = Depends(require_active_user)):
    if user.get("role") not in ("admin", "teacher"):
        return JSONResponse({"success": False, "error": "Không có quyền truy cập tác vụ phân tích lớp học."}, status_code=403)
    with _tasks_lock:
        task = tasks.get(task_id)
    if task is None:
        return JSONResponse({"success": False, "error": "Task not found"}, status_code=404)
    owner_id = task.get("owner_id")
    if owner_id and owner_id != user["id"] and user.get("role") != "admin":
        return JSONResponse({"success": False, "error": "Bạn không có quyền xem kết quả phân tích của giáo viên khác."}, status_code=403)
    payload = {k: v for k, v in task.items() if k != "owner_id"}
    return JSONResponse({"success": True, "data": jsonable_encoder(payload)})


@app.post("/ai_eval")
async def ai_eval_proxy_api(request: Request, user: dict = Depends(require_active_user)):
    """Fallback server-side AI Evaluator proxy matching `/api/ai-eval` contract."""
    try:
        body = await request.json()
    except Exception:
        return JSONResponse({"success": False, "error": "Payload JSON không hợp lệ."}, status_code=400)

    action = str((body or {}).get("action") or "").strip()
    if action not in ("ping_test", "eval_answer", "eval_conversation"):
        return JSONResponse({"success": False, "error": f"Action không hợp lệ: {action}"}, status_code=400)

    role = user.get("role")
    if action == "ping_test" and role != "admin":
        return JSONResponse({"success": False, "error": "Chỉ Quản trị viên mới được kiểm tra kết nối AI."}, status_code=403)
    if action == "eval_conversation" and role not in ("teacher", "admin"):
        return JSONResponse({"success": False, "error": "Chỉ Giáo viên hoặc Quản trị viên mới được đánh giá hội thoại 1-1."}, status_code=403)

    supabase_url, anon_key = get_supabase_config()
    svc_url, service_key = get_supabase_service_config()
    base_url = svc_url or supabase_url
    gs_row = None

    if base_url and (service_key or (role == "admin" and user.get("token"))):
        hdr_apikey = service_key or anon_key
        hdr_bearer = service_key or user.get("token")
        try:
            gs_resp = requests.get(
                f"{base_url}/rest/v1/global_settings",
                params={"id": "eq.1", "select": "gemini_api_key,gemini_api_url,gemini_model", "limit": "1"},
                headers={"apikey": hdr_apikey, "Authorization": f"Bearer {hdr_bearer}"},
                timeout=10,
            )
            if gs_resp.status_code == 200:
                rows = gs_resp.json()
                gs_row = rows[0] if isinstance(rows, list) and rows else None
        except Exception:
            gs_row = None

    endpoint = str((gs_row or {}).get("gemini_api_url") or DEFAULT_GEMINI_ENDPOINT).strip()
    model = str((gs_row or {}).get("gemini_model") or DEFAULT_GEMINI_MODEL).strip()
    stored_key = str((gs_row or {}).get("gemini_api_key") or get_gemini_api_key() or "").strip()

    if action == "ping_test":
        draft = (body or {}).get("draftConfig")
        if not isinstance(draft, dict):
            draft = {}
        test_key = str(draft.get("apiKey") or stored_key or "").strip()
        test_url = str(draft.get("apiUrl") or endpoint).strip()
        test_model = str(draft.get("model") or model).strip()
        if not test_key:
            return JSONResponse({"success": False, "error": "Chưa có AI API Key để kiểm tra."}, status_code=400)
        try:
            ai_resp = requests.post(
                test_url,
                headers={
                    "Content-Type": "application/json",
                    "Authorization": f"Bearer {test_key}",
                    "x-api-key": test_key,
                },
                json={
                    "model": test_model,
                    "messages": [{"role": "user", "content": 'Ping test. Reply with "OK".'}],
                    "max_tokens": 10,
                },
                timeout=20,
            )
            if not ai_resp.ok:
                return JSONResponse(
                    {"success": False, "error": f"API Lỗi HTTP {ai_resp.status_code}: {ai_resp.text[:200]}"},
                    status_code=502,
                )
            data = ai_resp.json()
            reply = str((((data.get("choices") or [{}])[0].get("message") or {}).get("content")) or "").strip()
            return JSONResponse({"success": True, "model": data.get("model") or test_model, "reply": reply})
        except Exception as exc:
            return JSONResponse({"success": False, "error": f"Không kết nối được AI API: {exc}"}, status_code=502)

    if not stored_key:
        return JSONResponse(
            {"success": False, "code": "AI_KEY_MISSING", "error": "Chưa cấu hình AI API Key trên hệ thống."},
            status_code=503,
        )

    raw_messages = (body or {}).get("messages")
    if not isinstance(raw_messages, list) or not (1 <= len(raw_messages) <= 2):
        return JSONResponse({"success": False, "error": "Danh sách messages không hợp lệ."}, status_code=400)
    safe_messages = []
    for msg in raw_messages:
        if not isinstance(msg, dict) or msg.get("role") not in ("system", "user") or msg.get("content") is None:
            return JSONResponse({"success": False, "error": "Cấu trúc message không hợp lệ."}, status_code=400)
        safe_messages.append({"role": msg["role"], "content": msg["content"]})

    try:
        temp_val = float((body or {}).get("temperature", 0.2))
    except (TypeError, ValueError):
        temp_val = 0.2
    temp_val = max(0.0, min(1.0, temp_val))

    try:
        upstream_resp = requests.post(
            endpoint,
            headers={
                "Content-Type": "application/json",
                "Authorization": f"Bearer {stored_key}",
                "x-api-key": stored_key,
            },
            json={"model": model, "messages": safe_messages, "temperature": temp_val},
            timeout=30,
        )
        if not upstream_resp.ok:
            return JSONResponse(
                {"success": False, "error": f"Upstream AI API HTTP {upstream_resp.status_code}: {upstream_resp.text[:200]}"},
                status_code=502,
            )
        data = upstream_resp.json()
        content = str((((data.get("choices") or [{}])[0].get("message") or {}).get("content")) or "")
        return JSONResponse({"success": True, "content": content, "model": data.get("model") or model})
    except Exception as exc:
        return JSONResponse({"success": False, "error": f"Lỗi gọi Upstream AI API: {exc}"}, status_code=502)


@app.get("/health")
def health_check():
    """Server health check — monitoring uptime, GPU status, queue."""
    uptime_sec = time.time() - SERVER_START_TIME
    gpu_info: dict = {"available": False}
    try:
        import torch

        if torch.cuda.is_available():
            gpu_info = {
                "available": True,
                "device_name": torch.cuda.get_device_name(0),
                "memory_allocated_mb": round(torch.cuda.memory_allocated(0) / 1024 / 1024, 1),
                "memory_reserved_mb": round(torch.cuda.memory_reserved(0) / 1024 / 1024, 1),
                "memory_total_mb": round(torch.cuda.get_device_properties(0).total_memory / 1024 / 1024, 1),
            }
    except Exception:
        pass

    active_tasks = sum(1 for t in tasks.values() if t.get("status") == "processing")
    completed_tasks = sum(1 for t in tasks.values() if t.get("status") == "completed")
    error_tasks = sum(1 for t in tasks.values() if t.get("status") == "error")

    return JSONResponse({
        "status": "healthy",
        "uptime_sec": round(uptime_sec, 1),
        "uptime_human": f"{int(uptime_sec // 3600)}h {int((uptime_sec % 3600) // 60)}m",
        "gpu": gpu_info,
        "gpu_queue_waiting": _gpu_queue_count,
        "gpu_active_task": _gpu_active_task[:8] if _gpu_active_task else None,
        "tasks_active": active_tasks,
        "tasks_completed": completed_tasks,
        "tasks_error": error_tasks,
        "public_url": PUBLIC_URL,
    })


if __name__ == "__main__":
    import asyncio

    PUBLIC_URL = start_cloudflare_tunnel(8000)
    if PUBLIC_URL:
        _update_supabase_url(PUBLIC_URL)
        print("\n" + "=" * 80)
        print(f"🚀 API IS LIVE AT: {PUBLIC_URL}")
        print("=" * 80 + "\n")
    else:
        print("⚠️ Running without Cloudflare tunnel (localhost only)")

    is_in_notebook = False
    try:
        loop = asyncio.get_running_loop()
        if loop and loop.is_running():
            is_in_notebook = True
    except RuntimeError:
        is_in_notebook = False

    if is_in_notebook:
        print("⚡ Phát hiện môi trường Jupyter / Kaggle (active event loop).")
        print(f"🚀 Đang khởi chạy Uvicorn trong background thread trên cổng 8000 (Public: {PUBLIC_URL})...")
        config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, log_level="info")
        server = uvicorn.Server(config)
        server_thread = threading.Thread(target=server.run, daemon=True)
        server_thread.start()
        print("✅ FastAPI Server đang chạy! Nhấn nút Stop / Interrupt cell để dừng.")
        try:
            while server_thread.is_alive():
                time.sleep(1)
        except KeyboardInterrupt:
            print("\nĐang tắt FastAPI Server...")
            server.should_exit = True
            server_thread.join(timeout=5)
    else:
        uvicorn.run(app, host="0.0.0.0", port=8000)


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/configs', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/configs/pronunciation.yaml
# Pronunciation Assessment — Kaggle/Local Edition
project:
  seed: 42
  device: cuda

paths:
  transformer_models_dir: transformer_models
  pronunciation_checkpoint: transformer_models/pronunciation.pt
  checkpoint_dir: checkpoints
  log_dir: logs
  cmudict_path: null

audio_preprocess:
  sample_rate: 16000
  peak_normalize: true
  target_peak: 0.95
  highpass_hz: 80.0
  denoise: true
  denoise_prop: 0.75
  speech_normalize: true
  speech_thresh_db: -35.0
  vad_frame_ms: 25.0
  vad_hop_ms: 10.0

inference:
  device: cuda
  max_audio_duration_sec: 3600
  max_duration_sec: null
  max_upload_mb: 300

asr:
  engine: qwen3  # Qwen3-ASR (Qwen/Qwen3-ASR-1.7B-hf): chuẩn xác, triệt tiêu ảo giác, có chấm phẩy và chữ thường/hoa
  model_name: Qwen/Qwen3-ASR-1.7B-hf
  language: en
  device: cuda
  torch_dtype: float16
  max_new_tokens: 128
  lang_id:
    enabled: false
    drop_languages: [vi, vie, vietnamese]
    min_confidence: 0.45
    text_vi_regex: false

eligibility:
  min_duration_sec: 0.8
  min_words: 2
  backchannel_words:
    - 'yes'
    - 'no'
    - yeah
    - yep
    - ok
    - okay
    - sure
    - uh
    - um
    - ah
    - hmm
    - mmm
    - mm
    - erm
    - er
    - mhm
    - right
    - alright
    - oh
    - cool
    - thank you
    - thanks
    - bye
    - bye bye
    - hello
    - hi

wavlm:
  model_name: microsoft/wavlm-large
  freeze: true
  use_lora: true
  lora_r: 8
  lora_alpha: 16
  lora_target_modules: [q_proj, v_proj]

transformer:
  num_layers: 3
  num_heads: 8
  ff_dim: 2048
  dropout: 0.1
  max_seq_len: 800

ctc_align:
  use_wavlm_ctc_head: true

phoneme_graph:
  hidden_dim: 256
  num_gat_layers: 2
  num_heads: 4
  dropout: 0.1
  edge_types:
    sequential: true
    same_word: true
    same_syllable: false

multitask:
  hidden_dim: 256
  dropout: 0.1
  utterance_aspects: [accuracy, fluency, completeness, prosodic, total]
  word_aspects: [accuracy, stress, total]
  phoneme_aspects: [accuracy]
  score_scale: 5.0

scorer:
  weights:
    utterance_total: 0.5
    word_total: 0.25
    phoneme_accuracy: 0.25
  phoneme_low_threshold: 1.2

  # Hiệu chuẩn âm học cơ sở (WavLM-Large + LoRA + GATv2)
  calibration:
    enabled: true
    slope: 1.0
    offset: 0.0
    short_duration_sec: 3.5
    short_duration_factor: 0.20
    long_duration_sec: 4.0
    long_duration_factor: 0.05
    long_duration_max_boost: 0.4
    phone_slope: 6.0
    phone_offset: -3.0

  # Hiệu chuẩn điểm Phát âm (Pronunciation - PR) theo chuẩn khảo thí thực tế
  pronunciation_calibration:
    enabled: true
    w_acc: 0.8655
    w_flu: 0.0767
    w_pro: -0.3757
    bias: 3.1879
    min_score: 1.0
    max_score: 9.5

  # Tham số trừ điểm chẩn đoán âm vị L2-MDD (ASHA)
  l2_mdd_penalty:
    enabled: true
    pen_bad: 0.04
    pen_warning: 0.015
    max_penalty: 0.35

  # Barem quy đổi 4 trụ cột IELTS Speaking (FC, LR, GRA) từ điểm âm học thực nghiệm
  ielts_calibration:
    enabled: true
    fc:
      slope: 0.1890
      bias: 5.5574
    lr:
      slope: 0.2162
      bias: 5.4123
    gra:
      slope: 0.6774
      bias: 2.4684

sentence_split:
  min_silence_sec: 0.35
  silence_thresh_db: -40
  min_segment_sec: 0.6
  max_segment_sec: 12.0
  padding_sec: 0.1
  trim_edges: false
  trim_min_sec: 0.05
  diarization_merge_gap_sec: 1.2
  intervening_backchannel_max_sec: 0.85

diarization:
  vad_threshold_db: -42.0
  consecutive_merge_gap_sec: 2.5
  min_similarity_threshold: 0.40


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/data', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/data/audio_preprocess.py
"""Shared audio preprocessing: resample → high-pass → denoise → speech-aware peak normalize."""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Tuple, Union

import torch
import torchaudio

PathLike = Union[str, Path]


def load_audio_file(path: PathLike) -> Tuple[torch.Tensor, int]:
    """Load audio as float tensor [channels, samples] without torchcodec."""
    import soundfile as sf

    try:
        data, sr = sf.read(str(path), dtype="float32")
        if data.ndim == 1:
            wav = torch.from_numpy(data).unsqueeze(0)
        else:
            wav = torch.from_numpy(data.T.copy())
        return wav, int(sr)
    except Exception:
        import subprocess
        import numpy as np
        cmd = ["ffmpeg", "-y", "-i", str(path), "-f", "f32le", "-ac", "1", "-ar", "16000", "-"]
        p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        if p.returncode != 0:
            raise RuntimeError(f"Failed to load audio {path}: {p.stderr.decode('utf-8', errors='ignore')}")
        data = np.frombuffer(p.stdout, dtype=np.float32).copy()
        return torch.from_numpy(data).unsqueeze(0), 16000


def save_audio_file(path: PathLike, wav: torch.Tensor, sample_rate: int) -> None:
    """Save audio tensor [channels, samples] without torchcodec."""
    import soundfile as sf

    x = wav.detach().cpu()
    if x.ndim == 1:
        data = x.numpy()
    elif x.ndim == 2:
        data = x.T.numpy()
    else:
        raise ValueError(f"Unexpected wav shape {tuple(x.shape)}")
    sf.write(str(path), data, sample_rate, subtype="FLOAT")


@dataclass
class PreprocessConfig:
    sample_rate: int = 16000
    peak_normalize: bool = True
    target_peak: float = 0.95
    highpass_hz: float = 80.0
    denoise: bool = False
    denoise_prop: float = 0.75
    speech_normalize: bool = True
    speech_thresh_db: float = -35.0
    vad_frame_ms: float = 25.0
    vad_hop_ms: float = 10.0

    @classmethod
    def from_dict(cls, cfg: Optional[dict]) -> "PreprocessConfig":
        if not cfg:
            return cls()
        fields = cls.__dataclass_fields__
        return cls(**{k: v for k, v in cfg.items() if k in fields})


def _to_mono(wav: torch.Tensor) -> torch.Tensor:
    return wav.mean(0) if wav.dim() > 1 else wav


def _resample(wav: torch.Tensor, sr: int, target_sr: int) -> torch.Tensor:
    if sr == target_sr:
        return wav
    return torchaudio.functional.resample(wav, sr, target_sr)


def _highpass(wav: torch.Tensor, sr: int, cutoff: float) -> torch.Tensor:
    if cutoff <= 0:
        return wav
    return torchaudio.functional.highpass_biquad(wav.unsqueeze(0), sr, cutoff).squeeze(0)


def frame_rms(
    wav: torch.Tensor,
    sample_rate: int,
    frame_ms: float = 25.0,
    hop_ms: float = 10.0,
) -> torch.Tensor:
    frame_len = max(1, int(sample_rate * frame_ms / 1000))
    hop = max(1, int(sample_rate * hop_ms / 1000))
    if wav.numel() < frame_len:
        return torch.tensor([wav.pow(2).mean().sqrt()])
    frames = wav.unfold(0, frame_len, hop)
    return torch.sqrt(frames.pow(2).mean(dim=1) + 1e-10)


def speech_frame_mask(
    wav: torch.Tensor,
    sample_rate: int,
    *,
    thresh_db: float = -35.0,
    frame_ms: float = 25.0,
    hop_ms: float = 10.0,
) -> torch.Tensor:
    """Boolean mask per analysis frame (True = speech)."""
    if wav.numel() == 0:
        return torch.zeros(0, dtype=torch.bool)
    rms = frame_rms(wav, sample_rate, frame_ms, hop_ms)
    ref = rms.max().clamp(min=1e-8)
    thresh = ref * (10 ** (thresh_db / 20.0))
    return rms >= thresh


def speech_sample_mask(
    wav: torch.Tensor,
    sample_rate: int,
    *,
    thresh_db: float = -35.0,
    frame_ms: float = 25.0,
    hop_ms: float = 10.0,
) -> torch.Tensor:
    """Expand frame-level speech mask to per-sample boolean mask."""
    n = wav.shape[0]
    if n == 0:
        return torch.zeros(0, dtype=torch.bool)
    frame_len = max(1, int(sample_rate * frame_ms / 1000))
    hop = max(1, int(sample_rate * hop_ms / 1000))
    frames = speech_frame_mask(wav, sample_rate, thresh_db=thresh_db, frame_ms=frame_ms, hop_ms=hop_ms)
    mask = torch.zeros(n, dtype=torch.bool)
    for i, is_speech in enumerate(frames):
        start = i * hop
        end = min(n, start + frame_len)
        if is_speech:
            mask[start:end] = True
    return mask


def trim_silence_edges(
    wav: torch.Tensor,
    sample_rate: int,
    *,
    thresh_db: float = -35.0,
    frame_ms: float = 25.0,
    hop_ms: float = 10.0,
    min_samples: int = 0,
) -> Tuple[torch.Tensor, int, int]:
    """Trim leading/trailing non-speech. Returns (clip, start_offset, end_offset) in samples."""
    n = wav.shape[0]
    if n == 0:
        return wav, 0, 0

    frames = speech_frame_mask(wav, sample_rate, thresh_db=thresh_db, frame_ms=frame_ms, hop_ms=hop_ms)
    speech_idx = torch.nonzero(frames, as_tuple=False).flatten()
    if speech_idx.numel() == 0:
        return wav, 0, 0

    hop = max(1, int(sample_rate * hop_ms / 1000))
    frame_len = max(1, int(sample_rate * frame_ms / 1000))
    first = int(speech_idx[0].item()) * hop
    last = min(n, int(speech_idx[-1].item()) * hop + frame_len)

    if last - first < min_samples:
        return wav, 0, 0
    return wav[first:last].contiguous(), first, n - last


def _peak_normalize(
    wav: torch.Tensor,
    target: float,
    speech_mask: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    if speech_mask is not None and speech_mask.any():
        peak = wav[speech_mask].abs().max()
    else:
        peak = wav.abs().max()
    if float(peak) < 1e-8:
        return wav
    return wav * (target / peak)


def _denoise(wav: torch.Tensor, sr: int, prop: float) -> torch.Tensor:
    try:
        import noisereduce as nr

        y = nr.reduce_noise(y=wav.detach().cpu().numpy(), sr=sr, stationary=True, prop_decrease=prop)
        return torch.from_numpy(y).to(dtype=wav.dtype, device=wav.device)
    except ImportError:
        return wav


def preprocess_waveform(
    wav: torch.Tensor,
    sample_rate: int,
    cfg: Optional[PreprocessConfig] = None,
) -> torch.Tensor:
    """Apply full chain to a mono waveform tensor."""
    cfg = cfg or PreprocessConfig()
    wav = _to_mono(wav)
    wav = _resample(wav, sample_rate, cfg.sample_rate)
    if cfg.highpass_hz > 0:
        wav = _highpass(wav, cfg.sample_rate, cfg.highpass_hz)
    if cfg.denoise:
        wav = _denoise(wav, cfg.sample_rate, cfg.denoise_prop)
    if cfg.peak_normalize:
        speech_mask = None
        if cfg.speech_normalize:
            speech_mask = speech_sample_mask(
                wav,
                cfg.sample_rate,
                thresh_db=cfg.speech_thresh_db,
                frame_ms=cfg.vad_frame_ms,
                hop_ms=cfg.vad_hop_ms,
            )
        wav = _peak_normalize(wav, cfg.target_peak, speech_mask)
    return wav.contiguous()


def load_waveform(
    path: PathLike,
    cfg: Optional[PreprocessConfig] = None,
    *,
    apply_preprocess: bool = True,
) -> torch.Tensor:
    """Load file; optionally run preprocessing chain."""
    cfg = cfg or PreprocessConfig()
    wav, sr = load_audio_file(path)
    if not apply_preprocess:
        wav = _to_mono(wav)
        wav = _resample(wav, sr, cfg.sample_rate)
        return wav.contiguous()
    return preprocess_waveform(wav, sr, cfg)


def truncate_waveform(
    wav: torch.Tensor,
    sample_rate: int,
    max_duration_sec: Optional[float],
) -> tuple[torch.Tensor, bool]:
    """Trim to max duration. None or <=0 = no limit."""
    if max_duration_sec is None or max_duration_sec <= 0:
        return wav, False
    max_samples = int(max_duration_sec * sample_rate)
    if wav.shape[0] <= max_samples:
        return wav, False
    return wav[:max_samples].contiguous(), True


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/data', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/data/audio_info.py
"""Audio file metadata helpers."""

from __future__ import annotations

from pathlib import Path
from typing import Union

PathLike = Union[str, Path]


def get_duration_sec(path: PathLike) -> float:
    """Return audio duration in seconds."""
    import soundfile as sf

    info = sf.info(str(path))
    return float(info.duration)


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/data', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/data/cmudict.py
"""
CMU Pronouncing Dictionary loader with Modern Vocabulary & Number Normalization.

Maps English words to ARPAbet phoneme sequences (same notation as SpeechOcean762).
Used when canonical phoneme sequence is needed for CTC forced alignment.
"""

from __future__ import annotations

import re
from pathlib import Path
from typing import Dict, List, Optional

# ARPAbet phoneme pattern (e.g. AH0, T, SH)
PHONE_RE = re.compile(r"^[A-Z]{1,2}\d?$")


def _ascii_fold_for_g2p(word: str) -> str:
    """Fold Unicode diacritics (including Vietnamese 'đ/Đ' and tones) to ASCII for CMU/G2P lookup."""
    if not word:
        return ""
    import unicodedata

    folded = word.replace("đ", "d").replace("Đ", "D")
    folded = unicodedata.normalize("NFKD", folded)
    folded = "".join(ch for ch in folded if not unicodedata.combining(ch))
    return re.sub(r"[^A-Za-z']", "", folded).strip()

NUMBER_MAP = {
    0: "zero", 1: "one", 2: "two", 3: "three", 4: "four", 5: "five",
    6: "six", 7: "seven", 8: "eight", 9: "nine", 10: "ten",
    11: "eleven", 12: "twelve", 13: "thirteen", 14: "fourteen", 15: "fifteen",
    16: "sixteen", 17: "seventeen", 18: "eighteen", 19: "nineteen", 20: "twenty",
    30: "thirty", 40: "forty", 50: "fifty", 60: "sixty", 70: "seventy",
    80: "eighty", 90: "ninety"
}

ORDINALS = {
    1: "first", 2: "second", 3: "third", 4: "fourth", 5: "fifth",
    6: "sixth", 7: "seventh", 8: "eighth", 9: "ninth", 10: "tenth",
    11: "eleventh", 12: "twelfth", 13: "thirteenth", 14: "fourteenth", 15: "fifteenth",
    16: "sixteenth", 17: "seventeenth", 18: "eighteenth", 19: "nineteenth", 20: "twentieth",
    30: "thirtieth", 40: "fortieth", 50: "fiftieth", 60: "sixtieth", 70: "seventieth",
    80: "eightieth", 90: "ninetieth"
}

CUSTOM_WORDS: Dict[str, List[str]] = {
    "anime": ["AE1", "N", "AH0", "M", "EY2"],
    "wifi": ["W", "AY1", "F", "AY2"],
    "tiktok": ["T", "IH1", "K", "T", "AA2", "K"],
    "youtube": ["Y", "UW1", "T", "UW2", "B"],
    "podcast": ["P", "AA1", "D", "K", "AE2", "S", "T"],
    "vlog": ["V", "L", "AA1", "G"],
    "app": ["AE1", "P"],
    "online": ["AO1", "N", "L", "AY2", "N"],
    "covid": ["K", "OW1", "V", "IH0", "D"],
    "smartphone": ["S", "M", "AA1", "R", "T", "F", "OW2", "N"],
    "laptop": ["L", "AE1", "P", "T", "AA2", "P"],
    "grab": ["G", "R", "AE1", "B"],
    "shopee": ["SH", "AA1", "P", "IY0"],
    "bun": ["B", "AH1", "N"],
    "bo": ["B", "OW1"],
    "pho": ["F", "AH1"],
    "selfie": ["S", "EH1", "L", "F", "IY0"],
    "meme": ["M", "IY1", "M"],
    "emoji": ["IH0", "M", "OW1", "JH", "IY0"],
    "blog": ["B", "L", "AA1", "G"],
    "influencer": ["IH1", "N", "F", "L", "UW0", "AH0", "N", "S", "ER0"],
    "internet": ["IH1", "N", "T", "ER0", "N", "EH2", "T"],
    "email": ["IY1", "M", "EY2", "L"],
    "website": ["W", "EH1", "B", "S", "AY2", "T"],
    "facebook": ["F", "EY1", "S", "B", "UH2", "K"],
    "instagram": ["IH1", "N", "S", "T", "AH0", "G", "R", "AE2", "M"],
    "zalo": ["Z", "AA1", "L", "OW0"],
    "ai": ["EY1", "AY1"],
    "ok": ["OW1", "K", "EY1"],
    "okay": ["OW1", "K", "EY1"],
}


def int_to_en(n: int, *, year_style: bool = False) -> str:
    """Convert integer to English words."""
    if n < 0:
        return "minus " + int_to_en(-n, year_style=year_style)
    if n <= 20:
        return NUMBER_MAP.get(n, str(n))
    if n < 100:
        tens, rem = divmod(n, 10)
        return NUMBER_MAP.get(tens * 10, "") + ((" " + NUMBER_MAP[rem]) if rem else "")
    if year_style and 1000 <= n <= 2099 and n % 100 != 0:
        return f"{int_to_en(n // 100)} {int_to_en(n % 100)}"
    if n < 1000:
        hundreds, rem = divmod(n, 100)
        return NUMBER_MAP.get(hundreds, "") + " hundred" + ((" " + int_to_en(rem)) if rem else "")
    if n < 1_000_000:
        thousands, rem = divmod(n, 1000)
        return int_to_en(thousands) + " thousand" + ((" " + int_to_en(rem)) if rem else "")
    if n < 1_000_000_000:
        millions, rem = divmod(n, 1_000_000)
        return int_to_en(millions) + " million" + ((" " + int_to_en(rem)) if rem else "")
    return str(n)


def expand_numbers_in_text(text: str) -> str:
    """Expand digits ($50, 50%, 3.5, 42, years) into English words for ASR/CTC alignment."""
    if not text:
        return ""

    def repl_cur(m: re.Match[str]) -> str:
        val = int(m.group(1))
        unit = "dollar" if val == 1 else "dollars"
        return f"{int_to_en(val)} {unit}"

    text = re.sub(r"\$\s*(\d+)", repl_cur, text)
    text = re.sub(r"(\d+)\s*%", lambda m: f"{int_to_en(int(m.group(1)))} percent", text)

    def repl_dec(m: re.Match[str]) -> str:
        int_p = int_to_en(int(m.group(1)))
        dec_p = " ".join(int_to_en(int(d)) for d in m.group(2))
        return f"{int_p} point {dec_p}"

    text = re.sub(r"\b(\d+)\.(\d+)\b", repl_dec, text)
    text = re.sub(r"\b(\d+)\b", lambda m: int_to_en(int(m.group(1)), year_style=True), text)
    return text


def normalize_text_for_phonemes(text: str) -> str:
    """Normalize text: convert digits/times/ordinals to words and normalize punctuation."""
    if not text:
        return ""
    text = re.sub(r"[’‘`´]", "'", text)
    text = re.sub(r"\[.*?\]|\(.*?\)", " ", text)

    # Time (7:30 -> seven thirty) before general number expansion
    def _repl_time(m: re.Match[str]) -> str:
        h, mn = int(m.group(1)), int(m.group(2))
        h_str = int_to_en(h)
        if mn == 0:
            return h_str + " o'clock "
        if mn < 10:
            return h_str + " oh " + int_to_en(mn) + " "
        return h_str + " " + int_to_en(mn) + " "

    text = re.sub(r"\b(\d{1,2}):(\d{2})\b", _repl_time, text)
    # Ordinals (1st -> first, 2nd -> second)
    text = re.sub(
        r"\b(\d+)(st|nd|rd|th)\b",
        lambda m: ORDINALS.get(int(m.group(1)), int_to_en(int(m.group(1))) + "th"),
        text,
        flags=re.I,
    )
    text = expand_numbers_in_text(text)
    return text.replace("-", " ")


def fallback_g2p(word: str) -> List[str]:
    """Lightweight rule-based Grapheme-to-Phoneme fallback for OOV words."""
    w = word.lower().strip()
    if not w:
        return ["AH0"]
    i = 0
    phones = []
    while i < len(w):
        pair = w[i:i+2]
        if pair in ("ch", "tch"):
            phones.append("CH"); i += 2; continue
        if pair == "sh":
            phones.append("SH"); i += 2; continue
        if pair == "th":
            phones.append("TH"); i += 2; continue
        if pair == "ph":
            phones.append("F"); i += 2; continue
        if pair == "ng":
            phones.append("NG"); i += 2; continue
        if pair in ("ee", "ea"):
            phones.append("IY1"); i += 2; continue
        if pair in ("oo",):
            phones.append("UW1"); i += 2; continue
        if pair in ("ai", "ay"):
            phones.append("EY1"); i += 2; continue
        if pair in ("oi", "oy"):
            phones.append("OY1"); i += 2; continue
        if pair in ("ou", "ow"):
            phones.append("AW1"); i += 2; continue
        if pair in ("er", "ir", "ur"):
            phones.append("ER1"); i += 2; continue
        if pair == "ar":
            phones.extend(["AA1", "R"]); i += 2; continue
        if pair == "or":
            phones.extend(["AO1", "R"]); i += 2; continue
        if pair == "qu":
            phones.extend(["K", "W"]); i += 2; continue
        if pair == "ck":
            phones.append("K"); i += 2; continue

        c = w[i]
        if c == "a": phones.append("AE1")
        elif c == "e":
            if i == len(w) - 1 and len(phones) > 0:
                pass
            else:
                phones.append("EH1")
        elif c == "i": phones.append("IH1")
        elif c == "o": phones.append("AA1")
        elif c == "u": phones.append("AH1")
        elif c == "b": phones.append("B")
        elif c == "c":
            if i + 1 < len(w) and w[i+1] in "eiy": phones.append("S")
            else: phones.append("K")
        elif c == "d": phones.append("D")
        elif c == "f": phones.append("F")
        elif c == "g":
            if i + 1 < len(w) and w[i+1] in "eiy": phones.append("JH")
            else: phones.append("G")
        elif c == "h": phones.append("HH")
        elif c == "j": phones.append("JH")
        elif c == "k": phones.append("K")
        elif c == "l": phones.append("L")
        elif c == "m": phones.append("M")
        elif c == "n": phones.append("N")
        elif c == "p": phones.append("P")
        elif c == "r": phones.append("R")
        elif c == "s": phones.append("S")
        elif c == "t": phones.append("T")
        elif c == "v": phones.append("V")
        elif c == "w": phones.append("W")
        elif c == "x": phones.extend(["K", "S"])
        elif c == "y":
            if i == 0: phones.append("Y")
            else: phones.append("IY0")
        elif c == "z": phones.append("Z")
        i += 1
    return phones or ["AH0"]


class CMUDict:
    """Load and query CMUdict for word -> phoneme lookup with Custom Words & Fallback G2P."""

    PHONE_RE = re.compile(r"^[A-Z]{1,2}\d?$")

    def __init__(self, dict_path: Optional[str] = None):
        self._lexicon: Dict[str, List[List[str]]] = {}
        if dict_path and Path(dict_path).exists():
            self._load_file(dict_path)
        else:
            self._load_nltk()

    def _load_file(self, path: str) -> None:
        """Parse standard CMUdict format: WORD  PHONE1 PHONE2 ..."""
        with open(path, encoding="latin-1") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith(";;;"):
                    continue
                if "(" in line:
                    word, rest = line.split("(", 1)
                    rest = rest.rstrip(")")
                    phones = rest.split()[1:]
                else:
                    parts = line.split()
                    word, phones = parts[0], parts[1:]
                key = word.lower()
                self._lexicon.setdefault(key, []).append(phones)

    def _load_nltk(self) -> None:
        """Fallback: download CMUdict via NLTK."""
        try:
            import nltk

            try:
                nltk.data.find("corpora/cmudict")
            except LookupError:
                nltk.download("cmudict", quiet=True)
            from nltk.corpus import cmudict

            for word, phones_list in cmudict.dict().items():
                self._lexicon[word.lower()] = [list(p) for p in phones_list]
        except Exception as exc:
            raise RuntimeError(
                "CMUdict not found. Provide data/cmudict/cmudict.dict or install nltk."
            ) from exc

    def lookup(self, word: str) -> Optional[List[str]]:
        """Return first pronunciation for word from CMUDict or CUSTOM_WORDS, or None."""
        w = word.lower().strip()
        variants = self._lexicon.get(w)
        if variants:
            return variants[0]
        if w in CUSTOM_WORDS:
            return CUSTOM_WORDS[w]
        return None

    def lookup_or_fallback(self, word: str) -> List[str]:
        """Return pronunciation from lexicon, custom words, or fallback G2P (never None)."""
        clean = _ascii_fold_for_g2p(word)
        if not clean:
            return ["AH0"]
        pron = self.lookup(clean)
        if pron:
            return pron
        return fallback_g2p(clean)

    def text_to_phonemes(self, text: str) -> List[str]:
        """Convert transcript to flat phoneme list, normalizing numbers and contractions."""
        norm_text = normalize_text_for_phonemes(text)
        phones: List[str] = []
        for word in norm_text.split():
            clean = _ascii_fold_for_g2p(word)
            if not clean:
                continue
            phones.extend(self.lookup_or_fallback(clean))
        return phones

    def words_to_phoneme_groups(self, text: str) -> List[dict]:
        """
        Return per-word phoneme groups for graph edge construction.
        Guarantees that every recognized word has phonemes (never dropped).
        """
        norm_text = normalize_text_for_phonemes(text)
        groups = []
        for raw_word in norm_text.split():
            clean = _ascii_fold_for_g2p(raw_word)
            if not clean:
                continue
            phones = self.lookup_or_fallback(clean)
            groups.append({"word": clean, "phones": phones})
        return groups


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/data', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/data/silence_split.py
"""Split audio into speech segments at silence boundaries."""

from __future__ import annotations

from dataclasses import dataclass, replace
from pathlib import Path
from typing import Any, Dict, List, Optional, Union

import torch

from data.audio_preprocess import PreprocessConfig, frame_rms, load_waveform, save_audio_file, trim_silence_edges

PathLike = Union[str, Path]


@dataclass
class SilenceSplitConfig:
    min_silence_sec: float = 0.45
    silence_thresh_db: float = -35.0
    min_segment_sec: float = 0.4
    max_segment_sec: Optional[float] = None
    padding_sec: float = 0.08
    frame_ms: float = 25.0
    hop_ms: float = 10.0
    trim_edges: bool = True
    trim_min_sec: float = 0.05

    @classmethod
    def from_dict(cls, cfg: Optional[Any]) -> "SilenceSplitConfig":
        if not cfg:
            return cls()
        if isinstance(cfg, cls):
            return cfg
        if hasattr(cfg, "__dict__") and not isinstance(cfg, dict):
            cfg = vars(cfg)
        if not isinstance(cfg, dict):
            return cls()
        fields = cls.__dataclass_fields__
        return cls(**{k: v for k, v in cfg.items() if k in fields})


def split_waveform(
    wav: torch.Tensor,
    sample_rate: int,
    cfg: Optional[SilenceSplitConfig] = None,
) -> List[Dict[str, float]]:
    """Return [{start_sec, end_sec}, ...] speech segments."""
    cfg = cfg or SilenceSplitConfig()
    if wav.numel() == 0:
        return []

    rms = frame_rms(wav, sample_rate, cfg.frame_ms, cfg.hop_ms)
    ref = rms.max().clamp(min=1e-8)
    thresh = ref * (10 ** (cfg.silence_thresh_db / 20.0))
    speech = rms >= thresh

    hop_sec = cfg.hop_ms / 1000.0
    min_silence_frames = max(1, int(cfg.min_silence_sec / hop_sec))
    min_seg_samples = int(cfg.min_segment_sec * sample_rate)
    pad_samples = int(cfg.padding_sec * sample_rate)
    max_samples = int(cfg.max_segment_sec * sample_rate) if cfg.max_segment_sec else None

    boundaries = [0]
    n = len(speech)
    i = 0
    while i < n:
        if not speech[i]:
            j = i
            while j < n and not speech[j]:
                j += 1
            if j - i >= min_silence_frames:
                split_sample = int((i + (j - i) // 2) * hop_sec * sample_rate)
                if split_sample > boundaries[-1] + min_seg_samples:
                    boundaries.append(split_sample)
            i = j
        else:
            i += 1
    boundaries.append(wav.shape[0])

    segments: List[Dict[str, float]] = []
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        s = max(0, start - pad_samples)
        e = min(wav.shape[0], end + pad_samples)
        if e - s < min_seg_samples:
            continue
        if max_samples and e - s > max_samples:
            while e - s > max_samples:
                segments.append({
                    "start_sec": round(s / sample_rate, 3),
                    "end_sec": round((s + max_samples) / sample_rate, 3),
                })
                s += max_samples
        segments.append({
            "start_sec": round(s / sample_rate, 3),
            "end_sec": round(e / sample_rate, 3),
        })

    if not segments:
        segments.append({"start_sec": 0.0, "end_sec": round(wav.shape[0] / sample_rate, 3)})
    return segments


def split_active_regions(
    wav: torch.Tensor,
    sample_rate: int,
    cfg: Optional[SilenceSplitConfig] = None,
) -> List[Dict[str, float]]:
    """Split diarized tracks (zero-padded between turns) on amplitude gaps."""
    cfg = cfg or SilenceSplitConfig()
    if wav.numel() == 0:
        return []

    mono = wav if wav.dim() == 1 else wav.mean(0)
    peak = float(mono.abs().max())
    if peak < 1e-6:
        return [{"start_sec": 0.0, "end_sec": round(mono.shape[0] / sample_rate, 3)}]

    thresh = peak * (10 ** (cfg.silence_thresh_db / 20.0))
    active = mono.abs() >= thresh
    min_seg = int(cfg.min_segment_sec * sample_rate)
    pad = int(cfg.padding_sec * sample_rate)

    segments: List[Dict[str, float]] = []
    n = mono.shape[0]
    i = 0
    while i < n:
        while i < n and not active[i]:
            i += 1
        if i >= n:
            break
        start = i
        while i < n and active[i]:
            i += 1
        end = i
        if end - start < min_seg:
            continue
        s = max(0, start - pad)
        e = min(n, end + pad)
        segments.append({
            "start_sec": round(s / sample_rate, 3),
            "end_sec": round(e / sample_rate, 3),
        })

    if not segments:
        segments.append({"start_sec": 0.0, "end_sec": round(n / sample_rate, 3)})
    return segments


def _is_sparse_track(wav: torch.Tensor) -> bool:
    """True when many samples are near-zero (typical after 2-speaker diarization)."""
    mono = wav if wav.dim() == 1 else wav.mean(0)
    if mono.numel() == 0:
        return False
    return float((mono.abs() < 1e-5).float().mean()) > 0.2


def export_segments(
    wav: torch.Tensor,
    sample_rate: int,
    segments: List[Dict[str, float]],
    output_dir: PathLike,
    prefix: str = "sent",
    *,
    split_cfg: Optional[SilenceSplitConfig] = None,
) -> List[Dict[str, Any]]:
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    cfg = split_cfg or SilenceSplitConfig()
    min_samples = int(cfg.min_segment_sec * sample_rate)
    trim_min_samples = int(cfg.trim_min_sec * sample_rate)
    exported: List[Dict[str, Any]] = []

    for seg in segments:
        s = int(seg["start_sec"] * sample_rate)
        e = int(seg["end_sec"] * sample_rate)
        clip = wav[s:e].contiguous()

        trim_start = trim_end = 0
        if cfg.trim_edges:
            clip, trim_start, trim_end = trim_silence_edges(
                clip,
                sample_rate,
                thresh_db=cfg.silence_thresh_db,
                frame_ms=cfg.frame_ms,
                hop_ms=cfg.hop_ms,
                min_samples=trim_min_samples,
            )

        if clip.numel() < min_samples:
            continue

        idx = len(exported)
        path = out / f"{prefix}_{idx:03d}.wav"
        save_audio_file(path, clip.unsqueeze(0), sample_rate)
        duration = round(clip.shape[0] / sample_rate, 3)
        exported.append({
            "index": idx,
            "start_sec": round(seg["start_sec"] + trim_start / sample_rate, 3),
            "end_sec": round(seg["end_sec"] - trim_end / sample_rate, 3),
            "duration_sec": duration,
            "path": str(path),
            "preprocessed": True,
        })

    if not exported and wav.numel() > 0:
        mono = wav if wav.dim() == 1 else wav.mean(0)
        min_samples = int(cfg.min_segment_sec * sample_rate)
        if mono.numel() >= min_samples:
            path = out / f"{prefix}_000.wav"
            save_audio_file(path, mono.unsqueeze(0), sample_rate)
            exported.append({
                "index": 0,
                "start_sec": 0.0,
                "end_sec": round(mono.shape[0] / sample_rate, 3),
                "duration_sec": round(mono.shape[0] / sample_rate, 3),
                "path": str(path),
                "preprocessed": True,
            })
    return exported


def export_diarization_clips(
    source_audio: PathLike,
    diarize_segments: list,
    output_dir: PathLike,
    speaker_key: str,
    preprocess: Optional[PreprocessConfig] = None,
    *,
    merge_gap_sec: float = 1.2,
    min_duration_sec: float = 0.6,
    prefix: str = "sent",
    intervening_backchannel_max_sec: float = 0.85,
) -> List[Dict[str, Any]]:
    """Cut clips from original audio using ECAPA diarization time spans."""
    label_map = {
        "A": "Speaker A",
        "B": "Speaker B",
        "TEACHER": "Teacher",
        "T": "Teacher",
        "STUDENT": "Student",
        "S": "Student",
    }
    label = label_map.get(speaker_key.upper(), speaker_key)
    accepted_labels = {label}
    if label == "Speaker A":
        accepted_labels.add("Teacher")
    elif label == "Speaker B":
        accepted_labels.add("Student")
    elif label == "Teacher":
        accepted_labels.add("Speaker A")
    elif label == "Student":
        accepted_labels.add("Speaker B")

    distinct_speakers = {
        getattr(s, "speaker", None) if not isinstance(s, dict) else s.get("speaker")
        for s in diarize_segments
    }
    has_other_speaker = any(sp and sp not in accepted_labels for sp in distinct_speakers)

    merged: List[tuple[float, float]] = []
    if has_other_speaker:
        # Dialogue-aware turn extraction:
        # Intervening other-speaker segments of brief duration (<= intervening_backchannel_max_sec)
        # are non-floor-holding backchannels (e.g. teacher saying 'wow', 'right') and do not break
        # the target speaker's conversational turn if the target continues.
        # But genuine questions/speech (> intervening_backchannel_max_sec) strictly break the turn.
        blocks: List[Dict[str, Any]] = []
        for s in diarize_segments:
            sp = getattr(s, "speaker", None) if not isinstance(s, dict) else s.get("speaker")
            is_target = sp in accepted_labels
            start = float(getattr(s, "start", 0) if not isinstance(s, dict) else s.get("start", 0))
            end = float(getattr(s, "end", 0) if not isinstance(s, dict) else s.get("end", 0))
            if end <= start:
                continue
            dur = end - start
            if not is_target and dur <= intervening_backchannel_max_sec and blocks and blocks[-1]["is_target"]:
                continue
            if not blocks or blocks[-1]["is_target"] != is_target:
                blocks.append({"is_target": is_target, "spans": [(start, end)]})
            else:
                blocks[-1]["spans"].append((start, end))

        for block in blocks:
            if not block["is_target"]:
                continue
            block_merged: List[tuple[float, float]] = []
            for start, end in block["spans"]:
                if block_merged and (start - block_merged[-1][1]) <= merge_gap_sec:
                    block_merged[-1] = (block_merged[-1][0], max(block_merged[-1][1], end))
                else:
                    block_merged.append((start, end))
            merged.extend(block_merged)
    else:
        # Single speaker mode: merge consecutive spans by gap
        spans: List[tuple[float, float]] = []
        for s in diarize_segments:
            sp = getattr(s, "speaker", None) if not isinstance(s, dict) else s.get("speaker")
            if sp is not None and sp not in accepted_labels:
                continue
            start = float(getattr(s, "start", 0) if not isinstance(s, dict) else s.get("start", 0))
            end = float(getattr(s, "end", 0) if not isinstance(s, dict) else s.get("end", 0))
            if end > start:
                spans.append((start, end))
        spans.sort()
        for start, end in spans:
            if end <= start:
                continue
            if merged and start - merged[-1][1] <= merge_gap_sec:
                merged[-1] = (merged[-1][0], max(merged[-1][1], end))
            else:
                merged.append((start, end))

    preprocess = preprocess or PreprocessConfig()
    track_preprocess = replace(preprocess, denoise=False)
    wav = load_waveform(source_audio, track_preprocess)
    sr = track_preprocess.sample_rate
    mono = wav if wav.dim() == 1 else wav.mean(0)

    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    min_samples = int(min_duration_sec * sr)
    exported: List[Dict[str, Any]] = []

    for start, end in merged:
        s = max(0, int(start * sr))
        e = min(mono.shape[0], int(end * sr))
        clip = mono[s:e].contiguous()
        if clip.numel() < min_samples:
            continue
        idx = len(exported)
        path = out / f"{prefix}_{idx:03d}.wav"
        save_audio_file(path, clip.unsqueeze(0), sr)
        exported.append({
            "index": idx,
            "start_sec": round(s / sr, 3),
            "end_sec": round(e / sr, 3),
            "duration_sec": round(clip.shape[0] / sr, 3),
            "path": str(path),
            "preprocessed": True,
        })
    return exported


def split_audio_file(
    audio_path: PathLike,
    output_dir: PathLike,
    preprocess: Optional[PreprocessConfig] = None,
    split_cfg: Optional[dict] = None,
    prefix: str = "sent",
) -> List[Dict[str, Any]]:
    preprocess = preprocess or PreprocessConfig()
    cfg = SilenceSplitConfig.from_dict(split_cfg)
    wav = load_waveform(audio_path, preprocess)
    sr = preprocess.sample_rate
    mono = wav if wav.dim() == 1 else wav.mean(0)
    if _is_sparse_track(mono):
        segments = split_active_regions(mono, sr, cfg)
        return export_segments(mono, sr, segments, output_dir, prefix, split_cfg=cfg)
    segments = split_waveform(wav, sr, cfg)
    return export_segments(wav, sr, segments, output_dir, prefix, split_cfg=cfg)


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/models', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/models/__init__.py
from .wavlm_encoder import WavLMEncoder
from .transformer_encoder import TaskTransformerEncoder
from .ctc_aligner import CTCAligner
from .phoneme_graph import PhonemeGraphNetwork
from .multitask_heads import MultiTaskHeads
from .pronunciation_scorer import PronunciationScorer
from .pronunciation_model import PronunciationAssessmentModel
from .l2_mdd_model import L2MDDModel
from .asha_phonology import ASHAPhonologicalAnalyzer

__all__ = [
    "WavLMEncoder", "TaskTransformerEncoder", "CTCAligner",
    "PhonemeGraphNetwork", "MultiTaskHeads", "PronunciationScorer",
    "PronunciationAssessmentModel", "L2MDDModel", "ASHAPhonologicalAnalyzer",
]


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/models', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/models/asha_phonology.py
"""
ASHA Phonological Processes & Distinctive Articulatory Features Engine.

Based on standards from the American Speech-Language-Hearing Association (ASHA)
and International Phonetic Alphabet (IPA) articulatory dimensions:
  1. Place of Articulation (Vị trí cấu âm)
  2. Manner of Articulation (Phương thức cấu âm)
  3. Voicing / Phonation (Độ rung dây thanh quản)
  4. Vowel Features: Height, Backness, Tenseness, Rounding
"""

from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple


@dataclass(frozen=True)
class ArticulatoryFeatures:
    phoneme: str          # ARPAbet (e.g. 'TH', 'DH', 'AA1')
    ipa: str              # Standard IPA symbol (e.g. 'θ', 'ð', 'ɑː')
    is_vowel: bool
    name_vi: str          # Descriptive Vietnamese name
    place: str            # bilabial, labiodental, dental, alveolar, postalveolar, palatal, velar, glottal
    manner: str           # stop, fricative, affricate, nasal, liquid, glide, vowel
    voicing: str          # voiced, voiceless
    vowel_height: Optional[str] = None      # high, mid-high, mid, low
    vowel_backness: Optional[str] = None    # front, central, back
    vowel_tense: Optional[bool] = None      # True=tense, False=lax
    vowel_rounded: Optional[bool] = None    # True=rounded, False=unrounded


# ── Comprehensive Articulatory Database for ARPAbet Phonemes ────────
PHONEME_FEATURES: Dict[str, ArticulatoryFeatures] = {
    # ── Consonants ──────────────────────────────────────────────
    "P": ArticulatoryFeatures("P", "p", False, "Âm bật môi - vô thanh", "bilabial", "stop", "voiceless"),
    "B": ArticulatoryFeatures("B", "b", False, "Âm bật môi - hữu thanh", "bilabial", "stop", "voiced"),
    "T": ArticulatoryFeatures("T", "t", False, "Âm bật nướu - vô thanh", "alveolar", "stop", "voiceless"),
    "D": ArticulatoryFeatures("D", "d", False, "Âm bật nướu - hữu thanh", "alveolar", "stop", "voiced"),
    "K": ArticulatoryFeatures("K", "k", False, "Âm bật ngạc mềm - vô thanh", "velar", "stop", "voiceless"),
    "G": ArticulatoryFeatures("G", "ɡ", False, "Âm bật ngạc mềm - hữu thanh", "velar", "stop", "voiced"),
    "F": ArticulatoryFeatures("F", "f", False, "Âm xát răng môi - vô thanh", "labiodental", "fricative", "voiceless"),
    "V": ArticulatoryFeatures("V", "v", False, "Âm xát răng môi - hữu thanh", "labiodental", "fricative", "voiced"),
    "TH": ArticulatoryFeatures("TH", "θ", False, "Âm xát liên răng - vô thanh", "dental", "fricative", "voiceless"),
    "DH": ArticulatoryFeatures("DH", "ð", False, "Âm xát liên răng - hữu thanh", "dental", "fricative", "voiced"),
    "S": ArticulatoryFeatures("S", "s", False, "Âm xát rãnh nướu - vô thanh", "alveolar", "fricative", "voiceless"),
    "Z": ArticulatoryFeatures("Z", "z", False, "Âm xát rãnh nướu - hữu thanh", "alveolar", "fricative", "voiced"),
    "SH": ArticulatoryFeatures("SH", "ʃ", False, "Âm xát sau nướu - vô thanh", "postalveolar", "fricative", "voiceless"),
    "ZH": ArticulatoryFeatures("ZH", "ʒ", False, "Âm xát sau nướu - hữu thanh", "postalveolar", "fricative", "voiced"),
    "CH": ArticulatoryFeatures("CH", "tʃ", False, "Âm tắc - xát vòm nướu vô thanh", "postalveolar", "affricate", "voiceless"),
    "JH": ArticulatoryFeatures("JH", "dʒ", False, "Âm tắc - xát vòm nướu hữu thanh", "postalveolar", "affricate", "voiced"),
    "M": ArticulatoryFeatures("M", "m", False, "Âm mũi hai môi - hữu thanh", "bilabial", "nasal", "voiced"),
    "N": ArticulatoryFeatures("N", "n", False, "Âm mũi chân răng - hữu thanh", "alveolar", "nasal", "voiced"),
    "NG": ArticulatoryFeatures("NG", "ŋ", False, "Âm mũi ngạc mềm - hữu thanh", "velar", "nasal", "voiced"),
    "L": ArticulatoryFeatures("L", "l", False, "Âm ngạc bên (liquid) - hữu thanh", "alveolar", "liquid", "voiced"),
    "R": ArticulatoryFeatures("R", "r", False, "Âm uốn lưỡi (liquid) - hữu thanh", "postalveolar", "liquid", "voiced"),
    "W": ArticulatoryFeatures("W", "w", False, "Âm lướt tròn môi ngạc mềm", "velar", "glide", "voiced"),
    "Y": ArticulatoryFeatures("Y", "j", False, "Âm lướt vòm cứng", "palatal", "glide", "voiced"),
    "HH": ArticulatoryFeatures("HH", "h", False, "Âm xát thanh môn - vô thanh", "glottal", "fricative", "voiceless"),

    # ── Vowels ──────────────────────────────────────────────────
    "IY": ArticulatoryFeatures("IY", "iː", True, "Nguyên âm trước - cao căng", "palatal", "vowel", "voiced", "high", "front", True, False),
    "IH": ArticulatoryFeatures("IH", "ɪ", True, "Nguyên âm trước - cao lỏng", "palatal", "vowel", "voiced", "high", "front", False, False),
    "EY": ArticulatoryFeatures("EY", "eɪ", True, "Nguyên âm đôi trước - đóng", "palatal", "vowel", "voiced", "mid-high", "front", True, False),
    "EH": ArticulatoryFeatures("EH", "e", True, "Nguyên âm trước - trung bình mở", "palatal", "vowel", "voiced", "mid", "front", False, False),
    "AE": ArticulatoryFeatures("AE", "æ", True, "Nguyên âm trước - thấp bẹt", "alveolar", "vowel", "voiced", "low", "front", False, False),
    "AA": ArticulatoryFeatures("AA", "ɑː", True, "Nguyên âm sau - thấp mở rộng", "velar", "vowel", "voiced", "low", "back", True, False),
    "AO": ArticulatoryFeatures("AO", "ɔː", True, "Nguyên âm sau - trung bình tròn môi", "velar", "vowel", "voiced", "mid", "back", True, True),
    "OW": ArticulatoryFeatures("OW", "oʊ", True, "Nguyên âm đôi sau - đóng tròn môi", "velar", "vowel", "voiced", "mid-high", "back", True, True),
    "UH": ArticulatoryFeatures("UH", "ʊ", True, "Nguyên âm sau - cao lỏng tròn môi", "velar", "vowel", "voiced", "high", "back", False, True),
    "UW": ArticulatoryFeatures("UW", "uː", True, "Nguyên âm sau - cao căng tròn môi", "velar", "vowel", "voiced", "high", "back", True, True),
    "AH": ArticulatoryFeatures("AH", "ʌ", True, "Nguyên âm giữa - thấp trung tính", "central", "vowel", "voiced", "mid-low", "central", False, False),
    "AX": ArticulatoryFeatures("AX", "ə", True, "Nguyên âm lướt Schwa trung tính", "central", "vowel", "voiced", "mid", "central", False, False),
    "ER": ArticulatoryFeatures("ER", "ɜːr", True, "Nguyên âm giữa uốn lưỡi", "postalveolar", "vowel", "voiced", "mid", "central", True, False),
    "AY": ArticulatoryFeatures("AY", "aɪ", True, "Nguyên âm đôi mở -> cao", "central", "vowel", "voiced", "low", "front", True, False),
    "AW": ArticulatoryFeatures("AW", "aʊ", True, "Nguyên âm đôi mở -> tròn môi", "central", "vowel", "voiced", "low", "back", True, True),
    "OY": ArticulatoryFeatures("OY", "ɔɪ", True, "Nguyên âm đôi sau -> trước", "velar", "vowel", "voiced", "mid", "front", True, True),
}


def get_features(phoneme: str) -> Optional[ArticulatoryFeatures]:
    """Retrieve features for any phoneme, stripping stress numbers."""
    if not phoneme:
        return None
    cleaned = phoneme.strip().upper()
    base = cleaned.rstrip("012")
    if base in PHONEME_FEATURES:
        return PHONEME_FEATURES[base]
    # Fallback to IPA lookup if given an IPA glyph directly
    for feat in PHONEME_FEATURES.values():
        if feat.ipa == phoneme or feat.phoneme == cleaned:
            return feat
    return None


@dataclass
class ASHADiagnosis:
    rule_id: str                      # Standard ASHA code e.g. 'stopping', 'devoicing'
    rule_name_vi: str                 # Tên quy tắc tiếng Việt
    rule_name_en: str                 # ASHA English Name
    severity: str                     # 'critical', 'warning', 'minor'
    target_ipa: str                   # e.g. '/θ/'
    actual_ipa: str                   # e.g. '/t/'
    feature_contrast: str             # So sánh đặc tính cấu âm đối lập
    articulatory_tip: str             # Hướng dẫn định vị cơ quan cấu âm chuẩn


class ASHAPhonologicalAnalyzer:
    """ASHA Clinical Phonological Diagnostic Engine."""

    @classmethod
    def diagnose(
        cls,
        target_phone: str,
        actual_phone: str,
        mdd_class: str = "substitution",
        is_word_final: bool = False,
        in_cluster: bool = False,
        word: str = "",
    ) -> ASHADiagnosis:
        """Diagnose a pronunciation error using ASHA rules and distinctive features."""
        target_feat = get_features(target_phone)
        target_ipa = f"/{target_feat.ipa}/" if target_feat else f"/{target_phone}/"

        # ── 0. Correct Pronunciation ──
        if target_phone.rstrip("012") == actual_phone.rstrip("012") and mdd_class.lower() not in ("deletion", "addition"):
            return ASHADiagnosis(
                rule_id="correct",
                rule_name_vi="Phát âm chuẩn",
                rule_name_en="Correct Pronunciation",
                severity="none",
                target_ipa=target_ipa,
                actual_ipa=target_ipa,
                feature_contrast="Đạt chuẩn cấu âm",
                articulatory_tip="Phát âm tốt, duy trì độ chính xác này.",
            )

        # ── 1. Deletion Cases (ASHA Syllable Structure Processes) ──
        if mdd_class.lower() == "deletion" or actual_phone in ("[DELETION]", "[DEL]", "sil", ""):
            if is_word_final:
                return ASHADiagnosis(
                    rule_id="final_consonant_deletion",
                    rule_name_vi="Nuốt phụ âm cuối (Final Consonant Deletion)",
                    rule_name_en="Final Consonant Deletion (FCD)",
                    severity="critical",
                    target_ipa=target_ipa,
                    actual_ipa="[Bị nuốt / Bỏ qua]",
                    feature_contrast=f"Target: {target_feat.name_vi if target_feat else target_phone} -> Actual: Hoàn toàn bị lược bỏ âm đuôi",
                    articulatory_tip="Tiếng Anh bắt buộc phát âm dứt khoát âm cuối từ. Hãy hoàn thành cử động đóng cơ quan cấu âm (ngậm môi, chạm đầu lưỡi hoặc thổi xát hơi) trước khi chuyển sang từ tiếp theo.",
                )
            if in_cluster:
                return ASHADiagnosis(
                    rule_id="cluster_reduction",
                    rule_name_vi="Rút gọn tổ hợp phụ âm (Cluster Reduction)",
                    rule_name_en="Consonant Cluster Reduction",
                    severity="warning",
                    target_ipa=target_ipa,
                    actual_ipa="[Bị nuốt trong cụm]",
                    feature_contrast=f"Target: {target_feat.name_vi if target_feat else target_phone} bị bỏ qua trong chuỗi phụ âm liền kề",
                    articulatory_tip="Hãy tách chậm cụm phụ âm thành các âm đơn lẻ, tập chuyển đổi khẩu hình liên hoàn rồi mới tăng dần tốc độ để không bị rơi mất âm.",
                )
            return ASHADiagnosis(
                rule_id="omission",
                rule_name_vi="Nuốt âm / Bỏ sót âm vị (Omission)",
                rule_name_en="Sound Omission",
                severity="warning",
                target_ipa=target_ipa,
                actual_ipa="[Bỏ qua]",
                feature_contrast=f"Target: {target_feat.name_vi if target_feat else target_phone} không được cấu âm",
                articulatory_tip="Hãy chú ý phát âm đủ tất cả các âm tiết trong từ, tránh vội vã lướt qua các âm vị trọng tâm.",
            )

        # ── 2. Addition Cases ──────────────────────────────────────
        if mdd_class.lower() == "addition" or actual_phone in ("[ADDITION]", "[ADD]"):
            return ASHADiagnosis(
                rule_id="epenthesis",
                rule_name_vi="Chêm âm thừa (Epenthesis)",
                rule_name_en="Epenthesis / Vowel Addition",
                severity="minor",
                target_ipa=target_ipa,
                actual_ipa="[Âm chêm thừa]",
                feature_contrast="Xuất hiện thêm âm lướt / schwa không có trong cấu trúc từ chuẩn",
                articulatory_tip="Không thêm nguyên âm /ə/ (ơ) vào giữa các cụm phụ âm hoặc sau các âm đuôi. Dứt khoát dừng hơi ngay khi phụ âm kết thúc.",
            )

        # ── 3. Substitution Cases (Articulatory Contrast) ──────────
        actual_feat = get_features(actual_phone)
        actual_ipa = f"/{actual_feat.ipa}/" if actual_feat else f"/{actual_phone}/"

        if not target_feat or not actual_feat:
            return ASHADiagnosis(
                rule_id="substitution_generic",
                rule_name_vi="Thay thế âm vị (Sound Substitution)",
                rule_name_en="Phoneme Substitution",
                severity="warning",
                target_ipa=target_ipa,
                actual_ipa=actual_ipa,
                feature_contrast=f"Phát âm lệch từ {target_ipa} sang {actual_ipa}",
                articulatory_tip=f"Luyện tập phát âm đối chiếu giữa âm chuẩn {target_ipa} và âm nhầm lẫn {actual_ipa}.",
            )

        # Rule 3.1: Interdental Dentalization (θ, ð -> s, z, f, v, t, d)
        if target_feat.place == "dental" and actual_feat.place != "dental":
            if actual_feat.manner == "stop":
                return ASHADiagnosis(
                    rule_id="dental_stopping",
                    rule_name_vi="Thay âm liên răng bằng âm tắc (Dental Stopping)",
                    rule_name_en="Interdental Stopping",
                    severity="critical",
                    target_ipa=target_ipa,
                    actual_ipa=actual_ipa,
                    feature_contrast=f"Target: Âm xát liên răng ({target_feat.place}/{target_feat.manner}) -> Actual: Bị chặn thành âm tắc ({actual_feat.place}/{actual_feat.manner})",
                    articulatory_tip=f"Đặt nhẹ đầu lưỡi ở giữa hai rìa răng cửa trên và dưới. Thổi luồng hơi liên tục qua khe răng - lưỡi (không dùng đầu lưỡi chặn đứng dòng khí như âm {actual_ipa}).",
                )
            return ASHADiagnosis(
                rule_id="dentalization_substitution",
                rule_name_vi="Sai vị trí liên răng (Interdental Substitution)",
                rule_name_en="Interdental Substitution / Dentalization",
                severity="critical",
                target_ipa=target_ipa,
                actual_ipa=actual_ipa,
                feature_contrast=f"Target: Âm liên răng ({target_feat.place}) -> Actual: Đặt sai vị trí tại ({actual_feat.place})",
                articulatory_tip=f"Đầu lưỡi phải đưa ra ngoài giữa hai hàm răng. Tuyệt đối không thụt lưỡi vào chân răng tạo âm /s, z/ hoặc cắn môi dưới tạo âm /f, v/.",
            )

        # Rule 3.2: Stopping (Fricative/Affricate -> Stop)
        if target_feat.manner in ("fricative", "affricate") and actual_feat.manner == "stop":
            return ASHADiagnosis(
                rule_id="stopping",
                rule_name_vi="Thay âm xát bằng âm tắc (Stopping)",
                rule_name_en="Stopping of Fricatives/Affricates",
                severity="critical",
                target_ipa=target_ipa,
                actual_ipa=actual_ipa,
                feature_contrast=f"Target: Âm xát luồng hơi liên tục ({target_feat.manner}) -> Actual: Luồng khí bị chặn hoàn toàn ({actual_feat.manner})",
                articulatory_tip=f"Giữ khoảng hở hẹp giữa cơ quan cấu âm để luồng hơi thoát ra đều đặn tạo tiếng xì xát liên tục, không ngậm chặt môi/lưỡi chặn đứng hơi.",
            )

        # Rule 3.3: Devoicing (Final or Medial Consonant Devoicing)
        if target_feat.voicing == "voiced" and actual_feat.voicing == "voiceless":
            loc = "cuối từ" if is_word_final else ""
            return ASHADiagnosis(
                rule_id="devoicing",
                rule_name_vi=f"Mất âm hữu thanh {loc} (Devoicing)",
                rule_name_en="Consonant Devoicing",
                severity="warning",
                target_ipa=target_ipa,
                actual_ipa=actual_ipa,
                feature_contrast=f"Target: Dây thanh rung ({target_feat.voicing}) -> Actual: Mất rung ({actual_feat.voicing})",
                articulatory_tip="Đặt 2 ngón tay lên hõm cổ họng / thanh quản, chủ động duy trì độ rung của dây thanh quản cho đến khi dứt hẳn âm, tránh để âm bị xì hơi vô thanh.",
            )

        # Rule 3.4: Voicing (Voiceless -> Voiced)
        if target_feat.voicing == "voiceless" and actual_feat.voicing == "voiced":
            return ASHADiagnosis(
                rule_id="voicing",
                rule_name_vi="Hữu thanh hóa (Voicing)",
                rule_name_en="Pre-vocalic Voicing",
                severity="minor",
                target_ipa=target_ipa,
                actual_ipa=actual_ipa,
                feature_contrast=f"Target: Vô thanh bật hơi ({target_feat.voicing}) -> Actual: Rung thanh quản quá sớm ({actual_feat.voicing})",
                articulatory_tip="Dùng luồng hơi thuần túy từ phổi bật ra mà không rung dây thanh quản ở giai đoạn đầu của âm.",
            )

        # Rule 3.5: Velar / Palatal Fronting (Velar/Postalveolar -> Alveolar)
        if target_feat.place in ("velar", "postalveolar", "palatal") and actual_feat.place in ("alveolar", "dental", "bilabial"):
            return ASHADiagnosis(
                rule_id="fronting",
                rule_name_vi="Đưa âm ra trước vòm họng (Fronting)",
                rule_name_en="Velar / Palatal Fronting",
                severity="warning",
                target_ipa=target_ipa,
                actual_ipa=actual_ipa,
                feature_contrast=f"Target: Cấu âm tại ngạc sâu ({target_feat.place}) -> Actual: Bị đẩy ra phía trước răng ({actual_feat.place})",
                articulatory_tip="Hạ đầu lưỡi xuống sàn miệng, nâng phần cuống lưỡi / gốc lưỡi chạm vào ngạc mềm phía sau vòm họng để tạo âm vang sâu.",
            )

        # Rule 3.6: Gliding of Liquids (r, l -> w, j)
        if target_feat.manner == "liquid" and actual_feat.manner == "glide":
            return ASHADiagnosis(
                rule_id="gliding",
                rule_name_vi="Biến âm lỏng thành âm lướt (Gliding of Liquids)",
                rule_name_en="Gliding of Liquids",
                severity="warning",
                target_ipa=target_ipa,
                actual_ipa=actual_ipa,
                feature_contrast=f"Target: Âm uốn/bên ({target_feat.manner}) -> Actual: Chu môi thành âm lướt ({actual_feat.manner})",
                articulatory_tip=f"Không chu tròn môi như âm /w/. Giữ khóe miệng tự nhiên, uốn cong đầu lưỡi lên vòm họng (đối với /r/) hoặc chạm đầu lưỡi vào chân răng cửa trên (đối với /l/).",
            )

        # Rule 3.7: Deaffrication (tʃ, dʒ -> ʃ, s, t)
        if target_feat.manner == "affricate" and actual_feat.manner != "affricate":
            return ASHADiagnosis(
                rule_id="deaffrication",
                rule_name_vi="Mất tính chất tắc - xát (Deaffrication)",
                rule_name_en="Deaffrication",
                severity="warning",
                target_ipa=target_ipa,
                actual_ipa=actual_ipa,
                feature_contrast=f"Target: Âm kết hợp vừa chặn vừa xát ({target_feat.manner}) -> Actual: Chỉ còn âm đơn ({actual_feat.manner})",
                articulatory_tip="Cần thực hiện đủ 2 pha: (1) Ép đầu lưỡi chặn hơi tại nướu trên, (2) Bật hé lưỡi tạo luồng gió xát mạnh mẽ dứt khoát.",
            )

        # Rule 3.8: Vowel Tense vs Lax Confusion (i: vs ɪ, u: vs ʊ)
        if target_feat.is_vowel and actual_feat.is_vowel:
            if target_feat.vowel_tense != actual_feat.vowel_tense:
                tense_state = "Căng (Tense)" if target_feat.vowel_tense else "Lỏng/Ngắn (Lax)"
                actual_state = "Căng (Tense)" if actual_feat.vowel_tense else "Lỏng/Ngắn (Lax)"
                return ASHADiagnosis(
                    rule_id="vowel_tenseness",
                    rule_name_vi="Nhầm lẫn nguyên âm căng và lỏng (Vowel Tenseness)",
                    rule_name_en="Vowel Tenseness / Centralization",
                    severity="warning",
                    target_ipa=target_ipa,
                    actual_ipa=actual_ipa,
                    feature_contrast=f"Target: Nguyên âm {tense_state} -> Actual: Nguyên âm {actual_state}",
                    articulatory_tip="Điều chỉnh độ căng của cơ má và khóe miệng: Nguyên âm căng đòi hỏi căng cơ má và kéo dài âm; nguyên âm lỏng thả lỏng cơ miệng và phát âm dứt khoát ngắn hơn.",
                )

        # Default fallback contrast
        return ASHADiagnosis(
            rule_id="substitution_contrast",
            rule_name_vi="Lệch đặc tính cấu âm (Articulatory Deviation)",
            rule_name_en="Articulatory Feature Deviation",
            severity="warning",
            target_ipa=target_ipa,
            actual_ipa=actual_ipa,
            feature_contrast=f"Target: [{target_feat.place}, {target_feat.manner}, {target_feat.voicing}] -> Actual: [{actual_feat.place}, {actual_feat.manner}, {actual_feat.voicing}]",
            articulatory_tip=f"Chú ý căn chỉnh lại khẩu hình từ {actual_ipa} về âm chuẩn {target_ipa} ({target_feat.name_vi}).",
        )


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/models', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/models/l2_mdd_model.py
"""
L2-MDD (Mispronunciation Detection and Diagnosis) Model.

Directly loads and executes the L2-ARCTIC trained checkpoint (l2_mdd_best.pt):
  Waveform -> WavLM -> TaskTransformer -> CTC Align -> Phoneme Graph (GAT) -> PhonemeMDDHead (4 classes)

Classes:
  0: Correct
  1: Substitution
  2: Deletion
  3: Addition

Coupled with ASHA Phonological Diagnostic Engine for clinical-grade articulatory feedback.
"""

from __future__ import annotations

import os
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

from .wavlm_encoder import WavLMEncoder
from .transformer_encoder import TaskTransformerEncoder
from .ctc_aligner import CTCAligner, PhonemeAlignment
from .phoneme_graph import PhonemeGraphNetwork
from .asha_phonology import ASHAPhonologicalAnalyzer, ASHADiagnosis, get_features

ERR_CORRECT, ERR_SUB, ERR_DEL, ERR_ADD = 0, 1, 2, 3
ERR_NAMES = ("correct", "substitution", "deletion", "addition")

TYPICAL_SUBSTITUTIONS: Dict[str, List[str]] = {
    "TH": ["T", "S", "F"],
    "DH": ["D", "Z", "V"],
    "SH": ["S", "CH"],
    "ZH": ["Z", "JH"],
    "CH": ["SH", "T", "S"],
    "JH": ["CH", "Z", "D"],
    "V":  ["B", "F"],
    "W":  ["V"],
    "Z":  ["S"],
    "S":  ["SH", "Z"],
    "F":  ["P", "V"],
    "M":  ["N"],
    "N":  ["L", "NG"],
    "NG": ["N"],
    "R":  ["L", "W"],
    "L":  ["R", "N"],
    "P":  ["B"],
    "T":  ["D"],
    "K":  ["G"],
    "B":  ["P"],
    "D":  ["T"],
    "G":  ["K"],
    "AE": ["EH", "AA", "AH"],
    "EH": ["AE", "AH"],
    "IY": ["IH"],
    "IH": ["IY", "EH"],
    "UW": ["UH"],
    "UH": ["UW"],
    "AO": ["AA", "OW"],
    "AA": ["AO", "AH"],
    "ER": ["AH"],
}


class PhonemeMDDHead(nn.Module):
    """Classification head for 4 MDD classes (correct, substitution, deletion, addition)."""

    def __init__(self, input_dim: int = 256, num_classes: int = 4, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Dropout(dropout),
            nn.Linear(input_dim, num_classes),
        )

    def forward(self, node_features: torch.Tensor) -> torch.Tensor:
        return self.net(node_features)


class L2MDDModel(nn.Module):
    """Full L2-MDD architecture matching transformer_models/l2_mdd_best.pt."""

    def __init__(
        self,
        wavlm_name: str = "microsoft/wavlm-large",
        ff_dim: int = 2048,
        graph_hidden: int = 256,
        num_heads: int = 4,
        num_classes: int = 4,
        use_lora: bool = False,
        lora_r: int = 8,
        lora_alpha: int = 16,
        lora_target_modules: Optional[List[str]] = None,
        use_canon_conditioning: bool = True,
        canon_emb_dim: int = 256,
    ):
        super().__init__()
        self.wavlm = WavLMEncoder(
            model_name=wavlm_name,
            freeze=True,
            use_lora=use_lora,
            lora_r=lora_r,
            lora_alpha=lora_alpha,
            lora_target_modules=lora_target_modules,
        )
        d = self.wavlm.output_dim  # 1024

        self.task_transformer = TaskTransformerEncoder(
            input_dim=d,
            num_layers=3,
            num_heads=8,
            ff_dim=ff_dim,  # 2048 in l2_mdd_best.pt
            dropout=0.1,
            max_seq_len=800,
        )

        self.ctc_aligner = CTCAligner(input_dim=d, use_pretrained_bundle=False)
        self.use_canon_conditioning = bool(use_canon_conditioning)
        if self.use_canon_conditioning:
            self.canon_emb = nn.Embedding(self.ctc_aligner.num_tokens, canon_emb_dim)
            self.node_fusion = nn.Sequential(
                nn.Linear(d + canon_emb_dim + 4, d),
                nn.LayerNorm(d),
                nn.GELU(),
            )
        else:
            self.canon_emb = None
            self.node_fusion = None

        self.phoneme_graph = PhonemeGraphNetwork(
            input_dim=d,
            hidden_dim=graph_hidden,
            num_layers=2,
            num_heads=num_heads,
            dropout=0.1,
            edge_sequential=True,
            edge_same_word=True,
            edge_same_syllable=False,
        )
        self.mdd_head = PhonemeMDDHead(input_dim=graph_hidden, num_classes=num_classes)

    def _condition_nodes(
        self,
        acoustic_nodes: torch.Tensor,
        ctc_log_probs_uttr: torch.Tensor,
        spans: List[PhonemeAlignment],
        phoneme_tokens: List[str],
    ) -> torch.Tensor:
        if not self.use_canon_conditioning or self.canon_emb is None or acoustic_nodes.shape[0] == 0:
            return acoustic_nodes
        N = acoustic_nodes.shape[0]
        device = acoustic_nodes.device
        target_ids = self.ctc_aligner.phonemes_to_ids(phoneme_tokens[:N]).to(device)
        canon_vecs = self.canon_emb(target_ids)

        T_u = ctc_log_probs_uttr.shape[0]
        probs = ctc_log_probs_uttr.detach().float().exp()
        cues = []
        for i in range(N):
            tid = int(target_ids[i].item())
            sp = spans[i] if i < len(spans) else None
            if sp is not None and T_u > 0:
                sf = max(0, min(T_u - 1, int(sp.start_frame)))
                ef = max(sf, min(T_u - 1, int(sp.end_frame)))
                seg = probs[sf : ef + 1]
                p_tgt = seg[:, tid].max() if tid < seg.shape[1] else acoustic_nodes.new_tensor(0.0)
                p_blk = seg[:, 0].mean()
                other = seg.clone()
                other[:, :3] = 0.0
                if tid < other.shape[1]:
                    other[:, tid] = 0.0
                p_oth = other.max()
                dur_norm = min(1.0, (ef - sf + 1) / 10.0)
                cues.append(torch.stack([p_tgt, p_blk, p_oth, p_tgt.new_tensor(dur_norm)]))
            else:
                cues.append(acoustic_nodes.new_zeros(4))
        cues_t = torch.stack(cues, dim=0).to(device=device, dtype=acoustic_nodes.dtype)
        fused = torch.cat([acoustic_nodes, canon_vecs.to(dtype=acoustic_nodes.dtype), cues_t], dim=-1)
        return self.node_fusion(fused)

    @torch.no_grad()
    def forward_utterance(
        self,
        waveform: torch.Tensor,
        wav_length: torch.Tensor,
        phoneme_tokens: List[str],
        word_phone_ranges: List[Tuple[int, int]],
    ) -> Tuple[torch.Tensor, torch.Tensor, List[PhonemeAlignment]]:
        """
        Forward a single utterance through L2-MDD.

        Returns:
            mdd_logits: (N, 4) logits over (correct, sub, del, add)
            ctc_logits: (T, 72) frame-level phoneme logits
            spans: List[PhonemeAlignment]
        """
        if waveform.dim() == 1:
            waveform = waveform.unsqueeze(0)
        if wav_length.dim() == 0:
            wav_length = wav_length.unsqueeze(0)

        frame_feats = self.wavlm(waveform, wav_lengths=wav_length)
        T = frame_feats.shape[1]
        frame_lengths = self.wavlm.frame_lengths_from_samples(wav_length)
        pad_mask = (
            torch.arange(T, device=waveform.device).unsqueeze(0)
            >= frame_lengths.unsqueeze(1)
        )
        frame_feats = self.task_transformer(frame_feats, src_key_padding_mask=pad_mask)

        # Forced alignment & pooled phoneme node features
        T_0 = int(frame_lengths[0].item())
        ctc_logits = self.ctc_aligner.ctc_proj(frame_feats[0, :T_0])
        ctc_log_probs = F.log_softmax(ctc_logits.float(), dim=-1)
        spans, nodes = self.ctc_aligner.align_utterance(
            frame_feats[0, :T_0], phoneme_tokens, T_0, log_probs=ctc_log_probs.unsqueeze(0)
        )
        nodes = self._condition_nodes(nodes, ctc_log_probs, spans, phoneme_tokens)

        # Graph attention & MDD classification
        gat_nodes = self.phoneme_graph.forward_single(nodes, word_phone_ranges)
        mdd_logits = self.mdd_head(gat_nodes)

        return mdd_logits, ctc_logits, spans

    @torch.no_grad()
    def scan_phonemes(
        self,
        waveform: torch.Tensor,
        wav_length: torch.Tensor,
        phoneme_tokens: List[str],
        words: List[str],
        word_phone_ranges: List[Tuple[int, int]],
        word_ipas: Optional[List[str]] = None,
        sensitivity_threshold: float = 0.35,
    ) -> Dict[str, Any]:
        """
        Scan all phonemes in the utterance and detect suspicious pronunciations.
        
        Returns:
            suspicious_phonemes: List of diagnosed suspicious phonemes with Target -> Actual
            all_phoneme_diagnoses: Full list of all phonemes with MDD probabilities
            summary: Brief phonetic overview
        """
        self.eval()
        mdd_logits, ctc_logits, spans = self.forward_utterance(
            waveform, wav_length, phoneme_tokens, word_phone_ranges
        )

        probs = F.softmax(mdd_logits, dim=-1)
        preds = mdd_logits.argmax(-1).cpu().tolist()
        vocab = self.ctc_aligner.phoneme_vocab

        # Map each phoneme index to its containing word
        phone_to_word: Dict[int, Dict[str, Any]] = {}
        for w_idx, (ws, we) in enumerate(word_phone_ranges):
            w_text = words[w_idx] if w_idx < len(words) else ""
            w_ipa = word_ipas[w_idx] if word_ipas and w_idx < len(word_ipas) else ""
            for p_idx in range(ws, we):
                phone_to_word[p_idx] = {
                    "word": w_text,
                    "word_ipa": w_ipa,
                    "word_idx": w_idx,
                    "is_first": (p_idx == ws),
                    "is_last": (p_idx == we - 1),
                    "is_cluster": (we - ws > 2),
                }

        suspicious_list: List[Dict[str, Any]] = []
        full_list: List[Dict[str, Any]] = []

        for i, target_phone in enumerate(phoneme_tokens):
            pred_class_id = preds[i] if i < len(preds) else 0
            pred_class_name = ERR_NAMES[pred_class_id]
            p_vec = probs[i].cpu().tolist() if i < len(probs) else [1.0, 0.0, 0.0, 0.0]
            p_correct, p_sub, p_del, p_add = p_vec[0], p_vec[1], p_vec[2], p_vec[3]
            err_prob = p_sub + p_del + p_add
            sp = spans[i] if i < len(spans) else None
            w_ctx = phone_to_word.get(i, {})

            # ── Acoustic CTC & L2-MDD Sensitive Diagnosis ──
            # Whisper normalizes transcript to standard English, masking speech errors.
            # We directly evaluate acoustic CTC logits over the phoneme span [sf, ef]
            # to verify what the speaker actually pronounced vs the reference target.
            target_base = target_phone.rstrip("012")
            target_feat = get_features(target_phone)
            target_is_vowel = target_feat.is_vowel if target_feat else False
            typs = TYPICAL_SUBSTITUTIONS.get(target_base, [])

            actual_phone = target_phone
            is_suspicious = False
            best_cand = None
            cand_margin = 0.0

            if sp and sp.end_frame >= sp.start_frame:
                sf = max(0, sp.start_frame)
                ef = min(ctc_logits.shape[0] - 1, sp.end_frame)
                if ef >= sf:
                    span_ctc = ctc_logits[sf : ef + 1].max(0).values.clone()
                    blank_logit = span_ctc[0].item() if len(span_ctc) > 0 else -100.0
                    span_ctc[:3] = -1e9  # mask <pad>, <unk>, |

                    target_id = int(self.ctc_aligner.phonemes_to_ids([target_phone])[0].item())
                    target_logit = span_ctc[target_id].item() if (0 <= target_id < len(span_ctc)) else -100.0

                    # 1. Acoustic Deletion Check: Dropped coda consonant in word-final position
                    if w_ctx.get("is_last", False) and not target_is_vowel:
                        if target_logit < -10.5 and blank_logit > target_logit + 4.0 and (p_del >= 0.10 or ef - sf <= 1):
                            actual_phone = "[DELETION]"
                            pred_class_name = "deletion"
                            is_suspicious = True
                            err_prob = max(err_prob, 0.75)

                    # 2. Acoustic Substitution Check: Compare top candidates against target
                    if not is_suspicious:
                        top_indices = torch.topk(span_ctc, k=min(12, span_ctc.shape[0])).indices.cpu().tolist()
                        for tid in top_indices:
                            cand_phone = vocab[tid] if tid < len(vocab) else ""
                            if not cand_phone or cand_phone in ("<pad>", "<unk>", "|"):
                                continue
                            cand_feat = get_features(cand_phone)
                            cand_is_vowel = cand_feat.is_vowel if cand_feat else False
                            # Consonants must never be substituted by vowels and vice-versa
                            if cand_is_vowel != target_is_vowel:
                                continue
                            cand_base = cand_phone.rstrip("012")
                            if cand_base == target_base:
                                best_cand = target_phone
                                break

                            cand_logit = span_ctc[tid].item()
                            margin = cand_logit - target_logit

                            is_typ = cand_base in typs
                            manner_match = (cand_feat and target_feat and cand_feat.manner == target_feat.manner)

                            # Softened / calibrated acoustic threshold:
                            if target_is_vowel:
                                # Vowels: only test phonologically typical L2 pairs, ignoring non-typical pairs (coarticulation artifacts)
                                if not is_typ:
                                    continue
                                thresh = 1.8
                                if margin >= thresh and err_prob >= 0.08:
                                    best_cand = cand_phone
                                    cand_margin = margin
                                    break
                            else:
                                # Consonants: 1.2 for typical L2 pairs (CH->SH, TH->T/S, Z->S), 2.2 for manner match, 3.2 for dissimilar
                                thresh = 1.2 if is_typ else (2.2 if manner_match else 3.2)
                                if margin >= thresh and (is_typ or err_prob >= 0.15):
                                    best_cand = cand_phone
                                    cand_margin = margin
                                    break

                        if best_cand and best_cand.rstrip("012") != target_base:
                            actual_phone = best_cand
                            pred_class_name = "substitution"
                            is_suspicious = True
                            err_prob = max(err_prob, min(0.90, 0.50 + cand_margin * 0.12))

            # 3. Model Prior Fallback if no acoustic substitution triggered
            if not is_suspicious:
                thresh = float(sensitivity_threshold)
                if pred_class_id == ERR_DEL and p_del >= max(0.40, thresh) and p_del > p_correct:
                    actual_phone = "[DELETION]"
                    pred_class_name = "deletion"
                    is_suspicious = True
                elif pred_class_id == ERR_ADD and p_add >= max(0.40, thresh) and p_add > p_correct:
                    actual_phone = "[ADDITION]"
                    pred_class_name = "addition"
                    is_suspicious = True
                elif pred_class_id == ERR_SUB and p_sub >= max(0.45, thresh) and p_sub > p_correct:
                    actual_phone = typs[0] if typs else "[SUBSTITUTION]"
                    pred_class_name = "substitution"
                    is_suspicious = True

            # If target phone equals actual phone, it is NOT an error
            if actual_phone.rstrip("012") == target_phone.rstrip("012") and pred_class_name not in ("deletion", "addition"):
                is_suspicious = False

            # ASHA Diagnosis
            diag: ASHADiagnosis = ASHAPhonologicalAnalyzer.diagnose(
                target_phone=target_phone,
                actual_phone=actual_phone,
                mdd_class=pred_class_name,
                is_word_final=w_ctx.get("is_last", False),
                in_cluster=w_ctx.get("is_cluster", False),
                word=w_ctx.get("word", ""),
            )

            record = {
                "index": i,
                "word": w_ctx.get("word", ""),
                "word_ipa": w_ctx.get("word_ipa", ""),
                "target_phone": target_phone,
                "target_ipa": diag.target_ipa,
                "actual_phone": actual_phone,
                "actual_ipa": diag.actual_ipa,
                "mdd_class": pred_class_name,
                "error_type": pred_class_name,
                "error_probability": round(err_prob, 3),
                "is_suspicious": is_suspicious,
                "rule_id": diag.rule_id,
                "rule_name_vi": diag.rule_name_vi,
                "rule_name_en": diag.rule_name_en,
                "severity": diag.severity,
                "feature_contrast": diag.feature_contrast,
                "articulatory_tip": diag.articulatory_tip,
                "ctc_span": {
                    "start_frame": sp.start_frame if sp else 0,
                    "end_frame": sp.end_frame if sp else 0,
                },
            }

            full_list.append(record)
            if is_suspicious:
                suspicious_list.append(record)

        return {
            "suspicious_phonemes": suspicious_list,
            "all_phonemes": full_list,
            "num_total_phonemes": len(phoneme_tokens),
            "num_suspicious": len(suspicious_list),
            "suspicious_ratio": round(len(suspicious_list) / max(len(phoneme_tokens), 1), 3),
        }


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/models', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/models/checkpoint_utils.py
"""Resolve and load transformer checkpoints from transformer_models/."""

from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, Optional, Union

import torch
import torch.nn as nn

PathLike = Union[str, Path]

DEFAULT_MODEL_DIR = Path("transformer_models")
PRONUNCIATION_NAMES = ("pronunciation.pt", "best_model.pt")
MDD_NAMES = ("l2_mdd_best.pt", "l2_mdd.pt", "best_model.pt")


def _first_existing(base: Path, names: tuple[str, ...]) -> Optional[Path]:
    for name in names:
        p = base / name
        if p.is_file():
            return p
    return None


def _download_pronunciation_from_hub(config: Dict[str, Any]) -> Optional[Path]:
    paths = config.get("paths", {})
    repo_id = paths.get("pronunciation_hf_repo")
    if not repo_id:
        return None

    filename = paths.get("pronunciation_hf_filename", "pronunciation.pt")
    model_dir = Path(paths.get("transformer_models_dir", DEFAULT_MODEL_DIR))
    model_dir.mkdir(parents=True, exist_ok=True)
    target = model_dir / filename

    from huggingface_hub import hf_hub_download

    hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(model_dir))
    return target if target.is_file() else None


def resolve_checkpoint(
    config: Dict[str, Any],
    *,
    model: str,
    explicit: Optional[PathLike] = None,
) -> Path:
    """model: pronunciation | l2_mdd"""
    if explicit:
        p = Path(explicit)
        if p.is_file():
            return p
        raise FileNotFoundError(f"Checkpoint not found: {p}")

    paths = config.get("paths", {})
    key = "pronunciation_checkpoint" if model == "pronunciation" else "l2_mdd_checkpoint"
    configured = paths.get(key)
    if configured and Path(configured).is_file():
        return Path(configured)

    model_dir = Path(paths.get("transformer_models_dir", DEFAULT_MODEL_DIR))
    names = PRONUNCIATION_NAMES if model == "pronunciation" else MDD_NAMES
    found = _first_existing(model_dir, names)
    if found:
        return found

    legacy_dir = Path(
        paths.get("checkpoint_dir", "checkpoints" if model == "pronunciation" else "checkpoints/l2_mdd")
    )
    legacy = legacy_dir / "best_model.pt"
    if legacy.is_file():
        return legacy

    if model == "pronunciation":
        downloaded = _download_pronunciation_from_hub(config)
        if downloaded:
            return downloaded
        found = _first_existing(model_dir, names)
        if found:
            return found

    expected = ", ".join(names)
    raise FileNotFoundError(
        f"No {model} checkpoint. Place weights in {model_dir}/ ({expected}) "
        f"or set paths.{key} in config."
    )


def load_state_dict(path: PathLike, device: Union[str, torch.device] = "cpu") -> dict:
    st = torch.load(path, map_location=device, weights_only=False)
    if isinstance(st, dict) and "model_state_dict" in st:
        return st["model_state_dict"]
    return st


def load_model_weights(
    model: nn.Module,
    config: Dict[str, Any],
    *,
    model_kind: str,
    explicit: Optional[PathLike] = None,
    device: Union[str, torch.device] = "cpu",
    strict: bool = True,
) -> Path:
    path = resolve_checkpoint(config, model=model_kind, explicit=explicit)
    model.load_state_dict(load_state_dict(path, device), strict=strict)
    return path


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/models', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/models/ctc_aligner.py
"""
CTC Forced Alignment for phoneme-level timestamp extraction.

Step 3 of pipeline:
  1. Project WavLM frame features -> CTC log-probs over phoneme vocabulary.
  2. Run torchaudio.functional.forced_align with canonical phoneme sequence.
  3. Convert alignment spans -> pooled phoneme node features for Graph Attention.

Key conversion (alignment -> node features):
  For each phoneme p_i aligned to frame range [t_start, t_end]:
      node_feature_i = mean(frame_hidden[t_start:t_end+1])
  This pools acoustic+linguistic context from the task Transformer output
  into one vector per phoneme, which becomes the initial GAT node embedding.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

# Lazy import: torchaudio may fail on some Windows/Python combos
torchaudio = None
forced_align = None


def _ensure_torchaudio():
    global torchaudio, forced_align
    if torchaudio is not None:
        return True
    try:
        import torchaudio as _ta
        from torchaudio.functional import forced_align as _fa

        torchaudio = _ta
        forced_align = _fa
        return True
    except (ImportError, OSError):
        return False


# espeak-style phoneme set subset (ARPAbet compatible with SpeechOcean762)
DEFAULT_PHONEME_VOCAB = [
    "<pad>", "<unk>", "|",  # blank, unknown, word boundary
    "AA0", "AA1", "AA2", "AE0", "AE1", "AE2", "AH0", "AH1", "AH2",
    "AO0", "AO1", "AO2", "AW0", "AW1", "AW2", "AY0", "AY1", "AY2",
    "B", "CH", "D", "DH", "EH0", "EH1", "EH2", "ER0", "ER1", "ER2",
    "EY0", "EY1", "EY2", "F", "G", "HH", "IH0", "IH1", "IH2",
    "IY0", "IY1", "IY2", "JH", "K", "L", "M", "N", "NG",
    "OW0", "OW1", "OW2", "OY0", "OY1", "OY2", "P", "R", "S", "SH",
    "T", "TH", "UH0", "UH1", "UH2", "UW0", "UW1", "UW2",
    "V", "W", "Y", "Z", "ZH",
]


@dataclass
class PhonemeAlignment:
    """Single phoneme alignment result."""

    phoneme: str
    token_id: int
    start_frame: int
    end_frame: int
    confidence: float


# ARPAbet phone variants sometimes emitted by L2-ARCTIC / non-CMU textgrids
_PHONEME_ALIASES: Dict[str, str] = {
    "AX": "AH0",
    "AX0": "AH0",
    "AX1": "AH0",
    "AX2": "AH0",
    "AXR": "ER0",
    "AXR0": "ER0",
    "AXR1": "ER0",
    "AXR2": "ER0",
    "IX": "IH0",
    "IX0": "IH0",
    "IX1": "IH0",
    "IX2": "IH0",
    "UX": "UW0",
    "UX0": "UW0",
    "UX1": "UW0",
    "UX2": "UW0",
    "DX": "T",
    "NX": "N",
    "EL": "L",
    "EM": "M",
    "EN": "N",
    "ENG": "NG",
    "HV": "HH",
    "WH": "W",
    "Q": "T",
}

_ARPABET_VOWELS = frozenset({
    "AA", "AE", "AH", "AO", "AW", "AY", "EH", "ER", "EY", "IH", "IY", "OW", "OY", "UH", "UW"
})


def normalize_phoneme_token(p: str) -> str:
    """Normalize ARPAbet phoneme string (upper-case, map variants, supply default vowel stress)."""
    if not p:
        return "<unk>"
    s = p.strip()
    if s in ("<pad>", "<unk>", "|"):
        return s
    # Keep alphanumeric only (drops trailing '*', '?', punctuation)
    clean = "".join(ch for ch in s.upper() if ch.isalnum())
    if not clean:
        return "<unk>"
    if clean in _PHONEME_ALIASES:
        return _PHONEME_ALIASES[clean]
    base = clean.rstrip("0123456789")
    if base in _PHONEME_ALIASES:
        return _PHONEME_ALIASES[base]
    if base in _ARPABET_VOWELS:
        if len(clean) > len(base) and clean[-1] in ("0", "1", "2"):
            return f"{base}{clean[-1]}"
        return f"{base}1"
    return base if base else clean


class CTCAligner(nn.Module):
    """
    CTC head + forced alignment to map phoneme sequence -> frame spans.

    Can optionally use a pretrained torchaudio CTC bundle for alignment-only
    mode; default trains a lightweight linear CTC head on WavLM features.
    """

    def __init__(
        self,
        input_dim: int,
        phoneme_vocab: Optional[List[str]] = None,
        use_pretrained_bundle: bool = False,
        blank_id: int = 0,
    ):
        super().__init__()
        self.phoneme_vocab = phoneme_vocab or DEFAULT_PHONEME_VOCAB
        self.token2id = {p: i for i, p in enumerate(self.phoneme_vocab)}
        self.blank_id = blank_id
        self.num_tokens = len(self.phoneme_vocab)

        self.ctc_proj = nn.Linear(input_dim, self.num_tokens)
        self.use_pretrained_bundle = use_pretrained_bundle
        self._bundle = None

        if use_pretrained_bundle and _ensure_torchaudio():
            try:
                # Phoneme ASR model (espeak phonemes) when available
                self._bundle = torchaudio.pipelines.MMS_FA
            except AttributeError:
                self._bundle = None

    def phonemes_to_ids(self, phonemes: List[str]) -> torch.Tensor:
        """Map ARPAbet phoneme strings to token IDs."""
        unk_id = self.token2id["<unk>"]
        ids = []
        for p in phonemes:
            if p in self.token2id:
                ids.append(self.token2id[p])
            else:
                norm = normalize_phoneme_token(p)
                ids.append(self.token2id.get(norm, unk_id))
        return torch.tensor(ids, dtype=torch.long)

    def forward_ctc_logits(self, frame_features: torch.Tensor) -> torch.Tensor:
        """(B, T, D) -> (T, B, C) log-probs (float32) for torch.nn.functional.ctc_loss."""
        logits = self.ctc_proj(frame_features).float()
        log_probs = F.log_softmax(logits, dim=-1)
        return log_probs.transpose(0, 1)

    def _align_log_probs(self, frame_features: torch.Tensor) -> torch.Tensor:
        """(T, D) -> (1, T, C) log-probs (float32) for torchaudio forced_align (batch-first)."""
        logits = self.ctc_proj(frame_features.unsqueeze(0)).float()
        return F.log_softmax(logits, dim=-1)

    def _run_forced_align(
        self,
        log_probs: torch.Tensor,
        targets: torch.Tensor,
        input_lengths: torch.Tensor,
        target_lengths: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """forced_align wants (B, T, C) with B=1; GPU kernel errors on wrong layout."""
        assert log_probs.dim() == 3 and log_probs.shape[0] == 1
        try:
            return forced_align(
                log_probs.float(),
                targets.unsqueeze(0),
                input_lengths,
                target_lengths,
                blank=self.blank_id,
            )
        except RuntimeError:
            return forced_align(
                log_probs.float().cpu(),
                targets.unsqueeze(0).cpu(),
                input_lengths.cpu(),
                target_lengths.cpu(),
                blank=self.blank_id,
            )

    def align_utterance(
        self,
        frame_features: torch.Tensor,
        phonemes: List[str],
        frame_lengths: Optional[int] = None,
        log_probs: Optional[torch.Tensor] = None,
    ) -> Tuple[List[PhonemeAlignment], torch.Tensor]:
        """
        Force-align one utterance.

        Args:
            frame_features: (T, D) single utterance frame features.
            phonemes: canonical ARPAbet phoneme list from transcript/CMUdict.
            frame_lengths: number of valid frames T.
            log_probs: optional precomputed (1, T, C) log-probs.

        Returns:
            alignments: list of PhonemeAlignment with frame spans.
            node_features: (num_phonemes, D) pooled features for GAT nodes.
        """
        if frame_lengths is None:
            frame_lengths = frame_features.shape[0]
        frame_lengths = min(int(frame_lengths), int(frame_features.shape[0]))

        T, D = frame_features.shape
        if len(phonemes) == 0 or frame_lengths == 0:
            return [], frame_features.new_zeros(0, D)

        targets = self.phonemes_to_ids(phonemes).to(frame_features.device)
        t_list = targets.tolist()
        min_ctc_frames = len(t_list) + sum(1 for a, b in zip(t_list[:-1], t_list[1:]) if a == b)

        if not _ensure_torchaudio() or forced_align is None or frame_lengths < min_ctc_frames:
            return self._uniform_align(frame_features[:frame_lengths], phonemes)

        if log_probs is None:
            lp = self._align_log_probs(frame_features[:frame_lengths])  # (1, T, C)
        else:
            lp = log_probs if log_probs.dim() == 3 else log_probs.unsqueeze(0)
            lp = lp[:, :frame_lengths, :].float()

        input_lengths = torch.tensor([frame_lengths], device=lp.device)
        target_lengths = torch.tensor([len(targets)], device=lp.device)

        try:
            aligned_tokens, _align_scores = self._run_forced_align(
                lp, targets.to(lp.device), input_lengths, target_lengths
            )
        except RuntimeError:
            return self._uniform_align(frame_features[:frame_lengths], phonemes)

        if aligned_tokens.dim() > 1:
            aligned = aligned_tokens[0, :frame_lengths]
        else:
            aligned = aligned_tokens[:frame_lengths]

        spans = self._tokens_to_spans(aligned, targets, phonemes)
        node_features = self._pool_node_features(frame_features[:frame_lengths], spans)
        return spans, node_features

    def _tokens_to_spans(
        self,
        aligned: torch.Tensor,
        targets: torch.Tensor,
        phonemes: List[str],
    ) -> List[PhonemeAlignment]:
        """
        Parse per-frame CTC alignment into phoneme start/end spans with smooth interpolation.

        torchaudio.functional.forced_align outputs the aligned TOKEN ID (or blank_id=0) per frame
        in monotonic target order — NOT target position index 0..N-1.
        We extract maximal contiguous non-blank token runs and match them monotonically to targets.
        """
        spans: List[PhonemeAlignment] = []
        target_list = targets.tolist()
        N = len(phonemes)
        T = int(aligned.shape[0])

        if N == 0:
            return spans

        # 1. Extract contiguous non-blank token runs: (token_id, start_f, end_f)
        arr = aligned.tolist()
        runs: List[Tuple[int, int, int]] = []
        idx = 0
        while idx < T:
            tok = int(arr[idx])
            if tok != self.blank_id:
                s_f = idx
                while idx + 1 < T and int(arr[idx + 1]) == tok:
                    idx += 1
                runs.append((tok, s_f, idx))
            idx += 1

        # 2. Monotonically match runs to target_list[0..N-1]
        raw_spans: List[Optional[Tuple[int, int, float]]] = [None] * N
        r_idx = 0
        for i in range(N):
            tid = int(target_list[i])
            if r_idx < len(runs) and runs[r_idx][0] == tid:
                _, s_f, e_f = runs[r_idx]
                conf = min(1.0, (e_f - s_f + 1) / 3.0)
                raw_spans[i] = (s_f, e_f, conf)
                r_idx += 1
            else:
                # Lookahead up to a few runs in case a middle token run shifted/dropped
                found = -1
                for k in range(r_idx, min(len(runs), r_idx + 4)):
                    if runs[k][0] == tid:
                        found = k
                        break
                if found != -1:
                    _, s_f, e_f = runs[found]
                    conf = min(1.0, (e_f - s_f + 1) / 3.0)
                    raw_spans[i] = (s_f, e_f, conf)
                    r_idx = found + 1

        # Anchors: phonemes with valid CTC frame alignment
        anchors = [(i, raw_spans[i][0], raw_spans[i][1], raw_spans[i][2]) for i in range(N) if raw_spans[i] is not None]

        if not anchors:
            # Complete fallback: uniform split across all frames
            step = T / max(1, N)
            for i in range(N):
                s_f = int(i * step)
                e_f = min(T - 1, int((i + 1) * step))
                spans.append(PhonemeAlignment(phonemes[i], target_list[i], s_f, e_f, 0.5))
            return spans

        final_spans = [None] * N
        for idx_a, s_f, e_f, conf in anchors:
            final_spans[idx_a] = (s_f, e_f, conf)

        # Interpolate before first anchor
        first_idx, first_s, _, _ = anchors[0]
        if first_idx > 0:
            step = max(1, first_s // first_idx)
            for k in range(first_idx):
                s = int(k * step)
                e = min(first_s, int((k + 1) * step))
                final_spans[k] = (max(0, min(T - 1, s)), max(0, min(T - 1, e)), 0.0)

        # Interpolate between consecutive anchors
        for a_idx in range(len(anchors) - 1):
            i_curr, _, e_curr, _ = anchors[a_idx]
            i_next, s_next, _, _ = anchors[a_idx + 1]
            gap_count = i_next - i_curr - 1
            if gap_count > 0:
                gap_start = e_curr
                gap_end = s_next
                if gap_end <= gap_start:
                    gap_end = gap_start + gap_count
                gap_len = max(gap_count, gap_end - gap_start)
                step = gap_len / gap_count
                for step_i in range(gap_count):
                    target_i = i_curr + 1 + step_i
                    s = int(gap_start + step_i * step)
                    e = int(gap_start + (step_i + 1) * step)
                    s = max(0, min(T - 1, s))
                    e = max(s, min(T - 1, e))
                    final_spans[target_i] = (s, e, 0.0)

        # Interpolate after last anchor
        last_idx, _, last_e, _ = anchors[-1]
        if last_idx < N - 1:
            rem_count = N - 1 - last_idx
            rem_frames = max(rem_count, T - last_e)
            step = rem_frames / rem_count
            for step_i in range(rem_count):
                target_i = last_idx + 1 + step_i
                s = int(last_e + step_i * step)
                e = int(last_e + (step_i + 1) * step)
                s = max(0, min(T - 1, s))
                e = max(s, min(T - 1, e))
                final_spans[target_i] = (s, e, 0.0)

        for i in range(N):
            s_f, e_f, conf = final_spans[i]
            spans.append(PhonemeAlignment(phonemes[i], target_list[i], s_f, e_f, conf))

        return spans

    def _pool_node_features(
        self,
        frame_features: torch.Tensor,
        spans: List[PhonemeAlignment],
    ) -> torch.Tensor:
        """
        Pool frame hidden states over [t_start, t_end] for each phoneme.

        This is the critical bridge from CTC alignment to Graph Attention:
        each phoneme node receives a fixed-size embedding regardless of
        how many frames it spans.
        """
        nodes = []
        T = frame_features.shape[0]
        for span in spans:
            s = max(0, span.start_frame)
            e = min(T - 1, span.end_frame)
            if s <= e:
                pooled = frame_features[s : e + 1].mean(dim=0)
            else:
                pooled = frame_features.mean(dim=0)
            nodes.append(pooled)
        return torch.stack(nodes, dim=0) if nodes else frame_features.new_zeros(0, frame_features.shape[-1])

    def _uniform_align(
        self,
        frame_features: torch.Tensor,
        phonemes: List[str],
    ) -> Tuple[List[PhonemeAlignment], torch.Tensor]:
        """Fallback equal-split alignment when forced_align is unavailable."""
        T, D = frame_features.shape
        n = max(len(phonemes), 1)
        chunk = T // n
        spans = []
        for i, ph in enumerate(phonemes):
            s = i * chunk
            e = min(T - 1, (i + 1) * chunk - 1) if i < n - 1 else T - 1
            norm = normalize_phoneme_token(ph) if ph not in self.token2id else ph
            tid = self.token2id.get(norm, self.token2id["<unk>"])
            spans.append(
                PhonemeAlignment(ph, tid, s, e, 1.0)
            )
        node_features = self._pool_node_features(frame_features, spans)
        return spans, node_features

    def batch_align(
        self,
        frame_features: torch.Tensor,
        phoneme_lists: List[List[str]],
        frame_lengths: torch.Tensor,
        batch_log_probs: Optional[torch.Tensor] = None,
    ) -> List[Tuple[List[PhonemeAlignment], torch.Tensor]]:
        """Align a batch; returns per-utterance (spans, node_features)."""
        results = []
        B = frame_features.shape[0]
        for b in range(B):
            T_b = int(frame_lengths[b].item())
            lp_b = batch_log_probs[b : b + 1, :T_b] if batch_log_probs is not None else None
            spans, nodes = self.align_utterance(
                frame_features[b, :T_b],
                phoneme_lists[b],
                T_b,
                log_probs=lp_b,
            )
            results.append((spans, nodes))
        return results

    def ctc_loss(
        self,
        frame_features: torch.Tensor,
        phoneme_lists: List[List[str]],
        frame_lengths: torch.Tensor,
        batch_log_probs: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """Auxiliary CTC loss for training the alignment head."""
        if batch_log_probs is not None:
            log_probs = batch_log_probs.float().transpose(0, 1)
        else:
            log_probs = self.forward_ctc_logits(frame_features)
        targets_list = []
        target_lengths = []
        for phs in phoneme_lists:
            t = self.phonemes_to_ids(phs)
            targets_list.append(t)
            target_lengths.append(len(t))
        if not targets_list or sum(target_lengths) == 0:
            return frame_features.new_zeros(())
        targets = torch.cat(targets_list).to(frame_features.device)
        target_lengths_t = torch.tensor(target_lengths, device=frame_features.device)
        loss = F.ctc_loss(
            log_probs,
            targets,
            frame_lengths,
            target_lengths_t,
            blank=self.blank_id,
            zero_infinity=True,
        )
        return loss


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/models', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/models/multitask_heads.py
"""
Multi-task regression heads (GOPT-inspired).

Step 5: parallel heads for pronunciation aspects at 3 granularities.

Architecture mirrors GOPT (ICASSP 2022):
  - Each head: LayerNorm -> Linear(hidden, 1) regression.
  - Utterance heads: applied on utterance-level pooled representation.
  - Word heads: applied on mean-pool of phoneme nodes per word.
  - Phoneme heads: applied on each GAT output node.

Aspects mapped to SpeechOcean762 labels:
  Utterance: accuracy, fluency, completeness, prosodic, total
  Word:      accuracy, stress, total
  Phoneme:   accuracy (phones-accuracy)
"""

from __future__ import annotations

from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn


class RegressionHead(nn.Module):
    """Single-aspect regression head with layer norm (GOPT style)."""

    def __init__(self, input_dim: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Dropout(dropout),
            nn.Linear(input_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


class MultiTaskHeads(nn.Module):
    """Collection of aspect-specific heads at phoneme/word/utterance levels."""

    def __init__(
        self,
        input_dim: int,
        utterance_aspects: Optional[List[str]] = None,
        word_aspects: Optional[List[str]] = None,
        phoneme_aspects: Optional[List[str]] = None,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.utterance_aspects = utterance_aspects or [
            "accuracy", "fluency", "completeness", "prosodic", "total"
        ]
        self.word_aspects = word_aspects or ["accuracy", "stress", "total"]
        self.phoneme_aspects = phoneme_aspects or ["accuracy"]
        self.input_dim = input_dim

        self.utt_heads = nn.ModuleDict(
            {a: RegressionHead(input_dim, dropout) for a in self.utterance_aspects}
        )
        self.word_heads = nn.ModuleDict(
            {a: RegressionHead(input_dim, dropout) for a in self.word_aspects}
        )
        self.phoneme_heads = nn.ModuleDict(
            {a: RegressionHead(input_dim, dropout) for a in self.phoneme_aspects}
        )

    def pool_utterance(self, phoneme_embeddings: torch.Tensor) -> torch.Tensor:
        """Mean-pool all phoneme nodes -> utterance representation."""
        if phoneme_embeddings.shape[0] == 0:
            return phoneme_embeddings.new_zeros(self.input_dim)
        return phoneme_embeddings.mean(dim=0)

    def pool_words(
        self,
        phoneme_embeddings: torch.Tensor,
        word_phone_ranges: List[Tuple[int, int]],
    ) -> torch.Tensor:
        """Mean-pool phoneme nodes per word -> (num_words, D)."""
        word_embs = []
        for start, end in word_phone_ranges:
            if start < end:
                word_embs.append(phoneme_embeddings[start:end].mean(dim=0))
            else:
                word_embs.append(phoneme_embeddings.new_zeros(self.input_dim))
        return torch.stack(word_embs, dim=0) if word_embs else phoneme_embeddings.new_zeros(0, self.input_dim)

    def forward_single(
        self,
        phoneme_embeddings: torch.Tensor,
        word_phone_ranges: List[Tuple[int, int]],
    ) -> Dict[str, torch.Tensor]:
        """
        Predict all aspects for one utterance.

        Returns dict with keys:
          utterance_{aspect}, word_{aspect} (W,), phoneme_{aspect} (N,)
        """
        utt_repr = self.pool_utterance(phoneme_embeddings)
        word_repr = self.pool_words(phoneme_embeddings, word_phone_ranges)

        out: Dict[str, torch.Tensor] = {}
        for aspect, head in self.utt_heads.items():
            out[f"utterance_{aspect}"] = head(utt_repr.unsqueeze(0)).squeeze(0)
        for aspect, head in self.word_heads.items():
            if word_repr.shape[0] > 0:
                out[f"word_{aspect}"] = head(word_repr)
            else:
                out[f"word_{aspect}"] = phoneme_embeddings.new_zeros(0)
        for aspect, head in self.phoneme_heads.items():
            if phoneme_embeddings.shape[0] > 0:
                out[f"phoneme_{aspect}"] = head(phoneme_embeddings)
            else:
                out[f"phoneme_{aspect}"] = phoneme_embeddings.new_zeros(0)
        return out

    def forward_batch(
        self,
        phoneme_embeddings_list: List[torch.Tensor],
        word_phone_ranges_list: List[List[Tuple[int, int]]],
    ) -> List[Dict[str, torch.Tensor]]:
        return [
            self.forward_single(pe, wr)
            for pe, wr in zip(phoneme_embeddings_list, word_phone_ranges_list)
        ]


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/models', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/models/phoneme_graph.py
"""
Graph Attention Network over phoneme nodes.

Step 4: build per-utterance phoneme graph and apply GATv2Conv layers.

Graph construction per utterance:
  - Nodes: one per phoneme instance, initial feature = pooled frame embedding
    from CTC alignment (see ctc_aligner._pool_node_features).
  - Edges:
      * Sequential: (i, i+1) bidirectional — captures coarticulation context.
      * Same-word: all phoneme pairs within one word — captures lexical stress.
      * Same-syllable (optional): pairs within syllable group.

After GAT message passing, each node has context-aware embedding used by
multi-task regression heads at phoneme level; word/utterance levels pool
these node embeddings.
"""

from __future__ import annotations

from typing import List, Optional, Tuple

import torch
import torch.nn as nn

try:
    from torch_geometric.nn import GATv2Conv
except ImportError:
    GATv2Conv = None


class PhonemeGraphNetwork(nn.Module):
    """GATv2-based phoneme graph encoder."""

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 256,
        num_layers: int = 2,
        num_heads: int = 4,
        dropout: float = 0.1,
        edge_sequential: bool = True,
        edge_same_word: bool = True,
        edge_same_syllable: bool = False,
    ):
        super().__init__()
        if GATv2Conv is None:
            raise ImportError("torch_geometric is required for PhonemeGraphNetwork")

        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.edge_sequential = edge_sequential
        self.edge_same_word = edge_same_word
        self.edge_same_syllable = edge_same_syllable

        self.gat_layers = nn.ModuleList()
        for i in range(num_layers):
            in_ch = hidden_dim
            out_ch = hidden_dim // num_heads
            self.gat_layers.append(
                GATv2Conv(
                    in_channels=in_ch,
                    out_channels=out_ch,
                    heads=num_heads,
                    dropout=dropout,
                    concat=True,
                )
            )
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.output_dim = hidden_dim

    @staticmethod
    def build_edge_index(
        num_phonemes: int,
        word_phone_ranges: List[Tuple[int, int]],
        sequential: bool = True,
        same_word: bool = True,
        same_syllable: bool = False,
    ) -> torch.Tensor:
        """
        Build COO edge_index (2, E) for one utterance.

        Args:
            num_phonemes: total phoneme count.
            word_phone_ranges: list of (start, end) exclusive indices per word.
        """
        edges = set()

        if sequential:
            for i in range(num_phonemes - 1):
                edges.add((i, i + 1))
                edges.add((i + 1, i))

        if same_word:
            for start, end in word_phone_ranges:
                for i in range(start, end):
                    for j in range(start, end):
                        if i != j:
                            edges.add((i, j))

        if same_syllable:
            # simple heuristic: split each word's phones in half
            for start, end in word_phone_ranges:
                mid = (start + end) // 2
                for i in range(start, mid):
                    for j in range(start, mid):
                        if i != j:
                            edges.add((i, j))
                for i in range(mid, end):
                    for j in range(mid, end):
                        if i != j:
                            edges.add((i, j))

        if not edges:
            # self-loop fallback for single phoneme
            edges.add((0, 0))

        src, dst = zip(*edges)
        return torch.tensor([src, dst], dtype=torch.long)

    def forward_single(
        self,
        node_features: torch.Tensor,
        word_phone_ranges: List[Tuple[int, int]],
    ) -> torch.Tensor:
        """
        Args:
            node_features: (N, D) pooled phoneme embeddings from CTC align step.
            word_phone_ranges: phoneme index ranges per word.

        Returns:
            (N, hidden_dim) context-enriched phoneme embeddings.
        """
        x = self.input_proj(node_features)
        edge_index = self.build_edge_index(
            node_features.shape[0],
            word_phone_ranges,
            self.edge_sequential,
            self.edge_same_word,
            self.edge_same_syllable,
        ).to(node_features.device)

        for gat in self.gat_layers:
            x = gat(x, edge_index)
            x = self.dropout(torch.relu(x))
        return self.norm(x)

    def forward_batch(
        self,
        node_features_list: List[torch.Tensor],
        word_phone_ranges_list: List[List[Tuple[int, int]]],
    ) -> List[torch.Tensor]:
        """Process variable-size graphs per utterance."""
        outputs = []
        for nodes, ranges in zip(node_features_list, word_phone_ranges_list):
            if nodes.shape[0] == 0:
                outputs.append(nodes.new_zeros(0, self.output_dim))
            else:
                outputs.append(self.forward_single(nodes, ranges))
        return outputs


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/models', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/models/pronunciation_model.py
"""
End-to-end Pronunciation Assessment model.

Full pipeline:
  waveform -> WavLM -> TaskTransformer -> CTC align -> GAT graph -> multi-task heads
"""

from __future__ import annotations

from typing import Any, Dict, List, Optional

import torch
import torch.nn as nn

from .wavlm_encoder import WavLMEncoder
from .transformer_encoder import TaskTransformerEncoder
from .ctc_aligner import CTCAligner
from .phoneme_graph import PhonemeGraphNetwork
from .multitask_heads import MultiTaskHeads


class PronunciationAssessmentModel(nn.Module):
    """Full pronunciation assessment pipeline."""

    def __init__(self, config: Dict[str, Any]):
        super().__init__()
        wavlm_cfg = config.get("wavlm", {})
        trans_cfg = config.get("transformer", {})
        ctc_cfg = config.get("ctc_align", {})
        graph_cfg = config.get("phoneme_graph", {})
        mt_cfg = config.get("multitask", {})

        self.wavlm = WavLMEncoder(
            model_name=wavlm_cfg.get("model_name", "microsoft/wavlm-large"),
            freeze=wavlm_cfg.get("freeze", True),
            use_lora=wavlm_cfg.get("use_lora", False),
            lora_r=wavlm_cfg.get("lora_r", 8),
            lora_alpha=wavlm_cfg.get("lora_alpha", 16),
            lora_dropout=wavlm_cfg.get("lora_dropout", 0.05),
            lora_target_modules=wavlm_cfg.get("lora_target_modules"),
        )
        d = self.wavlm.output_dim

        self.task_transformer = TaskTransformerEncoder(
            input_dim=d,
            num_layers=trans_cfg.get("num_layers", 3),
            num_heads=trans_cfg.get("num_heads", 8),
            ff_dim=trans_cfg.get("ff_dim", 3072),
            dropout=trans_cfg.get("dropout", 0.1),
            max_seq_len=trans_cfg.get("max_seq_len", 2000),
        )

        self.ctc_aligner = CTCAligner(
            input_dim=d,
            use_pretrained_bundle=not ctc_cfg.get("use_wavlm_ctc_head", True),
        )

        graph_hidden = graph_cfg.get("hidden_dim", 256)
        self.phoneme_graph = PhonemeGraphNetwork(
            input_dim=d,
            hidden_dim=graph_hidden,
            num_layers=graph_cfg.get("num_gat_layers", 2),
            num_heads=graph_cfg.get("num_heads", 4),
            dropout=graph_cfg.get("dropout", 0.1),
            edge_sequential=graph_cfg.get("edge_types", {}).get("sequential", True),
            edge_same_word=graph_cfg.get("edge_types", {}).get("same_word", True),
            edge_same_syllable=graph_cfg.get("edge_types", {}).get("same_syllable", False),
        )

        self.multitask_heads = MultiTaskHeads(
            input_dim=graph_hidden,
            utterance_aspects=mt_cfg.get("utterance_aspects"),
            word_aspects=mt_cfg.get("word_aspects"),
            phoneme_aspects=mt_cfg.get("phoneme_aspects"),
            dropout=mt_cfg.get("dropout", 0.1),
        )

        self.sample_rate = config.get("train", {}).get("dataset", {}).get("sample_rate", 16000)

    def _frame_lengths_from_wave(self, wav_lengths: torch.Tensor) -> torch.Tensor:
        """Convert sample lengths to WavLM frame lengths using model conv math."""
        return self.wavlm.frame_lengths_from_samples(wav_lengths)

    def forward(
        self,
        waveforms: torch.Tensor,
        wav_lengths: torch.Tensor,
        phoneme_tokens: List[List[str]],
        word_phone_ranges: List[List[tuple]],
        return_alignments: bool = False,
    ) -> Dict[str, Any]:
        """
        Forward pass for a batch.

        Args:
            waveforms: (B, samples) padded.
            wav_lengths: (B,) actual sample counts.
            phoneme_tokens: list of phoneme strings per utterance.
            word_phone_ranges: word -> phoneme index ranges.

        Returns:
            dict with per-utterance predictions, optional alignments, ctc_loss.
        """
        # Step 1: WavLM features
        frame_feats = self.wavlm(waveforms, wav_lengths=wav_lengths)  # (B, T, D)
        frame_lengths = self._frame_lengths_from_wave(wav_lengths)

        # padding mask for transformer
        T = frame_feats.shape[1]
        pad_mask = torch.arange(T, device=waveforms.device).unsqueeze(0) >= frame_lengths.unsqueeze(1)

        # Step 2: task transformer
        frame_feats = self.task_transformer(frame_feats, src_key_padding_mask=pad_mask)

        # Step 3: CTC alignment -> phoneme node features
        align_results = self.ctc_aligner.batch_align(
            frame_feats, phoneme_tokens, frame_lengths
        )
        node_features_list = [nodes for _, nodes in align_results]
        alignments_list = [spans for spans, _ in align_results] if return_alignments else None

        # Step 4: graph attention
        graph_out = self.phoneme_graph.forward_batch(
            node_features_list, word_phone_ranges
        )

        # Step 5: multi-task heads
        predictions = self.multitask_heads.forward_batch(graph_out, word_phone_ranges)

        # auxiliary CTC loss
        ctc_loss = self.ctc_aligner.ctc_loss(frame_feats, phoneme_tokens, frame_lengths)

        out = {
            "predictions": predictions,
            "ctc_loss": ctc_loss,
            "graph_embeddings": graph_out,
        }
        if return_alignments:
            out["alignments"] = alignments_list
        return out

    @classmethod
    def from_config_path(cls, path: str) -> "PronunciationAssessmentModel":
        import yaml

        with open(path, encoding="utf-8") as f:
            config = yaml.safe_load(f)
        return cls(config)


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/models', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/models/pronunciation_scorer.py
"""
Pronunciation score aggregation (0-10 scale).

Step 6: combine multi-task head outputs into interpretable final scores.
SpeechOcean762 provides utterance-level scores (Total, Accuracy, Fluency, Prosodic).
L2-MDD provides full-phoneme error scanning with ASHA distinctive articulatory rules.
"""

from __future__ import annotations

from typing import Any, Dict, List, Optional, Tuple

import torch


# ── Standard ARPAbet to IPA Mapping ─────────────────────────────────
ARPABET_TO_IPA: Dict[str, str] = {
    # Vowels & Diphthongs
    "AA": "ɑː",
    "AE": "æ",
    "AH0": "ə",
    "AH": "ʌ",
    "AH1": "ʌ",
    "AH2": "ʌ",
    "AO": "ɔː",
    "AW": "aʊ",
    "AY": "aɪ",
    "EH": "e",
    "ER": "ɜːr",
    "ER0": "ər",
    "ER1": "ɜːr",
    "ER2": "ɜːr",
    "EY": "eɪ",
    "IH": "ɪ",
    "IH0": "ɪ",
    "IH1": "ɪ",
    "IH2": "ɪ",
    "IY": "iː",
    "IY0": "i",
    "IY1": "iː",
    "IY2": "iː",
    "OW": "oʊ",
    "OY": "ɔɪ",
    "UH": "ʊ",
    "UW": "uː",
    # Consonants
    "B": "b",
    "CH": "tʃ",
    "D": "d",
    "DH": "ð",
    "F": "f",
    "G": "ɡ",
    "HH": "h",
    "JH": "dʒ",
    "K": "k",
    "L": "l",
    "M": "m",
    "N": "n",
    "NG": "ŋ",
    "P": "p",
    "R": "r",
    "S": "s",
    "SH": "ʃ",
    "T": "t",
    "TH": "θ",
    "V": "v",
    "W": "w",
    "Y": "j",
    "Z": "z",
    "ZH": "ʒ",
}

def phone_to_ipa(phoneme: str) -> str:
    """Convert ARPAbet phoneme (e.g. 'TH', 'AE1') to standard IPA symbol (e.g. 'θ', 'æ')."""
    if not phoneme:
        return ""
    if phoneme in ARPABET_TO_IPA:
        return ARPABET_TO_IPA[phoneme]
    base = phoneme.rstrip("012")
    return ARPABET_TO_IPA.get(base, base.lower())

def format_phone_ipa(phoneme: str) -> str:
    """Format single phoneme in standard IPA notation e.g. /θ/, /ð/."""
    ipa = phone_to_ipa(phoneme)
    return f"/{ipa}/" if ipa else f"/{phoneme}/"

def phones_to_word_ipa(phones: List[str]) -> str:
    """Convert sequence of ARPAbet phonemes to standard dictionary word IPA.
    
    e.g. ['TH', 'IH1', 'NG', 'K'] -> '/θɪŋk/'
         ['W', 'EH1', 'DH', 'ER0'] -> '/ˈweðər/'
    """
    if not phones:
        return ""
    vowels = {"AA", "AE", "AH", "AO", "AW", "AY", "EH", "ER", "EY", "IH", "IY", "OW", "OY", "UH", "UW"}
    vowel_indices = [i for i, p in enumerate(phones) if p.rstrip("012") in vowels]
    primary_stress_idx = next((i for i, p in enumerate(phones) if len(p) > 1 and p[-1] == "1"), None)
    
    stress_mark_at = None
    if len(vowel_indices) > 1 and primary_stress_idx is not None:
        k = primary_stress_idx
        while k > 0 and (phones[k - 1].rstrip("012") not in vowels):
            k -= 1
        stress_mark_at = k
        
    parts = []
    for i, p in enumerate(phones):
        if stress_mark_at is not None and i == stress_mark_at:
            parts.append("ˈ")
        parts.append(phone_to_ipa(p))
        
    return f"/{''.join(parts)}/"


from .asha_phonology import (
    ASHAPhonologicalAnalyzer,
    ASHADiagnosis,
    get_features,
    ArticulatoryFeatures,
)


def _get_phoneme_tip(phoneme: str) -> str:
    """Return clinical-grade articulatory placement tip using ASHA distinctive features."""
    if not phoneme:
        return ""
    feat = get_features(phoneme)
    if not feat:
        return ""
    if feat.is_vowel:
        tense = "căng cơ má" if feat.vowel_tense else "thả lỏng cơ miệng"
        return f"{feat.name_vi}: chú ý độ mở vòm miệng và {tense}."
    return f"{feat.name_vi}: vị trí cấu âm tại {feat.place}, phương thức cấu âm {feat.manner} ({'rung thanh quản' if feat.voicing == 'voiced' else 'không rung thanh quản'})."


class PronunciationScorer:
    """
    Aggregate multi-granularity predictions into final 0-10 scores.

    Training targets are normalized to 0-2 (GOPT convention); this class
    denormalizes back to SpeechOcean762's 0-10 scale for reporting.
    Supports dynamic affine calibration, duration-aware bias correction,
    and phoneme-level re-scaling.
    """

    def __init__(
        self,
        score_scale: float = 5.0,
        weights: Optional[Dict[str, float]] = None,
        phoneme_low_threshold: float = 1.4,
        calibration: Optional[Dict[str, Any]] = None,
    ):
        self.score_scale = score_scale
        self.weights = weights or {
            "utterance_total": 0.5,
            "word_total": 0.25,
            "phoneme_accuracy": 0.25,
        }
        self.phoneme_low_threshold = phoneme_low_threshold
        self.calibration = calibration or {}

    def to_display_scale(
        self,
        score: float,
        *,
        is_utterance: bool = False,
        is_phone: bool = False,
        duration_sec: Optional[float] = None,
    ) -> float:
        """Map 0-2 normalized score -> 0-10 display scale.
        
        Applies affine calibration and duration-aware adjustment when enabled
        in config to stabilize predictions across varying clip lengths and quality levels.
        """
        raw = score * self.score_scale
        cal_cfg = self.calibration if self.calibration.get("enabled", False) else None

        if cal_cfg:
            if is_phone:
                phone_slope = float(cal_cfg.get("phone_slope", 7.0))
                phone_offset = float(cal_cfg.get("phone_offset", -4.0))
                raw = phone_slope * score + phone_offset
            elif is_utterance:
                slope = float(cal_cfg.get("slope", 1.39))
                offset = float(cal_cfg.get("offset", -3.20))
                cal = slope * raw + offset
                short_sec = float(cal_cfg.get("short_duration_sec", 3.5))
                factor = float(cal_cfg.get("short_duration_factor", 0.75))
                if duration_sec is not None and duration_sec < short_sec and raw < 8.2:
                    cal -= factor * (short_sec - duration_sec)

                # Long duration adjustment (compensates for embedding dispersion on long clips > 4.0s):
                long_sec = float(cal_cfg.get("long_duration_sec", 4.0))
                long_factor = float(cal_cfg.get("long_duration_factor", 0.28))
                max_boost = float(cal_cfg.get("long_duration_max_boost", 2.5))
                if duration_sec is not None and duration_sec > long_sec:
                    cal += min(max_boost, long_factor * (duration_sec - long_sec))
                raw = cal

        # Clamp to [0, 10] range
        clamped = min(10.0, max(0.0, raw))
        # Minimum floor to avoid misleading 0-scores for valid predictions
        if score > 0.01:
            clamped = max(0.5, clamped)
        return round(clamped, 2)

    def aggregate_utterance(
        self,
        predictions: Dict[str, torch.Tensor],
        duration_sec: Optional[float] = None,
    ) -> Dict[str, float]:
        """Build utterance-level score dict on 0-10 scale."""
        result = {}
        for key, val in predictions.items():
            if key.startswith("utterance_"):
                aspect = key.replace("utterance_", "")
                if isinstance(val, torch.Tensor):
                    val = float(val.detach().cpu().item())
                result[aspect] = self.to_display_scale(
                    val, is_utterance=True, duration_sec=duration_sec
                )

        # Multi-granularity anchor for Accuracy on longer utterances:
        # Prevents utterance pooling dispersion from degrading phoneme accuracy on long sentences.
        if "phoneme_accuracy" in predictions:
            pa = predictions["phoneme_accuracy"]
            if isinstance(pa, torch.Tensor) and pa.numel() > 0:
                pa_val = float(pa.mean().detach().cpu().item())
                pa_disp = self.to_display_scale(pa_val, is_phone=True)
                if "accuracy" in result:
                    if duration_sec is not None and duration_sec > 4.5:
                        w_pa = min(0.40, 0.15 + (duration_sec - 4.5) * 0.02)
                    else:
                        w_pa = 0.15
                    result["accuracy"] = round(min(10.0, max(0.5, (1.0 - w_pa) * result["accuracy"] + w_pa * pa_disp)), 2)

        # Multi-granularity anchor for Total score:
        # Total is anchored with word_total and phoneme_accuracy via final_score.
        calc_final = self.final_score(predictions, duration_sec=duration_sec)
        if "total" in result:
            w_final = min(0.60, 0.40 + (max(0.0, (duration_sec or 0.0) - 4.0) * 0.02))
            result["total"] = round(min(10.0, max(0.5, (1.0 - w_final) * result["total"] + w_final * calc_final)), 2)
        else:
            result["total"] = calc_final

        return result

    def final_score(
        self,
        predictions: Dict[str, torch.Tensor],
        duration_sec: Optional[float] = None,
    ) -> float:
        """
        Weighted combination of total/accuracy signals -> single 0-10 score.
        """
        parts = []
        w_sum = 0.0

        w_utt = float(self.weights.get("utterance_total", 0.5))
        w_word = float(self.weights.get("word_total", 0.25))
        w_phone = float(self.weights.get("phoneme_accuracy", 0.25))

        # Dynamically shift weight towards word and phoneme models on long clips:
        if duration_sec is not None and duration_sec > 4.5:
            shift = min(0.20, (duration_sec - 4.5) * 0.02)
            w_utt = max(0.30, w_utt - shift)
            w_word += shift / 2.0
            w_phone += shift / 2.0

        if "utterance_total" in predictions:
            v = predictions["utterance_total"]
            v = float(v.detach().cpu().item()) if isinstance(v, torch.Tensor) else v
            parts.append(w_utt * self.to_display_scale(v, is_utterance=True, duration_sec=duration_sec))
            w_sum += w_utt

        if "word_total" in predictions:
            wt = predictions["word_total"]
            if isinstance(wt, torch.Tensor) and wt.numel() > 0:
                v = float(wt.mean().detach().cpu().item())
                parts.append(w_word * self.to_display_scale(v, is_utterance=True, duration_sec=duration_sec))
                w_sum += w_word

        if "phoneme_accuracy" in predictions:
            pa = predictions["phoneme_accuracy"]
            if isinstance(pa, torch.Tensor) and pa.numel() > 0:
                v = float(pa.mean().detach().cpu().item())
                parts.append(w_phone * self.to_display_scale(v, is_phone=True))
                w_sum += w_phone

        if w_sum == 0:
            return 0.0
        score = sum(parts) / w_sum
        # Minimum floor for valid predictions
        if score > 0 and score < 0.5:
            score = 0.5
        return round(score, 2)

    def find_errors(
        self,
        predictions: Dict[str, torch.Tensor],
        phoneme_tokens: List[str],
        word_texts: List[str],
        word_phone_ranges: List[tuple],
        alignments: Optional[List[dict]] = None,
        threshold: Optional[float] = None,
    ) -> Dict[str, List[dict]]:
        """
        Identify low-scoring phonemes/words for feedback, grouped by severity,
        with standard IPA and word IPA for clear cross-referencing.

        Returns:
            {"phonemes": [...], "words": [...]} with scores on 0-10 scale and severity.
        """
        thr = threshold if threshold is not None else self.phoneme_low_threshold
        display_thr = self.to_display_scale(thr, is_phone=True)
        errors = {"phonemes": [], "words": []}

        # Map each phoneme index to its containing word and word IPA
        phone_to_word: Dict[int, dict] = {}
        word_ipas: List[str] = []
        for w_idx, (start_idx, end_idx) in enumerate(word_phone_ranges):
            w_text = word_texts[w_idx] if w_idx < len(word_texts) else ""
            w_phones = phoneme_tokens[start_idx:end_idx] if start_idx < end_idx else []
            w_ipa = phones_to_word_ipa(w_phones)
            word_ipas.append(w_ipa)
            for p_idx in range(start_idx, end_idx):
                phone_to_word[p_idx] = {
                    "word": w_text,
                    "word_ipa": w_ipa,
                }

        def get_severity(score_10: float) -> str:
            if score_10 < 4.0:
                return "critical"
            elif score_10 < 6.0:
                return "warning"
            else:
                return "minor"

        pa = predictions.get("phoneme_accuracy")
        if pa is not None and isinstance(pa, torch.Tensor):
            for i, (tok, score) in enumerate(zip(phoneme_tokens, pa.tolist())):
                display_score = self.to_display_scale(score, is_phone=True)
                if display_score < display_thr:
                    w_info = phone_to_word.get(i, {})
                    errors["phonemes"].append(
                        {
                            "index": i,
                            "phoneme": tok,
                            "ipa": format_phone_ipa(tok),
                            "word": w_info.get("word", ""),
                            "word_ipa": w_info.get("word_ipa", ""),
                            "score": display_score,
                            "severity": get_severity(display_score),
                            "tip": _get_phoneme_tip(tok),
                        }
                    )

        wt_acc = predictions.get("word_accuracy")
        wt_stress = predictions.get("word_stress")
        if wt_acc is not None and isinstance(wt_acc, torch.Tensor):
            for i, (word, acc_score) in enumerate(zip(word_texts, wt_acc.tolist())):
                stress_score = wt_stress[i].item() if (wt_stress is not None and isinstance(wt_stress, torch.Tensor) and i < len(wt_stress)) else acc_score
                combined_score = (acc_score + stress_score) / 2.0
                display_score = self.to_display_scale(combined_score)
                if display_score < display_thr:
                    w_ipa = word_ipas[i] if i < len(word_ipas) else ""
                    errors["words"].append(
                        {
                            "index": i, 
                            "word": word, 
                            "word_ipa": w_ipa,
                            "score": display_score,
                            "severity": get_severity(display_score)
                        }
                    )

        return errors

    def build_words_detail(
        self,
        predictions: Dict[str, torch.Tensor],
        phoneme_tokens: List[str],
        word_texts: List[str],
        word_phone_ranges: List[tuple],
    ) -> List[Dict[str, Any]]:
        """Build detailed word-by-word analysis with IPA and constituent phonemes for interactive UI."""
        words_detail = []
        pa = predictions.get("phoneme_accuracy")
        wt_acc = predictions.get("word_accuracy")
        wt_stress = predictions.get("word_stress")
        wt_total = predictions.get("word_total")

        for i, word in enumerate(word_texts):
            start_idx, end_idx = word_phone_ranges[i] if i < len(word_phone_ranges) else (0, 0)
            w_phones = phoneme_tokens[start_idx:end_idx] if start_idx < end_idx else []
            w_ipa = phones_to_word_ipa(w_phones)

            # Determine word score
            if wt_total is not None and isinstance(wt_total, torch.Tensor) and i < len(wt_total):
                w_score = self.to_display_scale(wt_total[i].item())
            elif wt_acc is not None and isinstance(wt_acc, torch.Tensor) and i < len(wt_acc):
                stress = wt_stress[i].item() if (wt_stress is not None and isinstance(wt_stress, torch.Tensor) and i < len(wt_stress)) else wt_acc[i].item()
                w_score = self.to_display_scale((wt_acc[i].item() + stress) / 2.0)
            else:
                w_score = 8.0

            # Constituent phonemes with calibrated IPA scoring
            phones_list = []
            for p_idx in range(start_idx, end_idx):
                p_tok = phoneme_tokens[p_idx]
                p_raw = pa[p_idx].item() if (pa is not None and isinstance(pa, torch.Tensor) and p_idx < len(pa)) else None
                p_score = self.to_display_scale(p_raw, is_phone=True) if p_raw is not None else w_score
                p_status = "good" if p_score >= 7.5 else ("warning" if p_score >= 5.5 else "bad")
                phones_list.append({
                    "phoneme": p_tok,
                    "ipa": format_phone_ipa(p_tok),
                    "ipa_char": phone_to_ipa(p_tok),
                    "score": p_score,
                    "status": p_status,
                    "tip": _get_phoneme_tip(p_tok),
                })

            if phones_list:
                phone_avg = sum(p["score"] for p in phones_list) / len(phones_list)
                phone_min = min(p["score"] for p in phones_list)
                # Blend word prediction with calibrated phoneme accuracy so word scores accurately reflect mispronounced sounds
                if wt_total is not None and isinstance(wt_total, torch.Tensor) and i < len(wt_total):
                    w_score = round(0.5 * w_score + 0.5 * phone_avg, 1)
                elif wt_acc is not None and isinstance(wt_acc, torch.Tensor) and i < len(wt_acc):
                    w_score = round(0.4 * w_score + 0.6 * phone_avg, 1)
                else:
                    w_score = round(phone_avg, 1)
                if phone_min < 5.0 and w_score > 6.0:
                    w_score = round(max(phone_min + 1.0, 4.5), 1)

            w_status = "good" if w_score >= 7.5 else ("warning" if w_score >= 5.5 else "bad")

            words_detail.append({
                "word": word,
                "word_ipa": w_ipa,
                "score": w_score,
                "status": w_status,
                "phonemes": phones_list,
            })

        return words_detail

    # ── L2-MDD Per-Turn Feedback (ASHA Clinical Diagnostic Engine) ───
    @staticmethod
    def generate_l2_turn_feedback(errors: Optional[Dict[str, List[dict]]]) -> Optional[str]:
        """Generate clinical-grade ASHA articulatory feedback from L2-MDD full phoneme scan."""
        if not errors:
            return None

        phones = errors.get("phonemes", [])
        if not phones:
            return "L2-MDD: Toàn bộ âm vị đều phát âm đạt chuẩn, không phát hiện âm lệch."

        parts = []
        seen_items = set()
        for p in phones:
            w_text = p.get("word", "")
            target_ipa = p.get("target_ipa") or format_phone_ipa(p.get("target_phone") or p.get("phoneme", ""))
            actual_ipa = p.get("actual_ipa", "")
            rule_name = p.get("rule_name_vi", "")
            tip = p.get("articulatory_tip") or p.get("tip") or ""

            key = f"{target_ipa}_{actual_ipa}_{w_text}"
            if key in seen_items:
                continue
            seen_items.add(key)

            if actual_ipa and actual_ipa != target_ipa:
                item = f"âm {target_ipa} bị lệch thành {actual_ipa}"
            else:
                item = f"âm {target_ipa}"

            if w_text:
                w_ipa = p.get("word_ipa", "")
                item += f" trong \"{w_text}\"" + (f" ({w_ipa})" if w_ipa else "")
            if rule_name:
                item += f" [{rule_name}]"
            if tip:
                item += f" — {tip}"
            parts.append(item)

        if not parts:
            return None
        return "L2-MDD (ASHA) lưu ý: " + "; ".join(parts[:3]) + "."

    # ── SpeechOcean Feedback Generation (No L2-MDD in overall) ──────
    @staticmethod
    def generate_transformer_feedback(
        scores_pronunciation: Optional[Dict[str, float]] = None,
        errors_pronunciation: Optional[Dict[str, List[dict]]] = None,
        transcript: str = "",
    ) -> Dict[str, Any]:
        """Generate structured feedback based purely on SpeechOcean762 scores with standard IPA."""
        feedback: Dict[str, Any] = {}

        ref_scores = scores_pronunciation or {}
        ref_errors = errors_pronunciation or {}

        if ref_scores:
            weak_words_p = [w for w in ref_errors.get("words", []) if w.get("score", 10.0) < 7.0]
            weak_phones_p = [p for p in ref_errors.get("phonemes", []) if p.get("score", 10.0) < 7.0]
            feedback["pronunciation_model"] = {
                "scores": ref_scores,
                "weak_words": [
                    {
                        "word": w["word"], 
                        "word_ipa": w.get("word_ipa", ""), 
                        "score": w["score"], 
                        "severity": w["severity"]
                    } for w in weak_words_p
                ],
                "weak_phonemes": [
                    {
                        "phoneme": p["phoneme"], 
                        "ipa": p.get("ipa") or format_phone_ipa(p["phoneme"]),
                        "word": p.get("word", ""),
                        "word_ipa": p.get("word_ipa", ""),
                        "score": p["score"], 
                        "severity": p["severity"], 
                        "tip": p.get("tip", "")
                    } for p in weak_phones_p
                ],
            }

        total = ref_scores.get("total", ref_scores.get("final", 0.0))
        acc = ref_scores.get("accuracy", 0.0)
        flu = ref_scores.get("fluency", 0.0)
        pro = ref_scores.get("prosodic", 0.0)

        # Determine level
        if total >= 8.5:
            level = "excellent"
            level_vi = "Xuất sắc"
        elif total >= 7.0:
            level = "good"
            level_vi = "Tốt"
        elif total >= 5.0:
            level = "average"
            level_vi = "Trung bình"
        elif total >= 3.0:
            level = "weak"
            level_vi = "Yếu"
        else:
            level = "critical"
            level_vi = "Cần cải thiện nhiều"

        feedback["level"] = level
        feedback["level_vi"] = level_vi

        # Build summary
        summary_parts = []
        summary_parts.append(f"📊 Mức đánh giá: **{level_vi}** ({total:.1f}/10)")

        if acc >= 7.0:
            summary_parts.append(f"✅ Phát âm chính xác tốt ({acc:.1f}/10)")
        elif acc >= 5.0:
            summary_parts.append(f"⚠️ Phát âm chính xác ở mức trung bình ({acc:.1f}/10)")
        else:
            summary_parts.append(f"❌ Phát âm chưa chính xác ({acc:.1f}/10), cần luyện tập thêm")

        if flu >= 7.0:
            summary_parts.append(f"✅ Nói trôi chảy ({flu:.1f}/10)")
        elif flu >= 5.0:
            summary_parts.append(f"⚠️ Còn ngắc ngứ, chưa thật trôi chảy ({flu:.1f}/10)")
        else:
            summary_parts.append(f"❌ Nói chưa trôi chảy ({flu:.1f}/10), thử nói chậm hơn và rõ ràng hơn")

        if pro >= 7.0:
            summary_parts.append(f"✅ Ngữ điệu tự nhiên ({pro:.1f}/10)")
        elif pro >= 5.0:
            summary_parts.append(f"⚠️ Ngữ điệu cần cải thiện ({pro:.1f}/10)")
        else:
            summary_parts.append(f"❌ Ngữ điệu đơn điệu ({pro:.1f}/10), thử nhấn mạnh từ quan trọng")

        feedback["summary"] = "\n".join(summary_parts)

        # Build tips from SpeechOcean error analysis using IPA
        tips: List[str] = []
        model_data = feedback.get("pronunciation_model", {})
        weak_phones = model_data.get("weak_phonemes", [])
        sorted_phones = sorted(weak_phones, key=lambda x: x["score"])
        for p in sorted_phones[:5]:  # Top 5 worst phonemes
            tip = p.get("tip", "")
            ph_ipa = p.get("ipa") or format_phone_ipa(p.get("phoneme", ""))
            w_text = p.get("word", "")
            w_ipa = p.get("word_ipa", "")
            in_word = f" trong từ \"{w_text}\" ({w_ipa})" if (w_text and w_ipa) else (f" trong từ \"{w_text}\"" if w_text else "")
            
            if tip:
                tips.append(f"🔤 Âm {ph_ipa}{in_word}: {tip} (điểm: {p['score']:.1f})")
            else:
                tips.append(f"🔤 Luyện phát âm {ph_ipa}{in_word} (điểm: {p['score']:.1f})")

        weak_words = model_data.get("weak_words", [])
        sorted_words = sorted(weak_words, key=lambda x: x["score"])
        for w in sorted_words[:5]:  # Top 5 worst words
            w_ipa = w.get("word_ipa", "")
            ipa_str = f" ({w_ipa})" if w_ipa else ""
            tips.append(f"📝 Từ \"{w['word']}\"{ipa_str}: cần luyện phát âm rõ hơn (điểm: {w['score']:.1f})")

        feedback["tips"] = tips

        return feedback


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/models', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/models/transformer_encoder.py
"""
Task-specific Transformer encoder stacked on WavLM features.

Step 2: refines frame-level representations for pronunciation assessment.
"""

from __future__ import annotations

import math
from typing import Optional

import torch
import torch.nn as nn


class TaskTransformerEncoder(nn.Module):
    """
    Additional Transformer encoder layers (2-4) on top of WavLM hidden states.

    Uses pre-norm TransformerEncoderLayer for training stability.
    """

    def __init__(
        self,
        input_dim: int,
        num_layers: int = 3,
        num_heads: int = 8,
        ff_dim: int = 4096,
        dropout: float = 0.1,
        max_seq_len: int = 2000,
    ):
        super().__init__()
        if input_dim % num_heads != 0:
            raise ValueError(
                f"input_dim ({input_dim}) must be divisible by num_heads ({num_heads})"
            )
        self.input_proj = nn.Linear(input_dim, input_dim)
        self.pos_encoding = SinusoidalPositionalEncoding(input_dim, max_seq_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=input_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_dim = input_dim

    def forward(
        self,
        x: torch.Tensor,
        src_key_padding_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Args:
            x: (B, T, D) frame features from WavLM.
            src_key_padding_mask: (B, T) True = padded frame (ignore).

        Returns:
            (B, T, D) refined frame features.
        """
        x = self.input_proj(x)
        x = self.pos_encoding(x)
        return self.encoder(x, src_key_padding_mask=src_key_padding_mask)


class SinusoidalPositionalEncoding(nn.Module):
    """Standard sinusoidal position encoding added to frame features."""

    def __init__(self, dim: int, max_len: int = 2000):
        super().__init__()
        pe = torch.zeros(max_len, dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, dim, 2, dtype=torch.float) * (-math.log(10000.0) / dim)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)
        if seq_len > self.pe.size(1):
            self._grow_pe(seq_len)
        return x + self.pe[:, :seq_len, :].to(dtype=x.dtype, device=x.device)

    def _grow_pe(self, seq_len: int) -> None:
        dim = self.pe.size(-1)
        pe = torch.zeros(seq_len, dim, device=self.pe.device, dtype=self.pe.dtype)
        position = torch.arange(0, seq_len, dtype=torch.float, device=self.pe.device).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, dim, 2, dtype=torch.float, device=self.pe.device)
            * (-math.log(10000.0) / dim)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = pe.unsqueeze(0)


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/models', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/models/wavlm_encoder.py
"""
WavLM Large feature extractor.

Step 1 of pipeline: waveform -> frame-level hidden states (B, T, D).
Supports frozen backbone or LoRA fine-tuning via PEFT.
"""

from __future__ import annotations

from contextlib import nullcontext
from typing import Optional

import os
import torch
import torch.nn as nn
from transformers import WavLMModel


class WavLMEncoder(nn.Module):
    """
    Wraps `microsoft/wavlm-large` for pronunciation feature extraction.

    Output: frame-level representations at ~20ms stride (50 Hz for 16kHz audio
    with conv subsampling factor 320: 16000/320 = 50 frames/sec).
    """

    def __init__(
        self,
        model_name: str = "microsoft/wavlm-large",
        freeze: bool = True,
        use_lora: bool = False,
        lora_r: int = 8,
        lora_alpha: int = 16,
        lora_dropout: float = 0.05,
        lora_target_modules: Optional[list] = None,
    ):
        super().__init__()
        if model_name and not os.path.exists(model_name) and (model_name.startswith("/") or "\\" in model_name):
            model_name = "microsoft/wavlm-large"
        self.wavlm = WavLMModel.from_pretrained(model_name)
        self.output_dim = self.wavlm.config.hidden_size  # 1024 for large

        if freeze and not use_lora:
            for p in self.wavlm.parameters():
                p.requires_grad = False
            self.wavlm.eval()

        self._frozen = freeze and not use_lora

        if use_lora:
            try:
                import peft.import_utils
                peft.import_utils.is_torchao_available = lambda: False
            except Exception:
                pass

            from peft import LoraConfig, get_peft_model

            target = lora_target_modules or ["q_proj", "v_proj"]
            lora_config = LoraConfig(
                r=lora_r,
                lora_alpha=lora_alpha,
                target_modules=target,
                lora_dropout=lora_dropout,
                bias="none",
            )
            self.wavlm = get_peft_model(self.wavlm, lora_config)

    def forward(
        self,
        waveform: torch.Tensor,
        wav_lengths: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Args:
            waveform: (B, num_samples) float tensor, 16kHz mono.
            wav_lengths: (B,) actual sample counts before padding.
            attention_mask: optional (B, num_samples), 1=valid 0=pad.

        Returns:
            hidden_states: (B, T_frames, D) last hidden layer output.
        """
        if attention_mask is None:
            B, S = waveform.shape
            if wav_lengths is not None:
                attention_mask = (
                    torch.arange(S, device=waveform.device).unsqueeze(0)
                    < wav_lengths.unsqueeze(1)
                ).long()
            else:
                # padded batch: trailing zeros from pad_sequence
                attention_mask = (waveform.abs() > 1e-8).long()

        if self._frozen:
            self.wavlm.eval()

        ctx = torch.no_grad if self._frozen else nullcontext
        with ctx():
            outputs = self.wavlm(
                input_values=waveform,
                attention_mask=attention_mask,
            )
        return outputs.last_hidden_state

    def frame_lengths_from_samples(self, wav_lengths: torch.Tensor) -> torch.Tensor:
        """Exact WavLM frame counts from raw sample lengths (not samples//320)."""
        return self.wavlm._get_feat_extract_output_lengths(wav_lengths).long()

    def frame_rate(self, sample_rate: int = 16000) -> float:
        """Approximate frame rate (Hz) of WavLM output."""
        # WavLM conv feature extractor: total stride 320 samples
        return sample_rate / 320.0


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/infer', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/infer/__init__.py
"""Inference: pronunciation scoring (+ optional diarize)."""

from infer.pipeline import SpeakingPipeline
from infer.pronunciation import Predictor
from infer.transcribe import transcribe_audio

__all__ = ["Predictor", "SpeakingPipeline", "transcribe_audio"]


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/infer', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/infer/device_utils.py
"""Resolve compute device for inference (GPU / CPU)."""

from __future__ import annotations

import os
from typing import Any, Dict, Optional

import torch


def resolve_device(
    config: Optional[Dict[str, Any]] = None,
    explicit: Optional[str] = None,
) -> str:
    """Prefer explicit arg → env → config → auto cuda/cpu."""
    if explicit:
        return _normalize(explicit)

    for key in ("DEVICE", "TORCH_DEVICE", "HF_DEVICE"):
        if os.getenv(key):
            return _normalize(os.getenv(key))

    cfg = config or {}
    inf = cfg.get("inference") or {}
    if inf.get("device"):
        return _normalize(inf["device"])

    proj = cfg.get("project") or {}
    if proj.get("device"):
        return _normalize(proj["device"])

    return "cuda" if torch.cuda.is_available() else "cpu"


def _normalize(device: str) -> str:
    device = device.strip().lower()
    if device in ("gpu", "cuda", "cuda:0"):
        return "cuda" if torch.cuda.is_available() else "cpu"
    if device.startswith("cuda") and not torch.cuda.is_available():
        return "cpu"
    return device


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/infer', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/infer/lang_id.py
"""Language identification — bilingual (EN/VI) detection and Vietnamese ratio estimation."""

from __future__ import annotations

import math
import re
import threading
import unicodedata
from pathlib import Path
from typing import Any, Dict, Optional, Sequence, Tuple

PathLike = str | Path

ALLOWED_LANGUAGES = ("en", "vi")

_VI_CHARS = re.compile(
    r"[\u0103\u0102\u00e2\u00c2\u00ea\u00ca\u00f4\u00d4\u01a1\u01a0\u01b0\u01af"
    r"\u00e0\u00e1\u00e3\u00e4\u00e5\u00e8\u00e9\u00eb\u00ec\u00ed\u00f2\u00f3"
    r"\u00f5\u00f9\u00fa\u00fd\u0111\u0110\u1ea0-\u1ef9]",
    re.UNICODE,
)

# Common Vietnamese words mis-heard by English-only Whisper
_VI_WORDS = re.compile(
    r"\b(V[AĂÂÁẠẢÃ]NG|KH[OÔÓỌỎÕ]NG|Đ[UƯÚỤỦŨ]|T[ÔƠÓỌỎÕ]I|C[OÔÓỌỎÕ]|M[UƯÚỤỦŨ]ON|"
    r"EM|ANH|CH[ÀAÁẠẢÃ]|KH[OÔ]|B[AĂÂÁẠẢÃ]N|NH[EÊÉẸẺẼ]|R[AĂÂÁẠẢÃ]T|N[ÀAÁẠẢÃ]Y|"
    r"G[IÍỊỈĨ]|H[OÔÓỌỎÕ]C|TI[ẾEÉẸẺẼ]NG|VI[EÊÉẸẺẼ]T)\b",
    re.IGNORECASE,
)

# Unambiguous toneless Vietnamese words (never everyday English tokens)
_STRICT_VI_ASCII_WORDS = frozenset({
    "xin", "chao", "thay", "vang", "tieng", "viet", "khong", "duoc", "nhung",
    "minh", "nhieu", "biet", "muon", "giup", "nguoi", "chua", "phai", "viec",
    "thuong", "truong", "ngay", "gioi", "khac", "cung", "nhau", "luon", "trong",
    "ngoai", "giua", "vay", "nhien", "chac", "tuy", "nhat", "nhi", "kiem",
    "luyen", "hieu", "khach", "nuoc", "quyen", "duong", "chung", "rieng",
    "xong", "nua", "truoc", "nhanh", "cham", "thich", "nghi", "vui", "nghe",
    "xem", "nhe", "nha", "kia", "quen", "theo", "chuc", "mung", "giao",
    "buoi", "nhom", "tuan", "phut", "giay",
})

# Context-dependent toneless VI words (e.g., pronouns/particles) safe when utterance is VI-anchored
_CONTEXTUAL_VI_ASCII_WORDS = frozenset({
    "em", "anh", "roi", "hoc", "nay", "qua", "sai", "ong",
})

# Toneless Vietnamese homographs that also exist in English; only VI if adjacent to definite VI tokens
_HOMOGRAPH_VI_ASCII_WORDS = frozenset({
    "con", "cho", "hay", "sao", "bao", "ta", "ra",
})

_WORD_TOKEN_RE = re.compile(r"[A-Za-z\u00C0-\u1EF9']+")

_classifier = None
_classifier_error: Optional[str] = None
_lock = threading.Lock()

# ISO 639-1 / SpeechBrain label fragments treated as Vietnamese
_VI_CODES = frozenset({"vi", "vie", "vietnamese"})
_EN_CODES = frozenset({"en", "eng", "english"})


def _load_asr_lang_cfg() -> Dict[str, Any]:
    try:
        import yaml
        from paths import PRONUNCIATION_CONFIG

        with open(PRONUNCIATION_CONFIG, encoding="utf-8") as f:
            cfg = yaml.safe_load(f) or {}
        return (cfg.get("asr") or {}).get("lang_id") or {}
    except Exception:
        return {}


def _normalize_lang(label: str) -> str:
    s = (label or "").strip().lower()
    if ":" in s:
        s = s.split(":", 1)[0].strip()
    if " " in s:
        s = s.split()[0]
    return s


def _get_classifier(device: str = "cpu"):
    global _classifier, _classifier_error
    if _classifier is not None:
        return _classifier
    if _classifier_error:
        return None
    with _lock:
        if _classifier is not None:
            return _classifier
        if _classifier_error:
            return None
        try:
            from speechbrain.inference.classifiers import EncoderClassifier

            from paths import ROOT

            savedir = ROOT / "pretrained_models" / "lang-id-voxlingua107-ecapa"
            _classifier = EncoderClassifier.from_hparams(
                source="speechbrain/lang-id-voxlingua107-ecapa",
                savedir=str(savedir),
                run_opts={"device": device},
            )
        except Exception as exc:
            _classifier_error = str(exc)
            print(f"[lang_id] Không tải được model LID: {exc}", flush=True)
            return None
        return _classifier


def _extract_binary_lid_probs(
    clf: Any,
    out_probs: Any,
    allowed_languages: Sequence[str] = ALLOWED_LANGUAGES,
) -> Optional[Tuple[str, float]]:
    """Restrict SpeechBrain 107-class output log-probs strictly to allowed languages (EN/VI)."""
    try:
        label_enc = getattr(getattr(clf, "hparams", None), "label_encoder", None)
        ind2lab = getattr(label_enc, "ind2lab", None)
        if not ind2lab or out_probs is None:
            return None

        probs_1d = out_probs.squeeze()
        if hasattr(probs_1d, "tolist"):
            vals = probs_1d.tolist()
        else:
            vals = list(probs_1d)

        allowed = tuple(_normalize_lang(c) for c in allowed_languages if _normalize_lang(c))
        lang_scores: Dict[str, float] = {}
        for idx, raw_lbl in ind2lab.items():
            code = _normalize_lang(str(raw_lbl))
            if code in _VI_CODES:
                code = "vi"
            elif code in _EN_CODES:
                code = "en"
            if code in allowed and int(idx) < len(vals):
                val = float(vals[int(idx)])
                if code not in lang_scores or val > lang_scores[code]:
                    lang_scores[code] = val

        if not lang_scores:
            return None

        # Normalize over ONLY the allowed subset using numerically stable softmax
        max_v = max(lang_scores.values())
        exps = {k: math.exp(v - max_v) for k, v in lang_scores.items()}
        denom = sum(exps.values()) or 1.0
        norm_probs = {k: v / denom for k, v in exps.items()}
        best_lang = max(norm_probs, key=lambda k: norm_probs[k])
        return best_lang, float(norm_probs[best_lang])
    except Exception:
        return None


def detect_audio_language(
    audio_path: PathLike,
    *,
    device: str = "cpu",
    allowed_languages: Sequence[str] = ALLOWED_LANGUAGES,
) -> Tuple[Optional[str], float]:
    """Return (language_code, confidence) restricted to allowed_languages (default: EN/VI)."""
    clf = _get_classifier(device)
    if clf is None:
        return None, 0.0
    try:
        out = clf.classify_file(str(audio_path))
        if not isinstance(out, (list, tuple)):
            return None, 0.0

        allowed = tuple(_normalize_lang(c) for c in allowed_languages if _normalize_lang(c))
        if len(out) >= 1 and allowed:
            restricted = _extract_binary_lid_probs(clf, out[0], allowed_languages=allowed)
            if restricted is not None:
                return restricted

        if len(out) >= 4:
            score = float(out[1]) if out[1] is not None else 0.0
            label = str(out[3])
        elif len(out) >= 2:
            score = float(out[0].max()) if hasattr(out[0], "max") else 0.0
            label = str(out[1][0]) if hasattr(out[1], "__getitem__") else str(out[1])
        else:
            return None, 0.0

        norm = _normalize_lang(label)
        if norm in _VI_CODES:
            norm = "vi"
        elif norm in _EN_CODES:
            norm = "en"
        if allowed and norm not in allowed:
            return None, 0.0
        return norm, score
    except Exception as exc:
        print(f"[lang_id] classify_file lỗi: {exc}", flush=True)
        return None, 0.0


def text_looks_vietnamese(text: str) -> bool:
    if not text or not text.strip():
        return False
    norm = unicodedata.normalize("NFC", text)
    if _VI_CHARS.search(norm):
        return True
    if _VI_WORDS.search(norm.upper()):
        return True
    return False


def classify_transcript_words(text: str, whisper_lang: str = "auto") -> Dict[str, Any]:
    """
    Classify lexical word tokens in a Whisper transcript into Vietnamese vs English.
    Returns word counts, percentages (strictly by word count), and labeled token lists.
    """
    if not text or not text.strip():
        return {
            "word_count": 0,
            "vi_word_count": 0,
            "en_word_count": 0,
            "vi_percent": 0.0,
            "en_percent": 0.0,
            "text_vi_percent": 0.0,
            "vi_words": [],
            "en_words": [],
        }

    norm = unicodedata.normalize("NFC", text)
    tokens = _WORD_TOKEN_RE.findall(norm)
    if not tokens:
        return {
            "word_count": 0,
            "vi_word_count": 0,
            "en_word_count": 0,
            "vi_percent": 0.0,
            "en_percent": 0.0,
            "text_vi_percent": 0.0,
            "vi_words": [],
            "en_words": [],
        }

    norm_wlang = _normalize_lang(whisper_lang or "auto")
    is_wlang_vi = norm_wlang in _VI_CODES
    has_diacritics = bool(_VI_CHARS.search(norm))

    # First pass: mark definite Vietnamese tokens (diacritics + strict unaccented VI words)
    definite_vi = [False] * len(tokens)
    lows = [tok.lower().strip("'") for tok in tokens]
    for i, (tok, low) in enumerate(zip(tokens, lows)):
        if _VI_CHARS.search(tok) or low in _STRICT_VI_ASCII_WORDS:
            definite_vi[i] = True

    has_anchor = has_diacritics or any(definite_vi)

    # Second pass: resolve contextual tokens and neighbor-anchored toneless homographs
    labels: list[str] = ["en"] * len(tokens)
    for i, low in enumerate(lows):
        if definite_vi[i]:
            labels[i] = "vi"
        elif (has_anchor or is_wlang_vi) and low in _CONTEXTUAL_VI_ASCII_WORDS:
            labels[i] = "vi"
        elif has_diacritics and low in _HOMOGRAPH_VI_ASCII_WORDS:
            prev_vi = i > 0 and definite_vi[i - 1]
            next_vi = (i + 1) < len(tokens) and definite_vi[i + 1]
            if prev_vi or next_vi:
                labels[i] = "vi"

    vi_words_list = [tok for tok, lbl in zip(tokens, labels) if lbl == "vi"]
    en_words_list = [tok for tok, lbl in zip(tokens, labels) if lbl != "vi"]
    total = len(tokens)
    vi_count = len(vi_words_list)
    en_count = total - vi_count
    vi_pct = round(100.0 * vi_count / total, 1) if total > 0 else 0.0
    en_pct = round(max(0.0, 100.0 - vi_pct), 1) if total > 0 else 0.0

    return {
        "word_count": total,
        "vi_word_count": vi_count,
        "en_word_count": en_count,
        "vi_percent": vi_pct,
        "en_percent": en_pct,
        "text_vi_percent": vi_pct,
        "vi_words": vi_words_list,
        "en_words": en_words_list,
    }


def analyze_vietnamese_text_ratio(text: str, whisper_lang: str = "auto") -> Dict[str, Any]:
    """
    Count Vietnamese vs non-Vietnamese words in a transcript.
    Uses accented Vietnamese letters (_VI_CHARS) plus unambiguous Vietnamese words.
    """
    res = classify_transcript_words(text, whisper_lang=whisper_lang)
    return {
        "word_count": res["word_count"],
        "vi_word_count": res["vi_word_count"],
        "en_word_count": res["en_word_count"],
        "text_vi_percent": res["text_vi_percent"],
    }


def compute_vietnamese_percent(
    acoustic_vi_prob: float,
    text: str = "",
    detected_lang: str = "en",
) -> Dict[str, Any]:
    """
    Combine Whisper acoustic P(vi | {en, vi}) and transcript lexical Vietnamese ratio
    into an interpretable Vietnamese percentage [0.0, 100.0].
    """
    vi_prob = max(0.0, min(1.0, float(acoustic_vi_prob)))
    en_prob = max(0.0, min(1.0, 1.0 - vi_prob))
    acoustic_vi_pct = round(vi_prob * 100.0, 1)

    text_stats = analyze_vietnamese_text_ratio(text)
    vi_word_count = int(text_stats["vi_word_count"])
    word_count = int(text_stats["word_count"])
    text_vi_pct = float(text_stats["text_vi_percent"])

    norm_lang = _normalize_lang(detected_lang)
    is_vi_lang = norm_lang in _VI_CODES

    if vi_word_count > 0:
        vi_percent = 0.7 * text_vi_pct + 0.3 * acoustic_vi_pct
    elif is_vi_lang or acoustic_vi_pct >= 50.0:
        vi_percent = acoustic_vi_pct
    else:
        vi_percent = 0.3 * acoustic_vi_pct

    vi_percent = round(min(100.0, max(0.0, vi_percent)), 1)
    en_percent = round(min(100.0, max(0.0, 100.0 - vi_percent)), 1)
    return {
        "vi_percent": vi_percent,
        "en_percent": en_percent,
        "acoustic_vi_percent": acoustic_vi_pct,
        "text_vi_percent": text_vi_pct,
        "vi_word_count": vi_word_count,
        "word_count": word_count,
        "language_probs": {
            "en": round(en_prob, 4),
            "vi": round(vi_prob, 4),
        },
    }


def is_vietnamese_segment(
    audio_path: PathLike,
    transcript: str = "",
    *,
    device: str = "cpu",
    cfg: Optional[Dict[str, Any]] = None,
    whisper_lang: Optional[str] = None,
) -> Tuple[bool, str]:
    """
    True if segment is detected as Vietnamese speech.
    Returns (is_vi, reason).
    """
    cfg = cfg or _load_asr_lang_cfg()
    if not cfg.get("enabled", True):
        return False, ""

    # 1. Primary check: Whisper detected Vietnamese language token
    if whisper_lang and _normalize_lang(whisper_lang) in _VI_CODES:
        return True, f"whisper:{whisper_lang}"

    # 2. Text heuristics: regex on accents and common Vietnamese words
    if cfg.get("text_vi_regex", True) and text_looks_vietnamese(transcript):
        return True, "text"

    # 3. Audio classifier fallback (SpeechBrain ECAPA) restricted to allowed languages
    drop_langs = {_normalize_lang(x) for x in cfg.get("drop_languages", ["vi", "vie", "vietnamese"])}
    allowed_langs = tuple(cfg.get("allowed_languages") or ALLOWED_LANGUAGES)
    min_conf = float(cfg.get("min_confidence", 0.45))

    lang, conf = detect_audio_language(audio_path, device=device, allowed_languages=allowed_langs)
    if lang is None:
        return False, ""

    if lang in drop_langs and conf >= min_conf:
        return True, f"audio:{lang}:{conf:.2f}"

    if any(v in lang for v in ("viet", "vietnam")) and conf >= min_conf:
        return True, f"audio:{lang}:{conf:.2f}"

    return False, ""


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/infer', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/infer/load_progress.py
"""Thread-safe model loading progress for the web UI."""

from __future__ import annotations

import threading
import time
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

StepDef = Tuple[str, str, int]

STEPS: List[StepDef] = [
    ("config", "Đọc cấu hình", 3),
    ("pronunciation", "Chấm điểm: WavLM + Transformer", 38),
    ("pronunciation_ckpt", "Checkpoint pronunciation.pt", 12),
    ("l2_mdd", "Phân tích âm vị: WavLM + L2-MDD", 14),
    ("l2_mdd_ckpt", "Checkpoint l2_mdd_best.pt", 6),
    ("whisper_proc", "Whisper: tokenizer + processor", 5),
    ("whisper_model", "Whisper: tải weights ASR", 22),
]


@dataclass
class LoadStep:
    id: str
    label: str
    status: str = "pending"
    detail: str = ""


class LoadProgress:
    def __init__(self) -> None:
        self._lock = threading.Lock()
        self._steps: Dict[str, LoadStep] = {
            sid: LoadStep(sid, label) for sid, label, _ in STEPS
        }
        self._weights = {sid: w for sid, _, w in STEPS}
        self._current: Optional[str] = None
        self._error: Optional[str] = None
        self._done = False
        self._running_since: Optional[float] = None

    def reset(self) -> None:
        with self._lock:
            for sid, label, _ in STEPS:
                self._steps[sid] = LoadStep(sid, label)
            self._current = None
            self._error = None
            self._done = False
            self._running_since = None

    def start(self, step_id: str, detail: str = "") -> None:
        with self._lock:
            step = self._steps[step_id]
            step.status = "running"
            step.detail = detail
            self._current = step_id
            self._running_since = time.monotonic()
            label = step.label
        msg = f"{label}" + (f" — {detail}" if detail else "")
        print(f"[load] ▶ {msg}", flush=True)

    def finish(self, step_id: str, detail: str = "") -> None:
        with self._lock:
            step = self._steps[step_id]
            step.status = "done"
            if detail:
                step.detail = detail
            if self._current == step_id:
                self._current = None
                self._running_since = None
            label = step.label
        print(f"[load] ✓ {label}", flush=True)

    def fail_running(self, error: str) -> None:
        with self._lock:
            step_id = self._current
        if step_id:
            self.fail(step_id, error)

    def fail(self, step_id: str, error: str) -> None:
        with self._lock:
            self._steps[step_id].status = "error"
            self._steps[step_id].detail = error
            self._error = error
            label = self._steps[step_id].label
        print(f"[load] ✕ {label}: {error}", flush=True)

    def complete(self) -> None:
        with self._lock:
            self._done = True
            self._current = None
        print("[load] ✓ Tất cả model sẵn sàng", flush=True)

    def to_dict(self) -> dict:
        with self._lock:
            total = sum(self._weights.values())
            earned = 0.0
            steps_out = []
            all_done = True
            for sid, label, weight in STEPS:
                s = self._steps[sid]
                steps_out.append({
                    "id": sid,
                    "label": label,
                    "status": s.status,
                    "detail": s.detail,
                })
                if s.status != "done":
                    all_done = False
                if s.status == "done":
                    earned += weight
                elif s.status == "running":
                    earned += weight * 0.35

            if self._done or all_done:
                percent = 100
            else:
                percent = int(min(98, round(earned / total * 100)))

            current = self._steps[self._current] if self._current else None
            elapsed_sec = None
            if self._running_since is not None:
                elapsed_sec = int(time.monotonic() - self._running_since)

            return {
                "percent": percent,
                "current": self._current,
                "current_label": current.label if current else None,
                "current_detail": current.detail if current else None,
                "elapsed_sec": elapsed_sec,
                "steps": steps_out,
                "done": self._done,
                "error": self._error,
            }


progress = LoadProgress()


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/infer', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/infer/pipeline.py
"""Unified inference: diarize 2 speakers → split sentences → score each segment."""

from __future__ import annotations

import argparse
import json
import re
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

import torch
import yaml

from data.audio_preprocess import PreprocessConfig
from data.silence_split import SilenceSplitConfig, export_diarization_clips, split_audio_file
from infer.lang_id import is_vietnamese_segment

from infer.pronunciation import Predictor, L2MDDPredictor
from infer.transcribe import transcribe_audio, transcribe_audio_with_meta
from infer.device_utils import resolve_device
from models.pronunciation_scorer import PronunciationScorer
from paths import SPEAKER_DIARIZE_DIR, PRONUNCIATION_CONFIG, ROOT

SCORE_KEYS = ("accuracy", "fluency", "prosodic", "total")

DEFAULT_BACKCHANNELS = frozenset({
    "yes", "no", "yeah", "yep", "ok", "okay", "sure", "uh", "um", "ah", "hmm",
    "mmm", "mm", "erm", "er", "mhm", "huh",
    "right", "alright", "oh", "cool", "thank you", "thanks", "bye", "bye bye",
    "hello", "hi", "goodbye", "fine", "yup",
})

INCOMPLETE_CONNECTORS = frozenset({
    "is", "are", "was", "were", "my", "the", "a", "an", "and", "or", "because", "like", "for",
})


def _is_incomplete_turn(transcript: str) -> bool:
    words = [w.lower().strip() for w in re.sub(r"[^\w\s']", " ", transcript).split() if w.strip()]
    if not words:
        return False
    return words[-1] in INCOMPLETE_CONNECTORS


def check_scoring_eligibility(
    transcript: str,
    duration_sec: float,
    config: Optional[Dict[str, Any]] = None,
) -> Tuple[bool, str]:
    """
    Check if utterance has sufficient phonological content to be scored.
    Returns (is_eligible, reason).
    """
    if not transcript or not transcript.strip():
        return False, "empty_transcript"

    cfg = config or {}
    min_duration = float(cfg.get("min_duration_sec", 0.8))
    min_words = int(cfg.get("min_words", 2))
    raw_backchannels = cfg.get("backchannel_words")
    if raw_backchannels:
        backchannels = frozenset(str(w).lower().strip() for w in raw_backchannels)
    else:
        backchannels = DEFAULT_BACKCHANNELS

    clean_words = [
        w.lower().strip() for w in re.sub(r"[^\w\s']", " ", transcript).split() if w.strip()
    ]
    if not clean_words:
        return False, "no_content_words"

    # 1. Utterance matches a multi-word backchannel phrase (e.g. "thank you", "bye bye")
    norm_phrase = " ".join(clean_words)
    if norm_phrase in backchannels:
        return False, "short_backchannel"

    # Or each individual word is a backchannel/filler token (e.g. "oh yeah", "yes ok", "mmm")
    if all(w in backchannels for w in clean_words):
        return False, "short_backchannel"

    # 2. Utterance is too short in duration (aborted false starts < 0.8s) or too few content words
    if duration_sec > 0 and duration_sec < 0.8:
        return False, "too_short"
    if duration_sec > 0 and duration_sec < min_duration and len(clean_words) < min_words:
        return False, "too_short"

    return True, "eligible"


def _serialize_dialogue_turn(s: Dict[str, Any], role: Optional[str] = None) -> Dict[str, Any]:
    """Serialize a sentence record into a unified dialogue turn object."""
    resolved_role = role or s.get("role") or "student"
    return {
        "role": resolved_role,
        "scored": bool(s.get("scored") and s.get("scores")),
        "is_vietnamese": s.get("is_vietnamese", False),
        "language": s.get("language", "en"),
        "eligibility_reason": s.get("eligibility_reason"),
        "start_sec": s.get("start_sec"),
        "end_sec": s.get("end_sec"),
        "transcript": s.get("transcript", ""),
        "audio": s.get("audio"),
        "scores": s.get("scores"),
        "errors": s.get("errors"),
        "transformer_feedback": s.get("transformer_feedback"),
        "words_detail": s.get("words_detail"),
        "l2_mdd_feedback": s.get("l2_mdd_feedback"),
    }


def _make_unscored_turn(
    seg: Dict[str, Any],
    transcript: str,
    *,
    role: str,
    is_vietnamese: bool = False,
    language: str = "en",
    eligibility_reason: str = "unscored",
    transformer_feedback: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """Build a standardized unscored turn record preserving transcript and timing."""
    words_neutral = [{"word": w, "status": "neutral"} for w in transcript.split() if w.strip()]
    return {
        "index": seg["index"],
        "start_sec": seg["start_sec"],
        "end_sec": seg["end_sec"],
        "duration_sec": seg.get("duration_sec", 0.0),
        "audio": seg["path"],
        "turn_index": seg["index"],
        "transcript": transcript,
        "role": role,
        "scored": False,
        "is_vietnamese": is_vietnamese,
        "language": language,
        "eligibility_reason": eligibility_reason,
        "scores": None,
        "errors": {"phonemes": [], "words": []},
        "words_detail": words_neutral,
        "transformer_feedback": transformer_feedback or {},
    }


def _build_dialogue(
    teacher_sentences: List[Dict[str, Any]],
    student_sentences: List[Dict[str, Any]],
) -> Dict[str, Any]:
    """Chronological turns with teacher prompt attached before each student answer."""
    turns: List[Dict[str, Any]] = (
        [_serialize_dialogue_turn(s, role="teacher") for s in teacher_sentences]
        + [_serialize_dialogue_turn(s, role="student") for s in student_sentences]
    )
    turns.sort(key=lambda t: (t.get("start_sec") or 0, 0 if t["role"] == "teacher" else 1))

    student_turns: List[Dict[str, Any]] = []
    last_teacher: Optional[Dict[str, Any]] = None
    for t in turns:
        if t["role"] == "teacher":
            last_teacher = t
        else:
            student_turns.append({
                **t,
                "teacher_prompt": last_teacher["transcript"] if last_teacher else None,
                "teacher_prompt_start_sec": last_teacher.get("start_sec") if last_teacher else None,
                "teacher_prompt_end_sec": last_teacher.get("end_sec") if last_teacher else None,
                "teacher_prompt_audio": last_teacher.get("audio") if last_teacher else None,
            })

    return {"turns": turns, "student_turns": student_turns}


def _avg_scores(items: List[Dict[str, Any]]) -> Dict[str, float]:
    out: Dict[str, float] = {}
    scored_items = [x for x in items if x.get("scored") is True and x.get("scores")]
    for key in SCORE_KEYS:
        vals = [x["scores"][key] for x in scored_items if x["scores"].get(key) is not None]
        if vals:
            out[key] = round(sum(vals) / len(vals), 2)
    return out


def _build_summary(
    sentences: List[Dict[str, Any]],
    *,
    filtered_vi: int = 0,
    config: Optional[Dict[str, Any]] = None,
    **_ignored: Any,
) -> Dict[str, Any]:
    summary_scores = _avg_scores(sentences)
    # Mỗi lượt nói (turn) = một dòng — không ghép liền
    lines = [s["transcript"].strip() for s in sentences if s.get("transcript", "").strip()]
    full_transcript = "\n".join(lines)

    scored_count = sum(1 for s in sentences if s.get("scored") is True)
    vi_count = sum(1 for s in sentences if s.get("is_vietnamese") is True or s.get("language") == "vi")
    unscored_count = len(sentences) - scored_count

    # ── Calculate benchmark-calibrated IELTS 4 pillars & Overall from config ──
    if scored_count > 0:
        cfg = config or {}
        ielts_cfg = cfg.get("ielts_calibration") or {}
        fc_cfg = ielts_cfg.get("fc") or {}
        lr_cfg = ielts_cfg.get("lr") or {}
        gra_cfg = ielts_cfg.get("gra") or {}

        fc_slope = float(fc_cfg.get("slope", 0.1890))
        fc_bias = float(fc_cfg.get("bias", 5.5574))
        lr_slope = float(lr_cfg.get("slope", 0.2162))
        lr_bias = float(lr_cfg.get("bias", 5.4123))
        gra_slope = float(gra_cfg.get("slope", 0.6774))
        gra_bias = float(gra_cfg.get("bias", 2.4684))

        acc = summary_scores.get("accuracy", 7.0)

        fc = round(max(1.0, min(9.0, fc_slope * acc + fc_bias)), 1)
        lr = round(max(1.0, min(9.0, lr_slope * acc + lr_bias)), 1)
        gra = round(max(1.0, min(9.0, gra_slope * acc + gra_bias)), 1)
        pr = summary_scores.get("total", acc)
        overall = round((pr + fc + lr + gra) / 4.0, 1)

        summary_scores["context"] = fc
        summary_scores["lexical"] = lr
        summary_scores["grammar"] = gra
        summary_scores["overall"] = overall
        summary_scores["score_fluency_coherence"] = fc
        summary_scores["score_lexical"] = lr
        summary_scores["score_grammar"] = gra

    return {
        "sentence_count": len(sentences),
        "scored_sentence_count": scored_count,
        "unscored_sentence_count": unscored_count,
        "vietnamese_sentence_count": vi_count,
        "transcript": full_transcript,
        "transcript_lines": lines,
        "transcript_source": "whisper",
        "filtered_vi_count": filtered_vi or vi_count,
        "scores": summary_scores,
        "pronunciation_feedback": None,
        "pronunciation_feedback_source": "local" if sentences else None,
        "feedback": None,
        "feedback_source": None,
    }


class SpeakingPipeline:
    """Diarize 2 speakers → silence-split → per-sentence scoring with dual models."""

    def __init__(
        self,
        config_path: str | Path | None = None,
        device: Optional[str] = None,
        pronunciation_ckpt: Optional[str] = None,
        l2_mdd_ckpt: Optional[str] = None,
        enable_feedback: bool = True,
        load_progress: Optional[Any] = None,
        asr_device: Optional[str] = None,
        diarize_device: Optional[str] = None,
    ):
        config_path = Path(config_path or PRONUNCIATION_CONFIG)
        if load_progress is not None:
            load_progress.start("config")
        with open(config_path, encoding="utf-8") as f:
            self.config = yaml.safe_load(f)
        if load_progress is not None:
            load_progress.finish("config")

        # ── Multi-GPU Resolution (Kaggle 2x T4 or single GPU/CPU) ──
        num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
        if num_gpus >= 2:
            self.device = resolve_device(self.config, device or "cuda:1")
            self.asr_device = asr_device or "cuda:0"
            self.diarize_device = diarize_device or "cuda:0"
        else:
            self.device = resolve_device(self.config, device)
            self.asr_device = asr_device or (self.config.get("asr") or {}).get("device") or self.device
            self.diarize_device = diarize_device or self.device

        print(f"[Pipeline Parallelism] Devices: Scoring={self.device} | Whisper ASR={self.asr_device} | Diarizer={self.diarize_device}", flush=True)

        asr_cfg = self.config.setdefault("asr", {})
        asr_cfg["device"] = self.asr_device
        self.preprocess = PreprocessConfig.from_dict(self.config.get("audio_preprocess"))
        wavlm_name = self.config.get("wavlm", {}).get("model_name", "microsoft/wavlm-large")
        
        # ── Load Pronunciation Model (SpeechOcean762) on self.device (cuda:1) ──
        self.pronunciation = Predictor(
            config_path,
            pronunciation_ckpt,
            self.device,
            load_progress=load_progress,
            model_step="pronunciation",
            ckpt_step="pronunciation_ckpt",
            wavlm_name=wavlm_name,
        )
        self.pronunciation.preprocess = self.preprocess

        # ── Load L2-MDD Model on self.device (cuda:1) ──
        self.l2_mdd: Optional[L2MDDPredictor] = None
        try:
            self.l2_mdd = L2MDDPredictor(
                config_path,
                l2_mdd_ckpt,
                self.device,
                load_progress=load_progress,
                model_step="l2_mdd",
                ckpt_step="l2_mdd_ckpt",
            )
            self.l2_mdd.preprocess = self.preprocess
            print(f"[OK] L2-MDD model loaded successfully on {self.device}")
        except Exception as e:
            print(f"[WARN] L2-MDD model not available: {e}")
            self.l2_mdd = None
        
        if not SPEAKER_DIARIZE_DIR.is_dir():
            raise FileNotFoundError(f"speaker-diarize not found at {SPEAKER_DIARIZE_DIR}")
        if str(SPEAKER_DIARIZE_DIR) not in sys.path:
            sys.path.insert(0, str(SPEAKER_DIARIZE_DIR))
        from speaker_diarize.pipeline import TwoSpeakerSplitter
        
        # ── Load TwoSpeakerSplitter on self.diarize_device (cuda:0) ──
        diar_cfg = self.config.get("diarization") or {}
        vad_thresh = float(diar_cfg.get("vad_threshold_db", -42.0))
        consec_gap = float(diar_cfg.get("consecutive_merge_gap_sec", 2.5))
        min_sim_thresh = float(diar_cfg.get("min_similarity_threshold", 0.40))
        self.diarizer = TwoSpeakerSplitter(
            device=self.diarize_device,
            vad_threshold_db=vad_thresh,
            cluster_window_sec=1.5,
            boundary_window_sec=0.5,
            min_speech_sec=0.25,
            min_segment_sec=0.3,
            merge_gap_sec=0.5,
            consecutive_merge_gap_sec=consec_gap,
            step_sec=0.50,
            boundary_step_sec=0.15,
            min_similarity_threshold=min_sim_thresh,
        )
        
        # ── Load Whisper Transcriber on self.asr_device (cuda:0) ──
        from infer.transcribe import get_transcriber
        self._asr_cfg = asr_cfg
        get_transcriber(load_progress, device=self.asr_device, asr_config=self._asr_cfg)

        self.enable_feedback = enable_feedback
        self.feedback_mode = "local"
        self._lang_id_cfg = self._asr_cfg.get("lang_id") or {}

    def _diarize_two_speakers(
        self,
        audio_path: Union[str, Path],
        output_dir: Optional[Union[str, Path]] = None,
        teacher_reference_path: Optional[Union[str, Path]] = None,
        teacher_embedding: Optional[Any] = None,
        student_embedding: Optional[Any] = None,
    ) -> Dict[str, Any]:
        result = self.diarizer.split_file(
            audio_path,
            output_dir,
            teacher_reference_path=teacher_reference_path,
            teacher_embedding=teacher_embedding,
            student_embedding=student_embedding,
            apply_denoise=self.preprocess.denoise,
        )
        out: Dict[str, Any] = {
            "segments": result.segments,
            "duration_sec": result.duration_sec,
            "teacher_cluster": result.teacher_cluster,
            "teacher_segments": result.teacher_segments,
            "student_segments": result.student_segments,
        }
        if result.teacher_path and result.student_path:
            out["teacher"] = str(result.teacher_path)
            out["student"] = str(result.student_path)
        return out

    def _collect_speaker_segments(
        self,
        output_dir: Path,
        speaker: str,
        *,
        source_audio: Union[str, Path],
        diarize_segments: list,
    ) -> List[Dict[str, Any]]:
        split_cfg = self.config.get("sentence_split") or {}
        track_preprocess = PreprocessConfig.from_dict(self.config.get("audio_preprocess"))
        track_preprocess.denoise = False
        merge_gap = float(split_cfg.get("diarization_merge_gap_sec", 1.2))

        segments = export_diarization_clips(
            source_audio,
            diarize_segments,
            output_dir,
            speaker,
            track_preprocess,
            merge_gap_sec=merge_gap,
            min_duration_sec=float(split_cfg.get("min_segment_sec", 0.6)),
            prefix=f"{speaker.lower()}_turn",
            intervening_backchannel_max_sec=float(split_cfg.get("intervening_backchannel_max_sec", 0.85)),
        )
        return segments

    @classmethod
    def calibrate_scores_with_phonemes(
        cls,
        pron_scores: Dict[str, Any],
        words_detail: List[Dict[str, Any]],
        config: Optional[Dict[str, Any]] = None,
    ) -> Dict[str, Any]:
        """Calibrate acoustic scores to human teacher benchmark scale (IELTS/Cambridge Speaking)
        with gentle L2-MDD phonological diagnostic fine-tuning.
        All weights and penalties are loaded from config (scorer.pronunciation_calibration and scorer.l2_mdd_penalty)."""
        calibrated = dict(pron_scores)

        base_acc = float(pron_scores.get("accuracy", 6.5))
        base_flu = float(pron_scores.get("fluency", 7.5))
        base_pro = float(pron_scores.get("prosodic", 7.5))

        cfg = config or {}
        p_cfg = cfg.get("pronunciation_calibration") or cfg.get("calibration") or {}
        enabled = p_cfg.get("enabled", True)

        if enabled:
            w_acc = float(p_cfg.get("w_acc", 0.8655))
            w_flu = float(p_cfg.get("w_flu", 0.0767))
            w_pro = float(p_cfg.get("w_pro", -0.3757))
            bias = float(p_cfg.get("bias", 3.4079))
            calibrated_p = w_acc * base_acc + w_flu * base_flu + w_pro * base_pro + bias
        else:
            calibrated_p = base_acc

        # L2-MDD diagnostic fine-tuning
        pen_cfg = cfg.get("l2_mdd_penalty") or p_cfg.get("l2_mdd_penalty") or {}
        pen_enabled = pen_cfg.get("enabled", True)
        if pen_enabled and words_detail:
            num_crit = 0
            num_warn = 0
            for w in words_detail:
                for ph in w.get("phonemes") or w.get("phones") or []:
                    st = ph.get("status")
                    if st == "bad":
                        num_crit += 1
                    elif st == "warning":
                        num_warn += 1

            if num_crit > 0 or num_warn > 0:
                pen_bad = float(pen_cfg.get("pen_bad", 0.04))
                pen_warn = float(pen_cfg.get("pen_warning", 0.015))
                max_pen = float(pen_cfg.get("max_penalty", 0.35))
                pen = min(max_pen, num_crit * pen_bad + num_warn * pen_warn)
                calibrated_p -= pen

        min_val = float(p_cfg.get("min_score", 1.0)) if enabled else 1.0
        max_val = float(p_cfg.get("max_score", 9.5)) if enabled else 10.0
        calibrated_p = round(max(min_val, min(max_val, calibrated_p)), 2)
        calibrated["accuracy"] = calibrated_p
        calibrated["total"] = calibrated_p
        calibrated["pronunciation"] = calibrated_p
        return calibrated

    def _process_segment(
        self,
        seg: Dict[str, Any],
        *,
        use_asr: bool,
        lang: Optional[str],
        score: bool = True,
        role: str = "student",
    ) -> Tuple[Optional[Dict[str, Any]], bool]:
        """Returns (sentence_dict, was_vi_filtered)."""
        if not use_asr:
            return None, False
        try:
            meta = transcribe_audio_with_meta(seg["path"], asr_config=self._asr_cfg)
            clean_tr = meta.get("text", "").strip()
            raw_tr = meta.get("raw_text", "").strip()
            detected_lang = meta.get("language", "en")
            is_vi_from_asr = bool(meta.get("is_vietnamese", False))
        except ValueError:
            return None, False
        if not clean_tr and not raw_tr:
            return None, False

        display_tr = raw_tr if is_vi_from_asr else clean_tr

        # ── Optional Vietnamese Guard (disabled by default in pure English mode) ──
        if self._lang_id_cfg.get("enabled", False):
            is_vi = is_vi_from_asr
            reason_vi = f"whisper:{detected_lang}" if is_vi_from_asr else "text"
            if not is_vi:
                is_vi, reason_vi = is_vietnamese_segment(
                    seg["path"],
                    raw_tr or clean_tr,
                    device=self.asr_device,
                    cfg=self._lang_id_cfg,
                    whisper_lang=detected_lang,
                )
            if is_vi:
                print(f"[lang_id] Phát hiện tiếng Việt ({reason_vi}): '{display_tr[:60]}...'. Bảo toàn turn với scored=False.")
                return _make_unscored_turn(
                    seg,
                    display_tr,
                    role=role,
                    is_vietnamese=True,
                    language="vi",
                    eligibility_reason="vietnamese_speech",
                    transformer_feedback={
                        "pedagogical_note": "Lượt nói bằng tiếng Việt (không tính vào điểm phát âm tiếng Anh)."
                    },
                ), True

        # ── Check Role Unscored (e.g. Teacher when score_teacher is False) ──
        if not score:
            return _make_unscored_turn(
                seg,
                clean_tr,
                role=role,
                eligibility_reason="role_unscored",
            ), False

        # ── Check Scoring Eligibility (Filter short backchannel / filler turns) ──
        elig_cfg = self.config.get("eligibility") or {}
        is_eligible, elig_reason = check_scoring_eligibility(
            clean_tr,
            seg.get("duration_sec", 0.0),
            config=elig_cfg,
        )
        if not is_eligible:
            print(f"[{role}] Lượt nói không đủ điều kiện chấm ({elig_reason}): '{clean_tr}'. Bảo toàn turn với scored=False.")
            return _make_unscored_turn(
                seg,
                clean_tr,
                role=role,
                eligibility_reason=elig_reason,
            ), False

        # ── Run Phoneme & Pronunciation Models (SpeechOcean762 + L2-MDD) ──
        try:
            track = self.assess_track(
                seg["path"], clean_tr, feedback=False, lang=lang,
                feedback_mode="local", truncate=False, apply_preprocess=False,
            )
        except Exception as assess_err:
            print(f"[{role}] assess_track notice for turn {seg['index']} ('{clean_tr[:40]}'): {assess_err}. Bảo toàn turn với scored=False.")
            return _make_unscored_turn(
                seg,
                clean_tr,
                role=role,
                eligibility_reason=f"assess_error: {str(assess_err)[:40]}",
            ), False

        return {
            "index": seg["index"],
            "start_sec": seg["start_sec"],
            "end_sec": seg["end_sec"],
            "duration_sec": seg["duration_sec"],
            "audio": seg["path"],
            "turn_index": seg["index"],
            "role": role,
            "scored": True,
            "is_vietnamese": False,
            "language": "en",
            "eligibility_reason": "scored",
            **track,
        }, False

    def assess_track(
        self,
        audio: Union[str, Path],
        transcript: str,
        *,
        feedback: Optional[bool] = None,
        lang: Optional[str] = None,
        feedback_mode: str = "auto",
        truncate: bool = False,
        apply_preprocess: bool = True,
    ) -> Dict[str, Any]:
        """Score a single track using both models and generate transformer feedback."""
        fb = self.enable_feedback if feedback is None else feedback

        # ── Run pronunciation model (SpeechOcean762) ──
        pron_result = self.pronunciation.predict(
            str(audio), transcript, fb, lang,
            feedback_mode=feedback_mode, truncate=truncate,
            apply_preprocess=apply_preprocess,
        )

        pron_scores = pron_result["scores"]
        pron_errors = pron_result["errors"]

        # ── Run L2-MDD model for full-phoneme scanning & ASHA error diagnosis ──
        # L2-MDD is the EXCLUSIVE authority for phoneme error detection and word error marking.
        final_errors = {"phonemes": [], "words": []}
        words_detail = []
        if self.l2_mdd is not None:
            try:
                l2_result = self.l2_mdd.predict(
                    str(audio), transcript, False, lang,
                    feedback_mode=feedback_mode, truncate=truncate,
                    apply_preprocess=apply_preprocess,
                )
                l2_errors = l2_result.get("errors", {})
                # Hybrid error detection:
                # 1. Use L2-MDD if it detected phonological errors (contains rich ASHA diagnostic tips)
                # 2. If L2-MDD detected 0 errors but SpeechOcean detected acoustic errors, use SpeechOcean
                l2_phones = l2_errors.get("phonemes") or []
                so_phones = (pron_errors or {}).get("phonemes") or []
                if l2_phones:
                    final_errors = l2_errors
                    words_detail = l2_result.get("words_detail") or []
                elif so_phones:
                    final_errors = pron_errors
                    words_detail = pron_result.get("words_detail") or []
                else:
                    final_errors = {"phonemes": [], "words": []}
                    words_detail = l2_result.get("words_detail") or pron_result.get("words_detail") or []
            except Exception as e:
                print(f"  [WARN] L2-MDD phoneme scan failed: {e}")
                words_detail = pron_result.get("words_detail") or []
                final_errors = pron_errors or {"phonemes": [], "words": []}
        else:
            # Fallback only if L2-MDD model checkpoint is unavailable
            final_errors = pron_errors or {"phonemes": [], "words": []}
            words_detail = pron_result.get("words_detail") or []

        # ── Hiệu chuẩn & Phân hóa điểm số dựa trên lỗi âm vị thực tế từ L2-MDD ──
        final_scores = self.calibrate_scores_with_phonemes(
            pron_scores, words_detail, config=self.config.get("scorer")
        )

        # Generate per-turn feedback
        transformer_feedback = PronunciationScorer.generate_transformer_feedback(
            scores_pronunciation=final_scores,
            errors_pronunciation=final_errors,
            transcript=transcript,
        )

        return {
            "audio": str(audio),
            "transcript": transcript,
            "scores": final_scores,
            "errors": final_errors,
            "alignments": pron_result.get("alignments"),
            "feedback": pron_result.get("feedback"),
            "feedback_source": pron_result.get("feedback_source"),
            "transformer_feedback": transformer_feedback,
            "l2_mdd_feedback": None,
            "words_detail": words_detail,
        }

    def _assess_speaker_sentences(
        self,
        *,
        output_dir: Path,
        speaker: str,
        source_audio: Union[str, Path],
        diarize_segments: list,
        use_asr: bool = True,
        feedback: Optional[bool] = None,
        lang: Optional[str] = None,
        score: bool = True,
        role: Optional[str] = None,
    ) -> Dict[str, Any]:
        fb = self.enable_feedback if feedback is None else feedback
        if not score:
            fb = False
        role = role or speaker
        segments = self._collect_speaker_segments(
            output_dir,
            speaker,
            source_audio=source_audio,
            diarize_segments=diarize_segments,
        )

        if not segments:
            print(f"[{role}] Không tách được lượt nói nào từ audio", flush=True)
            return {
                "role": role,
                "scored": score,
                "sentences": [],
                "sentence_count": 0,
                "vietnamese_sentence_count": 0,
                "transcript": "",
                "transcript_lines": [],
                "scores": {},
                "message": f"{role}: không tách được lượt nói nào từ audio",
            }

        sentences: List[Dict[str, Any]] = []
        filtered_vi = 0
        for seg in segments:
            item, was_vi = self._process_segment(
                seg, use_asr=use_asr, lang=lang, score=score, role=role,
            )
            if was_vi:
                filtered_vi += 1
            if item:
                sentences.append(item)

        # Merge consecutive incomplete turns split by teacher prompts (e.g. 'My favorite drink is...' + prompt + 'Coca-Cola')
        merged_sentences: List[Dict[str, Any]] = []
        i = 0
        while i < len(sentences):
            curr = sentences[i]
            if (
                i + 1 < len(sentences)
                and curr.get("scored")
                and _is_incomplete_turn(curr.get("transcript", ""))
            ):
                nxt = sentences[i + 1]
                gap = nxt.get("start_sec", 0) - curr.get("end_sec", 0)
                if 0 <= gap <= 3.5 and nxt.get("scored"):
                    try:
                        s_time = curr["start_sec"]
                        e_time = nxt["end_sec"]
                        comb_path = output_dir / f"merged_turn_{curr['index']:03d}_{nxt['index']:03d}.wav"
                        from data.audio_preprocess import load_waveform, save_audio_file, PreprocessConfig
                        p_cfg = PreprocessConfig.from_dict(self.config.get("audio_preprocess"))
                        p_cfg.denoise = False
                        wav = load_waveform(source_audio, p_cfg)
                        sr = p_cfg.sample_rate
                        mono = wav if wav.dim() == 1 else wav.mean(0)
                        s_samp = max(0, int(s_time * sr))
                        e_samp = min(mono.shape[0], int(e_time * sr))
                        clip = mono[s_samp:e_samp].contiguous().unsqueeze(0)
                        save_audio_file(comb_path, clip, sr)

                        comb_seg = {
                            "index": curr["index"],
                            "start_sec": s_time,
                            "end_sec": e_time,
                            "duration_sec": round((e_samp - s_samp) / sr, 3),
                            "path": str(comb_path),
                        }
                        comb_item, _ = self._process_segment(
                            comb_seg, use_asr=use_asr, lang=lang, score=score, role=role,
                        )
                        if comb_item:
                            merged_sentences.append(comb_item)
                            i += 2
                            continue
                    except Exception as me:
                        print(f"[{role}] Turn merge notice: {me}", flush=True)
            merged_sentences.append(curr)
            i += 1
        sentences = merged_sentences

        if not sentences and use_asr:
            print(f"[{role}] Không có lượt nói hợp lệ sau ASR ({len(segments)} turn, {filtered_vi} đoạn tiếng Việt)", flush=True)
            return {
                "role": role,
                "scored": score,
                "sentences": [],
                "sentence_count": 0,
                "vietnamese_sentence_count": 0,
                "transcript": "",
                "transcript_lines": [],
                "scores": {},
                "filtered_vi_count": filtered_vi,
                "message": f"{role}: không có câu nói tiếng Anh hợp lệ sau nhận diện",
            }

        summary = _build_summary(
            sentences,
            filtered_vi=filtered_vi,
            config=self.config.get("scorer"),
        )
        return {
            "role": role,
            "scored": score,
            "sentences": sentences,
            **summary,
        }

    def assess_conversation(
        self,
        audio: Union[str, Path],
        *,
        diarize_output_dir: Optional[Union[str, Path]] = None,
        use_asr: bool = True,
        feedback: Optional[bool] = None,
        lang: Optional[str] = None,
        teacher_voice: Optional[Union[str, Path]] = None,
        teacher_embedding: Optional[Any] = None,
        student_embedding: Optional[Any] = None,
        score_teacher: bool = False,
    ) -> Dict[str, Any]:
        """Diarize A/B → split each track by silence → score every sentence."""
        fb = self.enable_feedback if feedback is None else feedback
        audio = Path(audio)

        base_dir = Path(diarize_output_dir or audio.parent / f"{audio.stem}_split")
        split = self._diarize_two_speakers(
            audio,
            base_dir,
            teacher_reference_path=teacher_voice,
            teacher_embedding=teacher_embedding,
            student_embedding=student_embedding,
        )

        has_refs = (
            teacher_voice is not None
            or teacher_embedding is not None
            or student_embedding is not None
            or split.get("teacher_segments") is not None
            or split.get("student_segments") is not None
        )

        if has_refs:
            teacher_dir = base_dir / "teacher_sentences"
            student_dir = base_dir / "student_sentences"
            teacher_segs = split.get("teacher_segments") or []
            student_segs = split.get("student_segments") or []
            
            all_segs = split.get("segments") or []
            
            teacher = self._assess_speaker_sentences(
                output_dir=teacher_dir,
                speaker="Teacher",
                source_audio=audio,
                diarize_segments=all_segs if all_segs else teacher_segs,
                use_asr=use_asr,
                feedback=False,
                lang=lang,
                score=score_teacher and len(teacher_segs) > 0,
                role="teacher",
            )
            student = self._assess_speaker_sentences(
                output_dir=student_dir,
                speaker="Student",
                source_audio=audio,
                diarize_segments=all_segs if all_segs else student_segs,
                use_asr=use_asr,
                feedback=False,
                lang=lang,
                score=True,
                role="student",
            )
            dialogue = _build_dialogue(teacher.get("sentences", []), student.get("sentences", []))
            if fb and student.get("sentences"):
                lang_fb = _build_summary(
                    student["sentences"],
                    filtered_vi=student.get("filtered_vi_count", 0),
                    config=self.config.get("scorer"),
                )
                student["feedback"] = lang_fb["feedback"]
                student["feedback_source"] = lang_fb["feedback_source"]
                student["pronunciation_feedback"] = lang_fb.get("pronunciation_feedback")
                student["pronunciation_feedback_source"] = lang_fb.get("pronunciation_feedback_source")

            # ── Build overall transformer feedback for the student (SpeechOcean only) ──
            overall_tf = PronunciationScorer.generate_transformer_feedback(
                scores_pronunciation=student.get("scores", {}),
                errors_pronunciation=None,
                transcript=student.get("transcript", ""),
            )

            return {
                "mode": "teacher_student",
                "source_audio": str(audio),
                "duration_sec": split["duration_sec"],
                "teacher": teacher,
                "student": student,
                "dialogue": dialogue,
                "has_l2_mdd": self.l2_mdd is not None,
                "overall_transformer_feedback": overall_tf,
                "diarization": {
                    "teacher": str(split["teacher"]) if split.get("teacher") else None,
                    "student": str(split["student"]) if split.get("student") else None,
                },
            }

        speakers: Dict[str, Any] = {}
        for key, spk in [("A", "Speaker A"), ("B", "Speaker B")]:
            sent_dir = base_dir / f"speaker_{key}_sentences"
            speakers[key] = self._assess_speaker_sentences(
                output_dir=sent_dir,
                speaker=spk,
                use_asr=use_asr,
                feedback=fb,
                lang=lang,
                source_audio=audio,
                diarize_segments=split.get("segments"),
            )

        return {
            "mode": "diarize",
            "source_audio": str(audio),
            "duration_sec": split["duration_sec"],
            "speakers": speakers,
            "has_l2_mdd": self.l2_mdd is not None,
            "diarization": {
                "teacher": str(split["teacher"]) if split.get("teacher") else None,
                "student": str(split["student"]) if split.get("student") else None,
            },
        }

    def assess_single_speaker(
        self,
        audio: Union[str, Path],
        *,
        output_dir: Optional[Union[str, Path]] = None,
        use_asr: bool = True,
        feedback: Optional[bool] = None,
        lang: Optional[str] = None,
        role: str = "student",
        reference_text: Optional[str] = None,
        task_type: Optional[str] = None,
    ) -> Dict[str, Any]:
        """Chấm điểm phát âm trực tiếp cho 1 người nói (Luyện tập / Thi thử) KHÔNG cần tách giọng Diarization."""
        fb = self.enable_feedback if feedback is None else feedback
        audio = Path(audio)

        # Voice Activity Detection (VAD) check toàn bộ file audio trước khi tách câu và đưa vào Whisper
        from data.audio_preprocess import load_audio_file
        try:
            wav_check, _ = load_audio_file(audio)
            if wav_check.numel() > 0:
                mono_check = wav_check.mean(0) if wav_check.dim() > 1 else wav_check
                peak_amp = float(mono_check.abs().max())
                rms_amp = float(torch.sqrt(torch.mean(mono_check ** 2)))
                if peak_amp < 0.015 or rms_amp < 0.003:
                    print(f"[{role}] VAD: Không phát hiện tiếng người trong audio (peak={peak_amp:.4f}, rms={rms_amp:.4f}). Bỏ qua Whisper.", flush=True)
                    student_data = {
                        "role": role,
                        "scored": True,
                        "sentences": [],
                        "sentence_count": 0,
                        "vietnamese_sentence_count": 0,
                        "transcript": "",
                        "transcript_lines": [],
                        "scores": {"total": 0.0, "accuracy": 0.0, "fluency": 0.0, "prosodic": 0.0},
                        "filtered_vi_count": 0,
                        "message": f"{role}: Không phát hiện tiếng người trong bản ghi âm (VAD)",
                    }
                    return {
                        "role": role,
                        "audio": str(audio),
                        "student": student_data,
                        "dialogue": {"turns": [], "has_l2_mdd": self.l2_mdd is not None},
                        "has_l2_mdd": self.l2_mdd is not None,
                    }
        except Exception as vad_e:
            print(f"[{role}] VAD check warning: {vad_e}", flush=True)

        # Trường hợp Read Aloud: đã có sẵn văn bản bài đọc chuẩn (reference_text)
        # KHÔNG CẦN DÙNG WHISPER để nhận dạng đoán chữ! Đưa thẳng văn bản bài đọc vào assess_track
        # để WavLM + SpeechOcean762 + L2-MDD so khớp âm vị học (forced-alignment).
        if task_type == "read_aloud" and reference_text and reference_text.strip():
            clean_ref = reference_text.strip()
            print(f"[{role}] Read Aloud Mode: Bỏ qua Whisper ASR, sử dụng trực tiếp văn bản đề bài làm target text: {clean_ref[:60]}...", flush=True)
            try:
                track = self.assess_track(
                    audio,
                    clean_ref,
                    feedback=fb,
                    lang=lang,
                    feedback_mode=self.feedback_mode,
                )
                sentence_item = {
                    "index": 0,
                    "start_sec": 0.0,
                    "end_sec": 0.0,
                    "duration_sec": 0.0,
                    "audio": str(audio),
                    "turn_index": 0,
                    "transcript": clean_ref,
                    "role": role,
                    "scored": True,
                    "is_vietnamese": False,
                    "language": "en",
                    "eligibility_reason": "scored",
                    "scores": track["scores"],
                    "errors": track["errors"],
                    "words_detail": track.get("words_detail", []),
                    "transformer_feedback": track.get("transformer_feedback"),
                    "l2_mdd_feedback": None,
                    "feedback": track.get("feedback"),
                    "target_text_source": "provided_passage",
                }
                sentences = [sentence_item]
                summary = _build_summary(
                    sentences,
                    filtered_vi=0,
                )
                student_data = {
                    "role": role,
                    "scored": True,
                    "sentences": sentences,
                    **summary,
                }
                return {
                    "role": role,
                    "audio": str(audio),
                    "student": student_data,
                    "dialogue": {
                        "turns": [_serialize_dialogue_turn(sentence_item, role="student")],
                        "student_turns": sentences,
                        "has_l2_mdd": self.l2_mdd is not None,
                    },
                    "has_l2_mdd": self.l2_mdd is not None,
                    "overall_transformer_feedback": track.get("transformer_feedback"),
                }
            except Exception as e:
                print(f"[{role}] Read Aloud direct scoring warning: {e}. Fallback to segment-based assessment.", flush=True)

        base_dir = Path(output_dir or audio.parent / f"{audio.stem}_single_split")
        sent_dir = base_dir / "sentences"
        sent_dir.mkdir(parents=True, exist_ok=True)

        split_cfg = self.config.get("sentence_split") or self.config.get("silence_split")
        segments = split_audio_file(
            audio,
            sent_dir,
            preprocess=self.preprocess,
            split_cfg=split_cfg,
            prefix=f"{role.lower()}_turn",
        )

        if not segments:
            # Fallback nếu audio quá ngắn hoặc không tách được khoảng lặng
            segments = [{
                "index": 0,
                "start_sec": 0.0,
                "end_sec": 0.0,
                "duration_sec": 0.0,
                "path": str(audio),
            }]

        sentences: List[Dict[str, Any]] = []
        filtered_vi = 0
        for seg in segments:
            item, was_vi = self._process_segment(
                seg, use_asr=use_asr, lang=lang, score=True, role=role,
            )
            if was_vi:
                filtered_vi += 1
            if item:
                sentences.append(item)

        if not sentences:
            print(f"[{role}] Không có lượt nói hợp lệ sau nhận diện", flush=True)
            student_data = {
                "role": role,
                "scored": True,
                "sentences": [],
                "sentence_count": 0,
                "vietnamese_sentence_count": 0,
                "transcript": "",
                "transcript_lines": [],
                "scores": {"total": 0.0, "accuracy": 0.0, "fluency": 0.0, "prosodic": 0.0},
                "filtered_vi_count": filtered_vi,
                "message": f"{role}: không có lượt nói hợp lệ sau nhận diện",
            }
        else:
            summary = _build_summary(
                sentences,
                filtered_vi=filtered_vi,
            )
            student_data = {
                "role": role,
                "scored": True,
                "sentences": sentences,
                **summary,
            }

        dialogue = {
            "turns": [_serialize_dialogue_turn(s, role="student") for s in sentences],
            "student_turns": sentences,
        }

        overall_tf = PronunciationScorer.generate_transformer_feedback(
            scores_pronunciation=student_data.get("scores", {}),
            errors_pronunciation=None,
            transcript=student_data.get("transcript", ""),
        )

        return {
            "mode": "single_speaker",
            "source_audio": str(audio),
            "duration_sec": sum(s.get("duration_sec", 0) for s in segments),
            "student": student_data,
            "dialogue": dialogue,
            "has_l2_mdd": self.l2_mdd is not None,
            "overall_transformer_feedback": overall_tf,
            "diarization": {
                "teacher": None,
                "student": str(audio),
            },
        }


def main():
    p = argparse.ArgumentParser(description="2-speaker speaking evaluation")
    p.add_argument("--audio", required=True)
    p.add_argument("--config", default=str(PRONUNCIATION_CONFIG))
    p.add_argument("--pronunciation-ckpt", default=None)
    p.add_argument("--l2-mdd-ckpt", default=None)
    p.add_argument("--no-feedback", action="store_true")
    p.add_argument("--lang", choices=["vi", "en"], default="vi")
    p.add_argument("--output", default=None)
    p.add_argument("--device", default=None)
    args = p.parse_args()

    pipe = SpeakingPipeline(
        args.config,
        args.device,
        args.pronunciation_ckpt,
        l2_mdd_ckpt=args.l2_mdd_ckpt,
        enable_feedback=not args.no_feedback,
    )
    result = pipe.assess_conversation(args.audio, lang=args.lang)

    text = json.dumps(result, ensure_ascii=False, indent=2, default=str)
    print(text)
    out = args.output or str(ROOT / "logs" / "result.json")
    Path(out).parent.mkdir(parents=True, exist_ok=True)
    Path(out).write_text(text, encoding="utf-8")
    print(f"Saved: {out}")


if __name__ == "__main__":
    main()


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/infer', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/infer/pronunciation.py
"""Pronunciation scoring inference (SpeechOcean762 model + L2-MDD model)."""

from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any, Dict, Optional

import torch
import yaml

from data.audio_preprocess import PreprocessConfig, load_waveform, truncate_waveform
from data.cmudict import CMUDict
from env_config import load_project_env
from models.checkpoint_utils import load_model_weights, resolve_checkpoint
from models.pronunciation_model import PronunciationAssessmentModel
from models.pronunciation_scorer import PronunciationScorer
from paths import PRONUNCIATION_CONFIG

load_project_env()


def _words_to_phones_and_ranges(cmudict: CMUDict, text: str):
    groups = cmudict.words_to_phoneme_groups(text)
    tokens, words, ranges = [], [], []
    for g in groups:
        s = len(tokens)
        tokens.extend(g["phones"])
        words.append(g["word"])
        ranges.append((s, len(tokens)))
    return tokens, words, ranges


def _detect_lora_cfg_from_sd(sd: Dict[str, Any]) -> Dict[str, Any]:
    """Inspect a state_dict to extract LoRA rank & target module names automatically."""
    target_modules = []
    seen = set()
    lora_r = 8
    for k, v in sd.items():
        if ".lora_A." in k and k.endswith(".weight"):
            prefix = k.split(".lora_A.")[0]
            mod_name = prefix.rsplit(".", 1)[-1]
            if mod_name not in seen:
                seen.add(mod_name)
                target_modules.append(mod_name)
            if hasattr(v, "shape") and len(v.shape) == 2:
                lora_r = int(v.shape[0])
    use_lora = bool(target_modules) or any("lora" in k.lower() for k in sd.keys())
    return {
        "use_lora": use_lora,
        "lora_r": lora_r,
        "lora_alpha": lora_r * 2,
        "lora_target_modules": target_modules or ["q_proj", "v_proj"],
    }


class Predictor:
    def __init__(
        self,
        config_path: str | Path | None = None,
        checkpoint: Optional[str] = None,
        device: Optional[str] = None,
        load_progress: Optional[Any] = None,
        model_step: str = "pronunciation",
        ckpt_step: str = "pronunciation_ckpt",
        wavlm_name: str = "microsoft/wavlm-large",
    ):
        config_path = Path(config_path or PRONUNCIATION_CONFIG)
        with open(config_path, encoding="utf-8") as f:
            self.config = yaml.safe_load(f)
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.preprocess = PreprocessConfig.from_dict(self.config.get("audio_preprocess"))
        if load_progress is not None:
            load_progress.start(model_step, wavlm_name)
        try:
            ckpt_candidate = checkpoint
            if ckpt_candidate is None:
                try:
                    ckpt_candidate = resolve_checkpoint(self.config, model="pronunciation")
                except Exception:
                    ckpt_candidate = None

            if ckpt_candidate and os.path.isfile(ckpt_candidate):
                from models.checkpoint_utils import load_state_dict as _load_sd
                sd_sample = _load_sd(ckpt_candidate, "cpu")
                lora_info = _detect_lora_cfg_from_sd(sd_sample)
                if lora_info["use_lora"]:
                    if "wavlm" not in self.config:
                        self.config["wavlm"] = {}
                    self.config["wavlm"]["use_lora"] = True
                    self.config["wavlm"]["lora_r"] = lora_info["lora_r"]
                    self.config["wavlm"].setdefault("lora_alpha", lora_info["lora_alpha"])
                    self.config["wavlm"]["lora_target_modules"] = lora_info["lora_target_modules"]

            self.model = PronunciationAssessmentModel(self.config).to(self.device)
            if load_progress is not None:
                load_progress.finish(model_step)
                load_progress.start(ckpt_step, "pronunciation.pt")
            try:
                self._ckpt_path = load_model_weights(
                    self.model, self.config, model_kind="pronunciation", explicit=checkpoint, device=self.device
                )
            except FileNotFoundError as e:
                print(f"Warning: {e}")
                self._ckpt_path = None
            except Exception as exc:
                if load_progress is not None:
                    load_progress.fail(ckpt_step, str(exc))
                raise
            if load_progress is not None:
                load_progress.finish(ckpt_step, str(self._ckpt_path or "không có checkpoint"))
        except Exception as exc:
            if load_progress is not None:
                load_progress.fail(model_step, str(exc))
            raise
        self.model.eval()
        self.sr = self.preprocess.sample_rate
        inf = self.config.get("inference") or {}
        md = inf.get("max_duration_sec")
        if md is None and "max_duration_sec" not in inf:
            ds = self.config.get("train", {}).get("dataset", {})
            md = ds.get("max_duration_sec")
        self.max_duration_sec = None if md is None or md <= 0 else float(md)
        self.cmudict = CMUDict(self.config["paths"].get("cmudict_path"))
        mt = self.config["multitask"]
        scorer_cfg = self.config.get("scorer") or {}
        self.scorer = PronunciationScorer(
            score_scale=mt.get("score_scale", 5.0),
            weights=scorer_cfg.get("weights"),
            phoneme_low_threshold=scorer_cfg.get("phoneme_low_threshold", 1.4),
            calibration=scorer_cfg.get("calibration"),
        )

    def _phones_from_text(self, text: str):
        return _words_to_phones_and_ranges(self.cmudict, text)

    @torch.no_grad()
    def predict(
        self,
        audio: str,
        transcript: str,
        feedback: bool = True,
        lang: Optional[str] = None,
        feedback_mode: str = "auto",
        truncate: bool = False,
        *,
        apply_preprocess: bool = True,
    ) -> Dict[str, Any]:
        wav = load_waveform(audio, self.preprocess, apply_preprocess=apply_preprocess)
        truncated = False
        if truncate and self.max_duration_sec:
            wav, truncated = truncate_waveform(wav, self.sr, self.max_duration_sec)
        duration_sec = float(wav.shape[0]) / float(self.sr) if self.sr else None
        tokens, words, ranges = self._phones_from_text(transcript)
        if not tokens:
            raise ValueError(
                f"Không tra được phoneme cho transcript (CMUdict): {transcript!r}"
            )
        out = self.model(
            wav.unsqueeze(0).to(self.device),
            torch.tensor([wav.shape[0]], device=self.device),
            [tokens],
            [ranges],
            return_alignments=True,
        )
        pred = out["predictions"][0]
        scores = self.scorer.aggregate_utterance(pred, duration_sec=duration_sec)
        scores["final"] = self.scorer.final_score(pred, duration_sec=duration_sec)
        alignments = [
            {
                "phoneme": a.phoneme,
                "start_frame": a.start_frame,
                "end_frame": a.end_frame,
                "confidence": a.confidence,
            }
            for a in (out.get("alignments") or [[]])[0]
        ]
        
        errors = self.scorer.find_errors(pred, tokens, words, ranges, alignments=alignments)
        words_detail = self.scorer.build_words_detail(pred, tokens, words, ranges)
        result = {
            "transcript": transcript,
            "scores": scores,
            "errors": errors,
            "words_detail": words_detail,
            "truncated": truncated,
            "max_duration_sec": self.max_duration_sec,
            "alignments": alignments,
            "feedback": None,
            "feedback_source": None,
        }
        if feedback:
            # Feedback is handled in the Colab notebook directly
            pass
        return result


class L2MDDPredictor:
    """Predictor using the L2-MDD checkpoint for full-phoneme scanning and ASHA clinical diagnosis."""

    def __init__(
        self,
        config_path: str | Path | None = None,
        checkpoint: Optional[str] = None,
        device: Optional[str] = None,
        load_progress: Optional[Any] = None,
        model_step: str = "l2_mdd",
        ckpt_step: str = "l2_mdd_ckpt",
    ):
        config_path = Path(config_path or PRONUNCIATION_CONFIG)
        with open(config_path, encoding="utf-8") as f:
            self.config = yaml.safe_load(f)
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.preprocess = PreprocessConfig.from_dict(self.config.get("audio_preprocess"))

        if load_progress is not None:
            load_progress.start(model_step, "L2-MDD Model")
        try:
            # Load WavLM model name with graceful fallback if path doesn't exist
            wavlm_cfg = self.config.get("wavlm", {})
            wavlm_name = wavlm_cfg.get("model_name", "microsoft/wavlm-large")
            if wavlm_name and not os.path.exists(wavlm_name):
                wavlm_name = "microsoft/wavlm-large"

            from models.l2_mdd_model import L2MDDModel
            from models.checkpoint_utils import load_state_dict as _load_sd

            ckpt_path = None
            sd = None
            lora_info = {"use_lora": False, "lora_r": 8, "lora_alpha": 16, "lora_target_modules": ["q_proj", "v_proj"]}
            try:
                ckpt_path = resolve_checkpoint(self.config, model="l2_mdd", explicit=checkpoint)
                if ckpt_path and os.path.isfile(ckpt_path):
                    sd = _load_sd(ckpt_path, self.device)
                    lora_info = _detect_lora_cfg_from_sd(sd)
            except FileNotFoundError as e:
                print(f"Warning (L2-MDD): {e}")

            use_canon_cond = (sd is None) or any("canon_emb." in k for k in sd.keys())
            self.model = L2MDDModel(
                wavlm_name=wavlm_name,
                ff_dim=2048,
                use_lora=lora_info["use_lora"],
                lora_r=lora_info["lora_r"],
                lora_alpha=lora_info["lora_alpha"],
                lora_target_modules=lora_info["lora_target_modules"],
                use_canon_conditioning=use_canon_cond,
            ).to(self.device)

            if load_progress is not None:
                load_progress.finish(model_step)
                load_progress.start(ckpt_step, "l2_mdd_best.pt")

            if sd is not None:
                self.model.load_state_dict(sd, strict=False)
                self._ckpt_path = ckpt_path
            else:
                self._ckpt_path = None

            if load_progress is not None:
                load_progress.finish(ckpt_step, str(self._ckpt_path or "không có checkpoint"))
        except Exception as exc:
            if load_progress is not None:
                load_progress.fail(model_step, str(exc))
            raise

        self.model.eval()
        self.sr = self.preprocess.sample_rate
        inf = self.config.get("inference") or {}
        md = inf.get("max_duration_sec")
        self.max_duration_sec = None if md is None or md <= 0 else float(md)
        self.cmudict = CMUDict(self.config["paths"].get("cmudict_path"))

    def _phones_from_text(self, text: str):
        return _words_to_phones_and_ranges(self.cmudict, text)

    @torch.no_grad()
    def scan_phonemes(
        self,
        audio: str,
        transcript: str,
        sensitivity: float = 0.25,
        apply_preprocess: bool = True,
    ) -> Dict[str, Any]:
        """Scan all phonemes in the utterance using L2-MDD and generate ASHA diagnoses."""
        wav = load_waveform(audio, self.preprocess, apply_preprocess=apply_preprocess)
        tokens, words, ranges = self._phones_from_text(transcript)
        if not tokens:
            return {
                "suspicious_phonemes": [],
                "all_phonemes": [],
                "num_suspicious": 0,
                "words": [],
                "word_ipas": [],
                "word_phone_ranges": [],
            }

        from models.pronunciation_scorer import phones_to_word_ipa
        word_ipas = [phones_to_word_ipa(tokens[s:e]) for s, e in ranges]

        res = self.model.scan_phonemes(
            waveform=wav.to(self.device),
            wav_length=torch.tensor([wav.shape[0]], device=self.device),
            phoneme_tokens=tokens,
            words=words,
            word_phone_ranges=ranges,
            word_ipas=word_ipas,
            sensitivity_threshold=sensitivity,
        )
        res["words"] = words
        res["word_ipas"] = word_ipas
        res["word_phone_ranges"] = ranges
        return res

    @torch.no_grad()
    def predict(
        self,
        audio: str,
        transcript: str,
        feedback: bool = True,
        lang: Optional[str] = None,
        feedback_mode: str = "auto",
        truncate: bool = False,
        *,
        apply_preprocess: bool = True,
    ) -> Dict[str, Any]:
        """Scan phonemes and package into standard evaluation result."""
        scan_res = self.scan_phonemes(audio, transcript, apply_preprocess=apply_preprocess)
        suspicious = scan_res.get("suspicious_phonemes", [])
        all_phones = scan_res.get("all_phonemes", [])
        words = scan_res.get("words", [])
        word_ipas = scan_res.get("word_ipas", [])
        ranges = scan_res.get("word_phone_ranges", [])

        # Build words_detail directly and exclusively from L2-MDD phoneme scan (SpeechOcean never sets word status)
        words_detail = []
        for w_idx, (ws, we) in enumerate(ranges):
            w_text = words[w_idx] if w_idx < len(words) else ""
            w_ipa = word_ipas[w_idx] if w_idx < len(word_ipas) else ""
            w_phones = all_phones[ws:we] if ws < we and we <= len(all_phones) else []

            phones_list = []
            w_has_err = False
            w_has_crit = False
            for ph in w_phones:
                is_susp = ph.get("is_suspicious", False)
                is_crit = is_susp and ph.get("severity") in ("critical", "bad")
                p_status = "bad" if is_crit else ("warning" if is_susp else "good")
                if is_susp:
                    w_has_err = True
                if is_crit:
                    w_has_crit = True
                phones_list.append({
                    "phoneme": ph.get("target_phone", ""),
                    "ipa": ph.get("target_ipa", ""),
                    "actual_ipa": ph.get("actual_ipa", ""),
                    "status": p_status,
                    "tip": ph.get("articulatory_tip", ""),
                    "rule_name_vi": ph.get("rule_name_vi", ""),
                })

            w_status = "bad" if w_has_crit else ("warning" if w_has_err else "good")
            words_detail.append({
                "word": w_text,
                "word_ipa": w_ipa,
                "status": w_status,
                "phonemes": phones_list,
            })

        # Build errors dict formatted for downstream use
        errors = {
            "phonemes": suspicious,
            "words": [],
        }

        # Collect unique suspicious words with their diagnostic notes
        seen_words = set()
        for p in suspicious:
            w_text = p.get("word")
            if w_text and w_text not in seen_words:
                seen_words.add(w_text)
                errors["words"].append({
                    "word": w_text,
                    "word_ipa": p.get("word_ipa", ""),
                    "severity": p.get("severity", "warning"),
                    "rule_name_vi": p.get("rule_name_vi", ""),
                    "articulatory_tip": p.get("articulatory_tip", ""),
                })

        return {
            "transcript": transcript,
            "errors": errors,
            "words_detail": words_detail,
            "scan_result": scan_res,
            "num_suspicious": scan_res.get("num_suspicious", 0),
            "feedback": None,
            "feedback_source": "l2_mdd_asha",
        }


def main():
    import argparse
    p = argparse.ArgumentParser(description="Pronunciation scoring only")
    p.add_argument("--audio", required=True)
    p.add_argument("--transcript", required=True)
    p.add_argument("--config", default=str(PRONUNCIATION_CONFIG))
    p.add_argument("--checkpoint", default=None)
    p.add_argument("--no-feedback", action="store_true")
    p.add_argument("--output", default=None)
    args = p.parse_args()
    r = Predictor(args.config, args.checkpoint).predict(args.audio, args.transcript, not args.no_feedback)
    text = json.dumps(r, ensure_ascii=False, indent=2)
    print(text)
    if args.output:
        Path(args.output).write_text(text, encoding="utf-8")


if __name__ == "__main__":
    main()


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/infer', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/infer/transcribe.py
"""Speech-to-text for automatic transcript (Whisper Large-v3 Bilingual EN/VI)."""

from __future__ import annotations

import os
import re
import threading
from pathlib import Path
from typing import Any, Dict, Optional, Sequence, Tuple, Union

import torch
import torch.nn.functional as F
import torchaudio
import yaml

from data.audio_preprocess import load_audio_file
from data.cmudict import expand_numbers_in_text
from infer.device_utils import resolve_device
from paths import PRONUNCIATION_CONFIG, ROOT

PathLike = Union[str, Path]
WHISPER_SR = 16000
WHISPER_CHUNK_SEC = 30.0
WHISPER_DEFAULT_MODEL = "openai/whisper-large-v3"
LOCAL_WHISPER_DIR = ROOT / "pretrained_models" / "whisper-large-v3"
DEFAULT_ALLOWED_LANGUAGES = ("en", "vi")


def _local_whisper_ready(local_dir: Path = LOCAL_WHISPER_DIR) -> bool:
    if not local_dir.is_dir() or not (local_dir / "config.json").is_file():
        return False
    return any(
        (local_dir / fname).is_file()
        for fname in ("model.safetensors", "model.safetensors.index.json", "pytorch_model.bin")
    )


def _resolve_whisper_model_name(model_name: Optional[str]) -> str:
    raw = (model_name or "").strip()
    if not raw or raw == WHISPER_DEFAULT_MODEL:
        if _local_whisper_ready(LOCAL_WHISPER_DIR):
            return str(LOCAL_WHISPER_DIR)
        return WHISPER_DEFAULT_MODEL

    if os.path.exists(raw):
        return raw

    if raw.startswith("/") or raw.startswith("pretrained_models") or "\\" in raw or os.sep in raw:
        if _local_whisper_ready(LOCAL_WHISPER_DIR):
            return str(LOCAL_WHISPER_DIR)
        return WHISPER_DEFAULT_MODEL

    return raw

_transcriber: Optional["WhisperTranscriber"] = None
_transcriber_error: Optional[str] = None
_lock = threading.Lock()


def clean_whisper_hallucinations(text: str) -> str:
    """Clean common Whisper hallucinations, YouTube tags, and loop repetitions."""
    if not text:
        return ""

    # Remove bracketed/parenthesized annotations like [applause], (music), ***
    cleaned = re.sub(r"\[.*?\]|\(.*?\)", " ", text).strip()
    cleaned = re.sub(r"\*+", " ", cleaned).strip()

    # Hallucination phrases (subtitles, youtube, silence triggers)
    lower = cleaned.lower().strip(" .,!?")
    hallucination_phrases = {
        "thank you for watching", "thanks for watching", "please subscribe",
        "subtitles by", "like and subscribe", "see you next time", "bye bye",
        "thank you so much for watching", "watch next", "translated by",
    }
    for phrase in hallucination_phrases:
        if lower == phrase:
            return ""
        if lower.startswith(phrase + " "):
            cleaned = cleaned[len(phrase):].strip()
        if lower.endswith(" " + phrase):
            cleaned = cleaned[:-len(phrase)].strip()

    # Collapse repetitive loops (e.g. "and then and then and then")
    cleaned = re.sub(r"\b(\w+(?:\s+\w+){0,2})(?:\s+\1){2,}\b", r"\1", cleaned, flags=re.IGNORECASE)

    return " ".join(cleaned.split())


def _normalize_transcript(text: str, is_vietnamese: bool = False) -> str:
    """Normalize transcript for display or pronunciation assessment."""
    text = clean_whisper_hallucinations(text)
    if not text:
        return ""

    if is_vietnamese:
        # Preserve Vietnamese characters, accents, and case
        text = re.sub(r"[^\w\s\u00C0-\u1EF9']", " ", text)
        return " ".join(text.split())

    # English: expand numbers, standardize apostrophes, uppercase
    text = expand_numbers_in_text(text)
    text = text.replace("’", "'").replace("‘", "'")
    text = re.sub(r"[^\w\s']", " ", text.upper())
    return " ".join(text.split())


def _load_asr_config() -> Dict[str, Any]:
    if PRONUNCIATION_CONFIG.is_file():
        with open(PRONUNCIATION_CONFIG, encoding="utf-8") as f:
            cfg = yaml.safe_load(f) or {}
        return cfg.get("asr") or {}
    return {}


def _parse_dtype(name: Optional[str], device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    if not name or name == "float16":
        return torch.float16
    if name == "bfloat16":
        return torch.bfloat16
    return torch.float32


class WhisperTranscriber:
    """
    Bilingual Whisper ASR (EN/VI) — default: openai/whisper-large-v3 (128-mel).
    Restricts language detection to allowed_languages (default: ('en', 'vi'))
    and returns both acoustic probabilities and combined Vietnamese percentage.
    """

    def __init__(
        self,
        model_name: str = WHISPER_DEFAULT_MODEL,
        language: str = "auto",
        device: Optional[str] = None,
        load_progress: Optional[Any] = None,
        torch_dtype: Optional[str] = "float16",
        max_new_tokens: int = 440,
        asr_config: Optional[Dict[str, Any]] = None,
        allowed_languages: Optional[Sequence[str]] = None,
    ):
        from transformers import WhisperForConditionalGeneration, WhisperProcessor

        asr_cfg = asr_config if asr_config is not None else _load_asr_config()
        self.device = torch.device(
            device or resolve_device({"asr": asr_cfg}, asr_cfg.get("device"))
        )
        self.language = (language or asr_cfg.get("language", "auto")).strip().lower()

        raw_allowed = allowed_languages or asr_cfg.get("allowed_languages") or DEFAULT_ALLOWED_LANGUAGES
        norm_allowed = []
        for code in raw_allowed:
            c = str(code).strip().lower()
            if c and c not in norm_allowed:
                norm_allowed.append(c)
        self.allowed_languages: Tuple[str, ...] = tuple(norm_allowed or DEFAULT_ALLOWED_LANGUAGES)

        self.model_name = _resolve_whisper_model_name(model_name)
        self.max_new_tokens = max_new_tokens
        dtype = _parse_dtype(torch_dtype, self.device)

        if load_progress is not None:
            load_progress.start("whisper_proc", self.model_name)
        self.processor = WhisperProcessor.from_pretrained(self.model_name)
        if load_progress is not None:
            load_progress.finish("whisper_proc", self.model_name)
            load_progress.start("whisper_model", self.model_name)

        load_kw: Dict[str, Any] = {}
        if self.device.type == "cuda" and dtype != torch.float32:
            load_kw["torch_dtype"] = dtype
        try:
            import accelerate  # noqa: F401
            load_kw["low_cpu_mem_usage"] = True
        except ImportError:
            pass

        self.model = WhisperForConditionalGeneration.from_pretrained(self.model_name, **load_kw)
        if getattr(self.model, "generation_config", None) is not None:
            self.model.generation_config.max_length = None
        self.model.to(self.device)
        self.model.eval()

        if load_progress is not None:
            load_progress.finish("whisper_model", self.model_name)

    def _get_allowed_lang_token_ids(self) -> Dict[str, int]:
        gen_cfg = getattr(self.model, "generation_config", None)
        lang_to_id = getattr(gen_cfg, "lang_to_id", None) or {}
        tok = getattr(self.processor, "tokenizer", None)
        unk_id = getattr(tok, "unk_token_id", None) if tok is not None else None
        mapping: Dict[str, int] = {}
        for code in self.allowed_languages:
            tid = lang_to_id.get(f"<|{code}|>")
            if tid is None:
                tid = lang_to_id.get(code)
            if tid is None and tok is not None and hasattr(tok, "convert_tokens_to_ids"):
                cand = tok.convert_tokens_to_ids(f"<|{code}|>")
                if isinstance(cand, int) and cand >= 0 and cand != unk_id:
                    tid = cand
            if isinstance(tid, int) and tid >= 0:
                mapping[code] = int(tid)
        return mapping

    def detect_language_probs(self, input_features: torch.Tensor) -> Tuple[str, Dict[str, float]]:
        """
        Detect language restricted strictly to self.allowed_languages (default: EN vs VI)
        and return (best_language, {lang_code: probability}).
        """
        allowed = self.allowed_languages or DEFAULT_ALLOWED_LANGUAGES
        default_lang = allowed[0]

        if str(self.model_name).lower().endswith(".en"):
            return "en", {c: (1.0 if c == "en" else 0.0) for c in allowed}

        tid_map = self._get_allowed_lang_token_ids()
        codes = [c for c in allowed if c in tid_map]

        if codes and callable(self.model):
            try:
                gen_cfg = getattr(self.model, "generation_config", None)
                start_id = getattr(gen_cfg, "decoder_start_token_id", None)
                if start_id is None:
                    tok = getattr(self.processor, "tokenizer", None)
                    if tok is not None and hasattr(tok, "convert_tokens_to_ids"):
                        cand = tok.convert_tokens_to_ids("<|startoftranscript|>")
                        if isinstance(cand, int) and cand >= 0:
                            start_id = cand
                if start_id is None:
                    start_id = 50258

                batch_size = int(input_features.shape[0])
                dec_ids = torch.full(
                    (batch_size, 1),
                    int(start_id),
                    device=input_features.device,
                    dtype=torch.long,
                )
                with torch.no_grad():
                    out = self.model(
                        input_features=input_features[:, :, :3000],
                        decoder_input_ids=dec_ids,
                        use_cache=False,
                    )
                logits = getattr(out, "logits", None)
                tids = [tid_map[c] for c in codes]
                if logits is not None and int(logits.shape[-1]) > max(tids):
                    step_logits = logits[:, -1, :].float()
                    sub_logits = step_logits[:, tids]
                    probs_tensor = F.softmax(sub_logits, dim=-1).mean(dim=0)
                    probs = {c: float(probs_tensor[i].item()) for i, c in enumerate(codes)}
                    for c in allowed:
                        probs.setdefault(c, 0.0)
                    best_lang = max(codes, key=lambda c: probs[c])
                    return best_lang, probs
            except Exception:
                pass

        try:
            tokens = self.model.detect_language(input_features)
            lang_str = self.processor.tokenizer.decode(tokens[0]).strip().strip("<|>").lower()
            if lang_str not in allowed:
                lang_str = default_lang
            probs = {c: (1.0 if c == lang_str else 0.0) for c in allowed}
            return lang_str, probs
        except Exception:
            probs = {c: (1.0 if c == default_lang else 0.0) for c in allowed}
            return default_lang, probs

    def detect_language(self, input_features: torch.Tensor) -> str:
        """Detect language code restricted to self.allowed_languages ('en' or 'vi')."""
        lang, _ = self.detect_language_probs(input_features)
        return lang

    def _generate_with_lang(
        self,
        input_features: torch.Tensor,
        forced_lang: Optional[str] = None,
    ) -> Tuple[str, str]:
        """Generate transcription and return (raw_text, detected_language)."""
        name = self.model_name.lower()
        max_pos = getattr(getattr(self.model, "config", None), "max_target_positions", 448)
        max_tokens = max(1, min(getattr(self, "max_new_tokens", 440), max_pos - 12))

        # English-only checkpoints have no language tokens
        if name.endswith(".en"):
            ids = self.model.generate(input_features, max_new_tokens=max_tokens)
            text = self.processor.batch_decode(ids, skip_special_tokens=True)[0]
            return text, "en"

        # Determine target language
        target_lang = forced_lang
        if not target_lang:
            if self.language in ("auto", "", "none"):
                target_lang = self.detect_language(input_features)
            else:
                target_lang = self.language

        # Decode with native language/task args first (required for whisper-large-v3 on transformers >= 4.41/5.x)
        try:
            ids = self.model.generate(
                input_features,
                max_new_tokens=max_tokens,
                language=target_lang,
                task="transcribe",
            )
        except Exception:
            # Legacy fallback with explicit forced_decoder_ids for older transformers
            decoder_ids = self.processor.get_decoder_prompt_ids(
                language=target_lang,
                task="transcribe",
            )
            ids = self.model.generate(
                input_features,
                max_new_tokens=max_tokens,
                forced_decoder_ids=decoder_ids,
            )

        text = self.processor.batch_decode(ids, skip_special_tokens=True)[0]
        return text, target_lang

    def _transcribe_waveform_with_meta(
        self,
        wav: torch.Tensor,
        *,
        language_override: Optional[str] = None,
    ) -> Dict[str, Any]:
        """Process waveform tensor and return text and language metadata."""
        from infer.lang_id import compute_vietnamese_percent, text_looks_vietnamese

        target_mode = (language_override or self.language or "auto").strip().lower()
        inputs = self.processor(wav.numpy(), sampling_rate=WHISPER_SR, return_tensors="pt")
        model_dtype = getattr(self.model, "dtype", None)
        if model_dtype is not None:
            input_features = inputs.input_features.to(device=self.device, dtype=model_dtype)
        else:
            input_features = inputs.input_features.to(self.device)
            if self.device.type == "cuda" and input_features.dtype == torch.float32:
                input_features = input_features.half()

        # Pure forced-language mode (e.g., conversation/practice 'en'): skip language-ID forward pass
        if target_mode not in ("auto", "", "none"):
            raw_text, used_lang = self._generate_with_lang(input_features, forced_lang=target_mode)
            is_vi = used_lang in ("vi", "vie", "vietnamese")
            normalized_text = _normalize_transcript(raw_text, is_vietnamese=is_vi)
            return {
                "text": normalized_text,
                "raw_text": raw_text.strip(),
                "language": used_lang,
                "is_vietnamese": is_vi,
            }

        # Bilingual EN/VI auto mode (used by classroom language analysis)
        detected_lang, lang_probs = self.detect_language_probs(input_features)
        raw_text, used_lang = self._generate_with_lang(input_features, forced_lang=detected_lang)

        vi_prob = float(lang_probs.get("vi", 1.0 if used_lang == "vi" else 0.0))
        vi_stats = compute_vietnamese_percent(vi_prob, raw_text, detected_lang=used_lang)

        is_vi = (
            (used_lang in ("vi", "vie", "vietnamese"))
            or text_looks_vietnamese(raw_text)
            or (vi_stats["vi_percent"] >= 50.0)
        )
        final_lang = "vi" if is_vi else ("en" if used_lang not in ("vi", "vie", "vietnamese") else used_lang)

        normalized_text = _normalize_transcript(raw_text, is_vietnamese=is_vi)
        return {
            "text": normalized_text,
            "raw_text": raw_text.strip(),
            "language": final_lang,
            "is_vietnamese": is_vi,
            **vi_stats,
        }

    @torch.inference_mode()
    def transcribe_waveform(
        self,
        wav: Union[torch.Tensor, Any],
        sample_rate: int = WHISPER_SR,
        *,
        language: Optional[str] = None,
        skip_vad_check: bool = False,
    ) -> Dict[str, Any]:
        """Transcribe an in-memory 1D mono waveform without temporary file I/O."""
        from infer.lang_id import compute_vietnamese_percent

        if not isinstance(wav, torch.Tensor):
            wav = torch.as_tensor(wav, dtype=torch.float32)
        else:
            wav = wav.detach().cpu().float()
        if wav.dim() > 1:
            wav = wav.mean(0)
        if sample_rate != WHISPER_SR and wav.numel() > 0:
            wav = torchaudio.functional.resample(wav, sample_rate, WHISPER_SR)

        if wav.numel() == 0:
            raise ValueError("File audio rỗng")
        if not skip_vad_check:
            peak_amp = float(wav.abs().max())
            rms = float(torch.sqrt(torch.mean(wav ** 2)))
            if peak_amp < 0.015 or rms < 0.003:
                raise ValueError(f"Không phát hiện tiếng người trong audio (VAD: peak={peak_amp:.4f}, rms={rms:.4f})")

        duration_sec = float(wav.shape[0]) / float(WHISPER_SR)
        chunk_samples = int(WHISPER_CHUNK_SEC * WHISPER_SR)
        target_mode = (language or self.language or "auto").strip().lower()

        if wav.shape[0] <= chunk_samples:
            res = self._transcribe_waveform_with_meta(wav, language_override=language)
            res["duration_sec"] = duration_sec
            return res

        parts_text = []
        parts_raw = []
        detected_languages = []
        is_vi_flags = []
        chunk_weights = []
        chunk_vi_probs = []

        for start in range(0, wav.shape[0], chunk_samples):
            chunk = wav[start : start + chunk_samples]
            chunk_res = self._transcribe_waveform_with_meta(chunk, language_override=language)
            if chunk_res["text"] or chunk_res["raw_text"]:
                parts_text.append(chunk_res["text"])
                parts_raw.append(chunk_res["raw_text"])
                detected_languages.append(chunk_res["language"])
                is_vi_flags.append(chunk_res["is_vietnamese"])
                w_sec = max(0.1, float(chunk.shape[0]) / float(WHISPER_SR))
                chunk_weights.append(w_sec)
                if "acoustic_vi_percent" in chunk_res or "language_probs" in chunk_res:
                    c_vi_prob = float((chunk_res.get("language_probs") or {}).get("vi", chunk_res.get("acoustic_vi_percent", 0.0) / 100.0))
                    chunk_vi_probs.append(c_vi_prob)

        combined_text = " ".join(p for p in parts_text if p)
        combined_raw = " ".join(p for p in parts_raw if p)

        if target_mode not in ("auto", "", "none"):
            is_any_vi = any(is_vi_flags)
            return {
                "text": combined_text,
                "raw_text": combined_raw,
                "language": target_mode,
                "is_vietnamese": is_any_vi,
                "duration_sec": duration_sec,
            }

        total_w = sum(chunk_weights) or 1.0
        avg_vi_prob = sum(p * w for p, w in zip(chunk_vi_probs, chunk_weights)) / total_w if chunk_vi_probs else 0.0
        overall_lang = "vi" if (any(is_vi_flags) or avg_vi_prob >= 0.5) else (detected_languages[0] if detected_languages else "en")
        vi_stats = compute_vietnamese_percent(avg_vi_prob, combined_raw or combined_text, detected_lang=overall_lang)
        is_any_vi = any(is_vi_flags) or (vi_stats["vi_percent"] >= 50.0)

        return {
            "text": combined_text,
            "raw_text": combined_raw,
            "language": "vi" if is_any_vi else "en",
            "is_vietnamese": is_any_vi,
            "duration_sec": duration_sec,
            **vi_stats,
        }

    @torch.inference_mode()
    def transcribe_with_meta(
        self,
        audio_path: PathLike,
        *,
        language: Optional[str] = None,
    ) -> Dict[str, Any]:
        """Transcribe audio file and return metadata including detected language."""
        wav, sr = load_audio_file(audio_path)
        wav = wav.mean(0)
        return self.transcribe_waveform(wav, sample_rate=sr, language=language)

    @torch.inference_mode()
    def transcribe(self, audio_path: PathLike) -> str:
        """Transcribe audio file and return normalized text string."""
        meta = self.transcribe_with_meta(audio_path)
        return meta["text"]


class Qwen3Transcriber:
    """ASR transcriber powered by Qwen3-ASR (e.g. Qwen/Qwen3-ASR-1.7B-hf)."""

    def __init__(
        self,
        model_name: str = "Qwen/Qwen3-ASR-1.7B-hf",
        language: str = "English",
        device: Optional[str] = None,
        load_progress: Optional[Any] = None,
        torch_dtype: str = "float16",
        max_new_tokens: int = 128,
        asr_config: Optional[Dict[str, Any]] = None,
        allowed_languages: Optional[Sequence[str]] = None,
    ):
        from transformers import AutoProcessor, AutoModelForMultimodalLM

        self.model_name = model_name or "Qwen/Qwen3-ASR-1.7B-hf"
        self.language = language or "English"
        if str(self.language).lower() in ("en", "english"):
            self.language = "English"
        elif str(self.language).lower() in ("vi", "vietnamese"):
            self.language = "Vietnamese"
        self.max_new_tokens = max_new_tokens

        if device is not None:
            self.device = torch.device(device)
        else:
            self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        if load_progress is not None:
            load_progress.start("qwen3_proc", self.model_name)
        try:
            self.processor = AutoProcessor.from_pretrained(self.model_name, trust_remote_code=True)
        except Exception as e:
            if "qwen3_asr" in str(e) or "model type" in str(e) or isinstance(e, KeyError):
                raise RuntimeError(
                    "Transformers phiên bản hiện tại chưa hỗ trợ kiến trúc 'qwen3_asr'. "
                    "Vui lòng nâng cấp thư viện bằng lệnh: pip install -U 'transformers>=5.0.0'"
                ) from e
            raise

        if load_progress is not None:
            load_progress.finish("qwen3_proc", self.model_name)
            load_progress.start("qwen3_model", self.model_name)

        target_dtype = torch.float16 if str(torch_dtype).lower() in ("float16", "fp16") else torch.float32
        load_kw: Dict[str, Any] = {
            "torch_dtype": target_dtype,
            "trust_remote_code": True,
        }
        if self.device.type == "cuda":
            load_kw["device_map"] = {"": self.device}
        else:
            load_kw["device_map"] = "cpu"

        try:
            self.model = AutoModelForMultimodalLM.from_pretrained(
                self.model_name,
                **load_kw,
            )
        except Exception as e:
            if "qwen3_asr" in str(e) or "model type" in str(e) or isinstance(e, KeyError):
                raise RuntimeError(
                    "Transformers phiên bản hiện tại chưa hỗ trợ kiến trúc 'qwen3_asr'. "
                    "Vui lòng nâng cấp thư viện bằng lệnh: pip install -U 'transformers>=5.0.0'"
                ) from e
            raise

        if load_progress is not None:
            load_progress.finish("qwen3_model", self.model_name)

    @torch.inference_mode()
    def transcribe_waveform(
        self,
        waveform: torch.Tensor,
        sample_rate: int = 16000,
        *,
        language: Optional[str] = None,
    ) -> Dict[str, Any]:
        """Transcribe waveform tensor and return metadata matching evaluation format."""
        if waveform.dim() > 1:
            waveform = waveform.mean(0)

        if sample_rate != 16000:
            import torchaudio.transforms as T
            resampler = T.Resample(orig_freq=sample_rate, new_freq=16000)
            waveform = resampler(waveform)
            sample_rate = 16000

        wav_np = waveform.detach().cpu().numpy()
        duration_sec = float(len(wav_np) / sample_rate)

        target_lang = language or self.language
        if target_lang and str(target_lang).lower() in ("en", "english"):
            target_lang = "English"
        elif target_lang and str(target_lang).lower() in ("vi", "vietnamese"):
            target_lang = "Vietnamese"
        elif target_lang and str(target_lang).lower() in ("auto", "none"):
            target_lang = None

        inputs = self.processor.apply_transcription_request(
            audio=wav_np,
            language=target_lang,
        ).to(self.model.device, self.model.dtype)

        output_ids = self.model.generate(**inputs, max_new_tokens=self.max_new_tokens)
        generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]

        raw_text = self.processor.decode(generated_ids, return_format="transcription_only")[0].strip()
        normalized_text = _normalize_transcript(raw_text, is_vietnamese=False)

        return {
            "text": normalized_text,
            "raw_text": raw_text,
            "language": "en" if target_lang == "English" else (str(target_lang).lower() if target_lang else "en"),
            "is_vietnamese": False,
            "duration_sec": duration_sec,
            "vi_percent": 0.0,
            "char_count": len(raw_text),
            "word_count": len(raw_text.split()),
        }

    @torch.inference_mode()
    def transcribe_with_meta(
        self,
        audio_path: PathLike,
        *,
        language: Optional[str] = None,
    ) -> Dict[str, Any]:
        wav, sr = load_audio_file(audio_path)
        wav = wav.mean(0)
        return self.transcribe_waveform(wav, sample_rate=sr, language=language)

    @torch.inference_mode()
    def transcribe(self, audio_path: PathLike) -> str:
        meta = self.transcribe_with_meta(audio_path)
        return meta["text"]


def reset_transcriber() -> None:
    """Clear cached transcriber instance and free GPU memory."""
    global _transcriber, _transcriber_error
    with _lock:
        _transcriber = None
        _transcriber_error = None
        import gc
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def get_transcriber(
    load_progress: Optional[Any] = None,
    device: Optional[str] = None,
    asr_config: Optional[Dict[str, Any]] = None,
) -> Union[WhisperTranscriber, Qwen3Transcriber]:
    global _transcriber, _transcriber_error
    if _transcriber is not None:
        return _transcriber
    if _transcriber_error:
        raise RuntimeError(_transcriber_error)
    with _lock:
        if _transcriber is not None:
            return _transcriber
        if _transcriber_error:
            raise RuntimeError(_transcriber_error)
        try:
            asr = asr_config if asr_config is not None else _load_asr_config()
            target_device = device or asr.get("device")
            engine = str(asr.get("engine", "")).strip().lower()
            model_name = str(asr.get("model_name", "")).strip()

            if engine == "qwen3" or "qwen3" in model_name.lower():
                _transcriber = Qwen3Transcriber(
                    model_name=model_name or "Qwen/Qwen3-ASR-1.7B-hf",
                    language=asr.get("language", "English"),
                    device=target_device,
                    load_progress=load_progress,
                    torch_dtype=asr.get("torch_dtype", "float16"),
                    max_new_tokens=int(asr.get("max_new_tokens", 128)),
                    asr_config=asr,
                    allowed_languages=asr.get("allowed_languages"),
                )
            else:
                _transcriber = WhisperTranscriber(
                    model_name=model_name or WHISPER_DEFAULT_MODEL,
                    language=asr.get("language", "auto"),
                    device=target_device,
                    load_progress=load_progress,
                    torch_dtype=asr.get("torch_dtype", "float16"),
                    max_new_tokens=int(asr.get("max_new_tokens", 440)),
                    asr_config=asr,
                    allowed_languages=asr.get("allowed_languages"),
                )
        except Exception as exc:
            _transcriber_error = str(exc)
            if load_progress is not None:
                load_progress.fail_running(str(exc))
            raise RuntimeError(_transcriber_error) from exc
        return _transcriber


def transcribe_audio(
    audio_path: PathLike,
    *,
    asr_config: Optional[Dict[str, Any]] = None,
) -> str:
    """Convenience helper returning normalized transcript string."""
    return get_transcriber(asr_config=asr_config).transcribe(audio_path)


def transcribe_audio_with_meta(
    audio_path: PathLike,
    *,
    asr_config: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """Convenience helper returning transcript with language and detection metadata."""
    return get_transcriber(asr_config=asr_config).transcribe_with_meta(audio_path)


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/classroom_lang', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/classroom_lang/__init__.py
"""Standalone Classroom Language Ratio Analysis (Teacher vs Students, EN vs VI, Speech-Only)."""

from classroom_lang.analyzer import (
    ClassroomAnalysisConfig,
    ClassroomLanguageAnalyzer,
    analyze_classroom_audio,
)
from classroom_lang.vad_diarize import (
    ClassroomDiarizationResult,
    ClassroomTurn,
    SpeechRegion,
    detect_speech_regions,
    diarize_teacher_vs_students,
    normalize_embedding,
)

__all__ = [
    "ClassroomAnalysisConfig",
    "ClassroomLanguageAnalyzer",
    "ClassroomDiarizationResult",
    "ClassroomTurn",
    "SpeechRegion",
    "analyze_classroom_audio",
    "detect_speech_regions",
    "diarize_teacher_vs_students",
    "normalize_embedding",
]



In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/classroom_lang', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/classroom_lang/vad_diarize.py
"""Speech-only VAD and Teacher-only speaker verification for classroom recordings."""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, List, Optional, Sequence, Tuple

import numpy as np

SAMPLE_RATE = 16000


@dataclass
class SpeechRegion:
    """Contiguous voiced interval detected by VAD (silence strictly excluded)."""

    start_sec: float
    end_sec: float
    start_sample: int
    end_sample: int

    @property
    def duration_sec(self) -> float:
        return max(0.0, self.end_sec - self.start_sec)


@dataclass
class ClassroomTurn:
    """Single voiced speaker turn attributed to either 'teacher' or 'student'."""

    index: int
    role: str
    start_sec: float
    end_sec: float
    duration_sec: float
    teacher_similarity: float
    start_sample: int
    end_sample: int


@dataclass
class ClassroomDiarizationResult:
    """Full-session voiced turns and speech/silence duration breakdown."""

    turns: List[ClassroomTurn]
    speech_regions: List[SpeechRegion]
    total_audio_sec: float
    total_speech_sec: float
    silence_sec: float
    teacher_speech_sec: float
    student_speech_sec: float


def normalize_embedding(embedding: Any) -> np.ndarray:
    """Convert 1D or 2D embedding list/array into a unit-norm 1D float32 vector."""
    arr = np.asarray(embedding, dtype=np.float32)
    if arr.ndim == 2:
        if arr.shape[0] == 0:
            raise ValueError("Teacher voice embedding rỗng.")
        arr = np.mean(arr, axis=0)
    arr = np.squeeze(arr).astype(np.float32)
    if arr.ndim != 1 or arr.size == 0:
        raise ValueError("Teacher voice embedding không hợp lệ.")
    norm = float(np.linalg.norm(arr))
    if norm <= 1e-8:
        raise ValueError("Teacher voice embedding có norm bằng 0.")
    return arr / norm


def detect_speech_regions(
    wav: np.ndarray,
    sample_rate: int = SAMPLE_RATE,
    *,
    vad_threshold_db: float = -40.0,
    frame_ms: float = 20.0,
    hop_ms: float = 10.0,
    min_speech_sec: float = 0.25,
    min_silence_sec: float = 0.30,
    padding_sec: float = 0.12,
) -> List[SpeechRegion]:
    """
    Detect voiced speech regions while excluding silence and stationary room noise.
    Uses adaptive ambient noise-floor estimation bounded by vad_threshold_db.
    """
    mono = np.asarray(wav, dtype=np.float32)
    if mono.ndim > 1:
        mono = np.mean(mono, axis=0)
    n_samples = int(mono.shape[0])
    if n_samples == 0:
        return []

    peak = float(np.max(np.abs(mono)))
    if peak < 1e-5:
        return []

    frame_len = max(1, int(round(sample_rate * frame_ms / 1000.0)))
    hop_len = max(1, int(round(sample_rate * hop_ms / 1000.0)))
    if n_samples < frame_len:
        rms_val = float(np.sqrt(np.mean(mono ** 2) + 1e-12))
        db_val = 20.0 * np.log10(rms_val + 1e-12)
        if db_val >= vad_threshold_db:
            return [
                SpeechRegion(
                    start_sec=0.0,
                    end_sec=round(n_samples / float(sample_rate), 3),
                    start_sample=0,
                    end_sample=n_samples,
                )
            ]
        return []

    # Strided framing without copying large audio buffers
    n_frames = 1 + (n_samples - frame_len) // hop_len
    stride = mono.strides[0]
    frames = np.lib.stride_tricks.as_strided(
        mono,
        shape=(n_frames, frame_len),
        strides=(hop_len * stride, stride),
    )
    frame_rms = np.sqrt(np.mean(frames ** 2, axis=1) + 1e-12)
    frame_db = 20.0 * np.log10(frame_rms + 1e-12)

    speech_peak_db = float(np.percentile(frame_db, 98))
    if speech_peak_db < vad_threshold_db:
        return []

    noise_floor_db = float(np.percentile(frame_db, 15))
    spread_db = speech_peak_db - noise_floor_db
    if spread_db >= 10.0:
        adaptive_db = noise_floor_db + max(6.0, 0.25 * spread_db)
        effective_thresh_db = max(float(vad_threshold_db), min(adaptive_db, speech_peak_db - 18.0))
    else:
        effective_thresh_db = float(vad_threshold_db)

    active_frames = frame_db >= effective_thresh_db
    if not np.any(active_frames):
        return []

    # Convert active frames to raw sample intervals
    raw_spans: List[Tuple[int, int]] = []
    in_seg = False
    seg_start = 0
    for idx, is_act in enumerate(active_frames):
        if is_act and not in_seg:
            in_seg = True
            seg_start = idx * hop_len
        elif not is_act and in_seg:
            in_seg = False
            seg_end = min(n_samples, (idx - 1) * hop_len + frame_len)
            raw_spans.append((seg_start, seg_end))
    if in_seg:
        raw_spans.append((seg_start, n_samples))

    # Apply padding and merge micro-pauses shorter than min_silence_sec
    pad_samples = int(round(padding_sec * sample_rate))
    min_silence_samples = int(round(min_silence_sec * sample_rate))
    min_speech_samples = int(round(min_speech_sec * sample_rate))

    padded: List[List[int]] = []
    for s, e in raw_spans:
        ps = max(0, s - pad_samples)
        pe = min(n_samples, e + pad_samples)
        if padded and ps - padded[-1][1] <= min_silence_samples:
            padded[-1][1] = max(padded[-1][1], pe)
        else:
            padded.append([ps, pe])

    regions: List[SpeechRegion] = []
    for s, e in padded:
        if (e - s) < min_speech_samples:
            continue
        regions.append(
            SpeechRegion(
                start_sec=round(s / float(sample_rate), 3),
                end_sec=round(e / float(sample_rate), 3),
                start_sample=int(s),
                end_sample=int(e),
            )
        )
    return regions


def _find_best_split_sample(
    mono: np.ndarray,
    start_sample: int,
    end_sample: int,
    sample_rate: int,
) -> int:
    """Find a low-energy boundary near the upper end of a long speech span."""
    search_radius = int(1.5 * sample_rate)
    center = end_sample
    lo = max(start_sample + int(1.0 * sample_rate), center - search_radius)
    hi = min(len(mono), center + search_radius)
    if hi <= lo + int(0.1 * sample_rate):
        return center

    win = max(1, int(0.05 * sample_rate))
    best_pos = center
    best_energy = float("inf")
    for pos in range(lo, hi - win, win):
        energy = float(np.mean(mono[pos : pos + win] ** 2))
        if energy < best_energy:
            best_energy = energy
            best_pos = pos + win // 2
    return best_pos


def _smooth_role_labels(labels: np.ndarray, min_run_samples: int) -> np.ndarray:
    """Remove brief speaker label spikes shorter than min_run_samples inside a voiced block."""
    if len(labels) == 0 or min_run_samples <= 1:
        return labels
    out = labels.copy()
    n = len(out)
    i = 0
    while i < n:
        j = i + 1
        while j < n and out[j] == out[i]:
            j += 1
        run_len = j - i
        if run_len < min_run_samples:
            if i > 0:
                out[i:j] = out[i - 1]
            elif j < n:
                out[i:j] = out[j]
        i = j
    return out


def diarize_teacher_vs_students(
    wav: np.ndarray,
    teacher_embedding: Any,
    embedder: Any,
    sample_rate: int = SAMPLE_RATE,
    *,
    teacher_threshold: float = 0.40,
    vad_threshold_db: float = -40.0,
    min_speech_sec: float = 0.25,
    min_silence_sec: float = 0.30,
    window_sec: float = 1.5,
    step_sec: float = 0.5,
    max_turn_sec: float = 20.0,
    min_turn_sec: float = 0.35,
) -> ClassroomDiarizationResult:
    """
    Diarize ONLY the Teacher on voiced regions; all remaining voiced regions become Student.
    Silence intervals are strictly excluded from turns and duration totals.
    """
    mono = np.asarray(wav, dtype=np.float32)
    if mono.ndim > 1:
        mono = np.mean(mono, axis=0)
    n_samples = int(mono.shape[0])
    total_audio_sec = round(n_samples / float(sample_rate), 3)

    t_emb = normalize_embedding(teacher_embedding)
    regions = detect_speech_regions(
        mono,
        sample_rate=sample_rate,
        vad_threshold_db=vad_threshold_db,
        min_speech_sec=min_speech_sec,
        min_silence_sec=min_silence_sec,
    )

    if not regions:
        return ClassroomDiarizationResult(
            turns=[],
            speech_regions=[],
            total_audio_sec=total_audio_sec,
            total_speech_sec=0.0,
            silence_sec=total_audio_sec,
            teacher_speech_sec=0.0,
            student_speech_sec=0.0,
        )

    win_samples = max(1, int(round(window_sec * sample_rate)))
    step_samples = max(1, int(round(step_sec * sample_rate)))
    min_embed_samples = max(1, int(round(0.22 * sample_rate)))

    window_spans: List[Tuple[int, int]] = []
    window_audios: List[np.ndarray] = []

    for reg in regions:
        rs, re_ = reg.start_sample, reg.end_sample
        reg_len = re_ - rs
        if reg_len <= win_samples:
            if reg_len >= min_embed_samples:
                window_spans.append((rs, re_))
                window_audios.append(mono[rs:re_])
            continue

        pos = rs
        while pos + win_samples <= re_:
            window_spans.append((pos, pos + win_samples))
            window_audios.append(mono[pos : pos + win_samples])
            pos += step_samples
        # Cover tail of speech region so end of turn isn't lost
        if pos < re_ and (re_ - pos) >= min_embed_samples:
            tail_start = max(rs, re_ - win_samples)
            if not window_spans or window_spans[-1] != (tail_start, re_):
                window_spans.append((tail_start, re_))
                window_audios.append(mono[tail_start:re_])

    embeddings: Sequence[Optional[np.ndarray]] = []
    if window_audios:
        embeddings = embedder.embed_batch(window_audios, sample_rate=sample_rate)

    teacher_votes = np.zeros(n_samples, dtype=np.float32)
    student_votes = np.zeros(n_samples, dtype=np.float32)
    sim_sum = np.zeros(n_samples, dtype=np.float64)
    sim_cnt = np.zeros(n_samples, dtype=np.int32)

    for (ws, we), emb in zip(window_spans, embeddings):
        if emb is None:
            continue
        vec = np.asarray(emb, dtype=np.float32).reshape(-1)
        norm = float(np.linalg.norm(vec))
        if norm > 1e-8:
            vec = vec / norm
        sim = float(np.dot(vec, t_emb))
        if sim >= teacher_threshold:
            weight = max(0.05, sim - teacher_threshold + 0.05)
            teacher_votes[ws:we] += weight
        else:
            weight = max(0.05, teacher_threshold - sim + 0.05)
            student_votes[ws:we] += weight
        sim_sum[ws:we] += sim
        sim_cnt[ws:we] += 1

    min_run_samples = int(round(min_turn_sec * sample_rate))
    max_turn_samples = int(round(max_turn_sec * sample_rate)) if max_turn_sec and max_turn_sec > 0 else 0

    raw_turns: List[Tuple[str, int, int, float]] = []
    for reg in regions:
        rs, re_ = reg.start_sample, reg.end_sample
        reg_len = re_ - rs
        if reg_len <= 0:
            continue

        # 1 = teacher, 0 = student (only evaluated inside voiced region!)
        t_v = teacher_votes[rs:re_]
        s_v = student_votes[rs:re_]
        sample_labels = ((t_v >= s_v) & (t_v > 0.0)).astype(np.int8)
        sample_labels = _smooth_role_labels(sample_labels, min_run_samples=min_run_samples)

        idx = 0
        while idx < reg_len:
            j = idx + 1
            while j < reg_len and sample_labels[j] == sample_labels[idx]:
                j += 1
            role = "teacher" if sample_labels[idx] == 1 else "student"
            abs_s = rs + idx
            abs_e = rs + j

            # Sub-split very long uninterrupted turns so Whisper stays well within 30s chunks
            cur_s = abs_s
            while max_turn_samples > 0 and (abs_e - cur_s) > max_turn_samples:
                target_e = cur_s + max_turn_samples
                split_e = _find_best_split_sample(mono, cur_s, target_e, sample_rate)
                split_e = max(cur_s + int(1.0 * sample_rate), min(abs_e, split_e))
                cnt_slice = sim_cnt[cur_s:split_e]
                valid = cnt_slice > 0
                avg_sim = float(np.mean(sim_sum[cur_s:split_e][valid] / cnt_slice[valid])) if np.any(valid) else 0.0
                raw_turns.append((role, cur_s, split_e, avg_sim))
                cur_s = split_e

            if abs_e > cur_s:
                cnt_slice = sim_cnt[cur_s:abs_e]
                valid = cnt_slice > 0
                avg_sim = float(np.mean(sim_sum[cur_s:abs_e][valid] / cnt_slice[valid])) if np.any(valid) else 0.0
                raw_turns.append((role, cur_s, abs_e, avg_sim))
            idx = j

    turns: List[ClassroomTurn] = []
    teacher_speech_sec = 0.0
    student_speech_sec = 0.0
    for role, s, e, avg_sim in raw_turns:
        dur = (e - s) / float(sample_rate)
        if dur <= 0.0:
            continue
        if role == "teacher":
            teacher_speech_sec += dur
        else:
            student_speech_sec += dur
        turns.append(
            ClassroomTurn(
                index=len(turns),
                role=role,
                start_sec=round(s / float(sample_rate), 3),
                end_sec=round(e / float(sample_rate), 3),
                duration_sec=round(dur, 3),
                teacher_similarity=round(avg_sim, 4),
                start_sample=s,
                end_sample=e,
            )
        )

    total_speech_sec = round(teacher_speech_sec + student_speech_sec, 3)
    silence_sec = round(max(0.0, total_audio_sec - total_speech_sec), 3)

    return ClassroomDiarizationResult(
        turns=turns,
        speech_regions=regions,
        total_audio_sec=total_audio_sec,
        total_speech_sec=total_speech_sec,
        silence_sec=silence_sec,
        teacher_speech_sec=round(teacher_speech_sec, 3),
        student_speech_sec=round(student_speech_sec, 3),
    )



In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/classroom_lang', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/classroom_lang/analyzer.py
"""Classroom EN/VI language ratio analyzer (Teacher vs Students, word-count based, speech-only)."""

from __future__ import annotations

import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Union

import numpy as np
import torch
import torchaudio

from classroom_lang.vad_diarize import (
    SAMPLE_RATE,
    diarize_teacher_vs_students,
    normalize_embedding,
)
from data.audio_preprocess import _highpass, load_audio_file
from infer.lang_id import classify_transcript_words
from infer.transcribe import clean_whisper_hallucinations, get_transcriber
from paths import SPEAKER_DIARIZE_DIR

PathLike = Union[str, Path]


@dataclass
class ClassroomAnalysisConfig:
    """Configuration for full-class EN/VI ratio analysis."""

    teacher_threshold: float = 0.40
    vad_threshold_db: float = -40.0
    min_speech_sec: float = 0.25
    min_silence_sec: float = 0.30
    window_sec: float = 1.5
    step_sec: float = 0.5
    max_turn_sec: float = 20.0
    min_turn_sec: float = 0.35
    highpass_hz: float = 80.0


def _ensure_speaker_diarize_path() -> None:
    if SPEAKER_DIARIZE_DIR.is_dir() and str(SPEAKER_DIARIZE_DIR) not in sys.path:
        sys.path.insert(0, str(SPEAKER_DIARIZE_DIR))


def _load_mono_16k(audio_path: PathLike, highpass_hz: float = 80.0) -> np.ndarray:
    wav, sr = load_audio_file(audio_path)
    mono = wav.mean(0) if wav.dim() > 1 else wav
    if sr != SAMPLE_RATE and mono.numel() > 0:
        mono = torchaudio.functional.resample(mono, sr, SAMPLE_RATE)
    if highpass_hz > 0 and mono.numel() > 0:
        mono = _highpass(mono, SAMPLE_RATE, highpass_hz)
    return mono.detach().cpu().numpy().astype(np.float32)


def _aggregate_role_word_stats(
    segments: List[Dict[str, Any]],
    speech_sec: float,
) -> Dict[str, Any]:
    vi_words = sum(int(s.get("vi_words", 0)) for s in segments)
    en_words = sum(int(s.get("en_words", 0)) for s in segments)
    total_words = vi_words + en_words
    if total_words > 0:
        vi_pct = round(100.0 * vi_words / total_words, 1)
        en_pct = round(max(0.0, 100.0 - vi_pct), 1)
    else:
        vi_pct = 0.0
        en_pct = 0.0
    return {
        "vi_words": vi_words,
        "en_words": en_words,
        "total_words": total_words,
        "vi_percent": vi_pct,
        "en_percent": en_pct,
        "speech_sec": round(float(speech_sec), 2),
        "turn_count": len(segments),
    }


class ClassroomLanguageAnalyzer:
    """
    Analyze full classroom audio to compute Teacher vs Student % Vietnamese and % English.
    - Excludes silence intervals strictly via VAD.
    - Diarizes ONLY the Teacher using enrolled voice_embeddings; all other speech is Student.
    - Transcribes voiced turns with Whisper Large-v3 (EN/VI) and counts VI vs EN words.
    - Performs NO pronunciation scoring.
    """

    def __init__(
        self,
        *,
        embedder: Optional[Any] = None,
        transcriber: Optional[Any] = None,
        device: Optional[str] = None,
        config: Optional[ClassroomAnalysisConfig] = None,
    ) -> None:
        self.device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        self._embedder = embedder
        self._transcriber = transcriber
        self.config = config or ClassroomAnalysisConfig()

    def _get_embedder(self) -> Any:
        if self._embedder is not None:
            return self._embedder
        _ensure_speaker_diarize_path()
        from speaker_diarize.embedding import ERes2NetEmbedder

        self._embedder = ERes2NetEmbedder(device=self.device)
        return self._embedder

    def _get_transcriber(self) -> Any:
        if self._transcriber is not None:
            return self._transcriber
        dtype = "float16" if str(self.device).startswith("cuda") else "float32"
        self._transcriber = get_transcriber(
            device=self.device,
            asr_config={
                "model_name": "openai/whisper-large-v3",
                "language": "auto",
                "allowed_languages": ["en", "vi"],
                "device": self.device,
                "torch_dtype": dtype,
                "max_new_tokens": 440,
            },
        )
        return self._transcriber

    def extract_teacher_embedding(self, reference_audio: PathLike) -> np.ndarray:
        """Helper for standalone CLI usage when given a reference WAV instead of stored JSON."""
        mono = _load_mono_16k(reference_audio, highpass_hz=self.config.highpass_hz)
        embedder = self._get_embedder()
        vec = embedder.embed_direct(mono, sample_rate=SAMPLE_RATE)
        return normalize_embedding(vec)

    def analyze(
        self,
        audio_path: PathLike,
        teacher_embedding: Any,
        *,
        teacher_threshold: Optional[float] = None,
        vad_threshold_db: Optional[float] = None,
        progress_callback: Optional[Callable[[str], None]] = None,
    ) -> Dict[str, Any]:
        """Run full classroom EN/VI language ratio analysis on an audio recording."""
        cfg = self.config
        t_thresh = float(teacher_threshold if teacher_threshold is not None else cfg.teacher_threshold)
        v_thresh = float(vad_threshold_db if vad_threshold_db is not None else cfg.vad_threshold_db)

        if progress_callback:
            progress_callback("Đang tải audio buổi học & lọc bỏ vùng im lặng (VAD)...")

        mono = _load_mono_16k(audio_path, highpass_hz=cfg.highpass_hz)
        t_emb = normalize_embedding(teacher_embedding)
        embedder = self._get_embedder()

        if progress_callback:
            progress_callback("Đang nhận diện giọng Giáo viên vs Học viên trên các vùng nói...")

        diar = diarize_teacher_vs_students(
            mono,
            t_emb,
            embedder,
            sample_rate=SAMPLE_RATE,
            teacher_threshold=t_thresh,
            vad_threshold_db=v_thresh,
            min_speech_sec=cfg.min_speech_sec,
            min_silence_sec=cfg.min_silence_sec,
            window_sec=cfg.window_sec,
            step_sec=cfg.step_sec,
            max_turn_sec=cfg.max_turn_sec,
            min_turn_sec=cfg.min_turn_sec,
        )

        segments: List[Dict[str, Any]] = []
        if diar.turns:
            transcriber = self._get_transcriber()
            total_turns = len(diar.turns)
            for idx, turn in enumerate(diar.turns):
                if progress_callback and (idx == 0 or (idx + 1) % 5 == 0 or idx + 1 == total_turns):
                    progress_callback(f"Đang phiên âm vùng nói {idx + 1}/{total_turns} (Whisper Large-v3 EN/VI)...")

                seg_wav = mono[turn.start_sample : turn.end_sample]
                if seg_wav.size == 0:
                    continue

                try:
                    meta = transcriber.transcribe_waveform(
                        seg_wav,
                        sample_rate=SAMPLE_RATE,
                        language="auto",
                        skip_vad_check=True,
                    )
                except ValueError:
                    continue

                raw_text = clean_whisper_hallucinations(meta.get("raw_text") or meta.get("text") or "")
                if not raw_text:
                    continue

                w_lang = str(meta.get("language") or "en").lower()
                word_stats = classify_transcript_words(raw_text, whisper_lang=w_lang)
                if word_stats["word_count"] <= 0:
                    continue

                segments.append(
                    {
                        "index": len(segments),
                        "role": turn.role,
                        "start_sec": turn.start_sec,
                        "end_sec": turn.end_sec,
                        "duration_sec": turn.duration_sec,
                        "teacher_similarity": turn.teacher_similarity,
                        "whisper_lang": "vi" if w_lang in ("vi", "vie", "vietnamese") else "en",
                        "transcript": raw_text,
                        "vi_words": word_stats["vi_word_count"],
                        "en_words": word_stats["en_word_count"],
                        "total_words": word_stats["word_count"],
                        "vi_percent": word_stats["vi_percent"],
                        "en_percent": word_stats["en_percent"],
                    }
                )

        teacher_segs = [s for s in segments if s["role"] == "teacher"]
        student_segs = [s for s in segments if s["role"] == "student"]

        teacher_stats = _aggregate_role_word_stats(teacher_segs, diar.teacher_speech_sec)
        student_stats = _aggregate_role_word_stats(student_segs, diar.student_speech_sec)
        overall_words = _aggregate_role_word_stats(segments, diar.total_speech_sec)

        speech_ratio = (
            round(100.0 * diar.total_speech_sec / diar.total_audio_sec, 1)
            if diar.total_audio_sec > 0
            else 0.0
        )

        return {
            "mode": "classroom_language_analysis",
            "source_audio": str(audio_path),
            "settings": {
                "teacher_threshold": round(t_thresh, 3),
                "vad_threshold_db": round(v_thresh, 1),
                "allowed_languages": ["en", "vi"],
                "counting_method": "transcript_word_count",
            },
            "duration": {
                "total_audio_sec": round(diar.total_audio_sec, 2),
                "total_speech_sec": round(diar.total_speech_sec, 2),
                "silence_sec": round(diar.silence_sec, 2),
                "speech_ratio_percent": speech_ratio,
                "teacher_speech_sec": round(diar.teacher_speech_sec, 2),
                "student_speech_sec": round(diar.student_speech_sec, 2),
            },
            "teacher": teacher_stats,
            "student": student_stats,
            "overall": {
                "vi_words": overall_words["vi_words"],
                "en_words": overall_words["en_words"],
                "total_words": overall_words["total_words"],
                "vi_percent": overall_words["vi_percent"],
                "en_percent": overall_words["en_percent"],
                "total_speech_sec": round(diar.total_speech_sec, 2),
                "silence_sec": round(diar.silence_sec, 2),
                "total_audio_sec": round(diar.total_audio_sec, 2),
            },
            "segments": segments,
        }


def analyze_classroom_audio(
    audio_path: PathLike,
    teacher_embedding: Any,
    *,
    teacher_threshold: float = 0.40,
    vad_threshold_db: float = -40.0,
    embedder: Optional[Any] = None,
    transcriber: Optional[Any] = None,
    device: Optional[str] = None,
    progress_callback: Optional[Callable[[str], None]] = None,
) -> Dict[str, Any]:
    """Convenience function to run full-classroom EN/VI ratio analysis."""
    analyzer = ClassroomLanguageAnalyzer(
        embedder=embedder,
        transcriber=transcriber,
        device=device,
        config=ClassroomAnalysisConfig(
            teacher_threshold=teacher_threshold,
            vad_threshold_db=vad_threshold_db,
        ),
    )
    return analyzer.analyze(
        audio_path,
        teacher_embedding,
        teacher_threshold=teacher_threshold,
        vad_threshold_db=vad_threshold_db,
        progress_callback=progress_callback,
    )



In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/classroom_lang', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/classroom_lang/cli.py
"""Standalone CLI for Classroom EN/VI Language Ratio Analysis."""

from __future__ import annotations

import argparse
import json
from pathlib import Path

from classroom_lang.analyzer import ClassroomAnalysisConfig, ClassroomLanguageAnalyzer


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Analyze Teacher vs Student % Vietnamese / % English in a full classroom recording."
    )
    parser.add_argument("--audio", required=True, help="Path to full classroom audio file.")
    group = parser.add_mutually_exclusive_group(required=True)
    group.add_argument(
        "--teacher-embedding",
        help="Path to JSON file containing teacher voice_embeddings array.",
    )
    group.add_argument(
        "--teacher-audio",
        help="Path to clean reference WAV/MP3 of the teacher voice.",
    )
    parser.add_argument("--threshold", type=float, default=0.40, help="Teacher cosine similarity threshold (default: 0.40).")
    parser.add_argument("--vad-db", type=float, default=-40.0, help="VAD floor threshold in dBFS (default: -40.0).")
    parser.add_argument("--device", default=None, help="Device override (e.g. cuda:0 or cpu).")
    parser.add_argument("--out", default=None, help="Optional output JSON report path.")
    args = parser.parse_args()

    analyzer = ClassroomLanguageAnalyzer(
        device=args.device,
        config=ClassroomAnalysisConfig(
            teacher_threshold=args.threshold,
            vad_threshold_db=args.vad_db,
        ),
    )

    if args.teacher_embedding:
        raw = json.loads(Path(args.teacher_embedding).read_text(encoding="utf-8"))
        teacher_emb = raw.get("voice_embeddings", raw) if isinstance(raw, dict) else raw
    else:
        teacher_emb = analyzer.extract_teacher_embedding(args.teacher_audio)

    report = analyzer.analyze(
        args.audio,
        teacher_emb,
        teacher_threshold=args.threshold,
        vad_threshold_db=args.vad_db,
        progress_callback=lambda msg: print(f"[Classroom] {msg}", flush=True),
    )

    t = report["teacher"]
    s = report["student"]
    d = report["duration"]
    print("\n" + "=" * 72)
    print(f"Tổng thời lượng: {d['total_audio_sec']}s | Vùng nói: {d['total_speech_sec']}s | Im lặng loại bỏ: {d['silence_sec']}s")
    print(f"Giáo viên (Teacher): VI {t['vi_percent']}% ({t['vi_words']} từ) | EN {t['en_percent']}% ({t['en_words']} từ) | Nói {t['speech_sec']}s")
    print(f"Học viên  (Student): VI {s['vi_percent']}% ({s['vi_words']} từ) | EN {s['en_percent']}% ({s['en_words']} từ) | Nói {s['speech_sec']}s")
    print("=" * 72 + "\n")

    if args.out:
        out_path = Path(args.out)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        out_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"Saved report to: {out_path}")


if __name__ == "__main__":
    main()



In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize/__init__.py
"""2-speaker audio splitting with ERes2Net-Large."""

from speaker_diarize.denoise import denoise_with_deepfilternet, level_audio_to_target
from speaker_diarize.embedding import ERes2NetEmbedder
from speaker_diarize.pipeline import DiarizationSegment, SplitResult, TwoSpeakerSplitter
from speaker_diarize.segmentation import RmsVad

__all__ = [
    "DiarizationSegment",
    "ERes2NetEmbedder",
    "RmsVad",
    "denoise_with_deepfilternet",
    "level_audio_to_target",
    "SplitResult",
    "TwoSpeakerSplitter",
]


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize/audio_io.py
"""Load and save audio files."""

from __future__ import annotations

from pathlib import Path

import numpy as np

SAMPLE_RATE = 16000


def load_audio(path: str | Path) -> tuple[np.ndarray, int]:
    """Load mono float32 audio, resampled to 16 kHz if needed."""
    import soundfile as sf
    import torch
    import torchaudio

    try:
        audio, sr = sf.read(str(path), dtype="float32", always_2d=True)
        mono = audio.mean(axis=1)
        if sr != SAMPLE_RATE:
            wav = torchaudio.functional.resample(
                torch.from_numpy(mono).unsqueeze(0), sr, SAMPLE_RATE
            )
            mono = wav.squeeze(0).numpy()
        return mono.astype(np.float32), SAMPLE_RATE
    except Exception:
        import subprocess
        cmd = [
            "ffmpeg", "-y", "-i", str(path),
            "-f", "f32le", "-ac", "1", "-ar", str(SAMPLE_RATE), "-"
        ]
        p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        if p.returncode != 0:
            raise RuntimeError(f"Failed to load audio {path}: {p.stderr.decode('utf-8', errors='ignore')}")
        mono = np.frombuffer(p.stdout, dtype=np.float32).copy()
        return mono, SAMPLE_RATE


def save_audio(path: str | Path, audio: np.ndarray, sample_rate: int = SAMPLE_RATE) -> None:
    import soundfile as sf

    Path(path).parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), audio, sample_rate)


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize/clustering.py
"""Online 2-speaker clustering with EMA centroid updates."""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np

DEFAULT_THRESHOLD = 0.25  # SpeechBrain cosine-distance verification default


@dataclass
class ClusterAssignment:
    speaker: str
    confidence: float
    calibrating: bool


class TwoSpeakerClusterer:
    """Assign embeddings to Speaker A / B with online centroid updates."""

    def __init__(
        self,
        num_speakers: int = 2,
        threshold: float = DEFAULT_THRESHOLD,
        ema_alpha: float = 0.15,
        min_separation: float = 0.15,
    ) -> None:
        if num_speakers != 2:
            raise ValueError("Only 2-speaker mode is supported")
        self.threshold = threshold
        self.ema_alpha = ema_alpha
        self.min_separation = min_separation
        self._centroids: list[np.ndarray | None] = [None, None]
        self._labels = ["Speaker A", "Speaker B"]
        self._pending: list[np.ndarray] = []
        self._calibrated = False

    @property
    def calibrating(self) -> bool:
        return not self._calibrated

    def assign(self, embedding: np.ndarray) -> ClusterAssignment:
        emb = self._normalize(embedding)

        if not self._calibrated:
            return self._calibrate(emb)

        sim_a = float(np.dot(emb, self._centroids[0]))
        sim_b = float(np.dot(emb, self._centroids[1]))
        dist_a = 1.0 - sim_a
        dist_b = 1.0 - sim_b

        if dist_a <= dist_b:
            idx, confidence = 0, 1.0 - dist_a
        else:
            idx, confidence = 1, 1.0 - dist_b

        self._update_centroid(idx, emb)
        return ClusterAssignment(
            speaker=self._labels[idx],
            confidence=float(np.clip(confidence, 0.0, 1.0)),
            calibrating=False,
        )

    def _calibrate(self, emb: np.ndarray) -> ClusterAssignment:
        self._pending.append(emb)

        if self._centroids[0] is None:
            self._centroids[0] = emb.copy()
            return ClusterAssignment(speaker="calibrating", confidence=0.0, calibrating=True)

        dist_to_a = 1.0 - float(np.dot(emb, self._centroids[0]))
        if dist_to_a > self.threshold and self._centroids[1] is None:
            self._centroids[1] = emb.copy()
            if self._centroid_distance() >= self.min_separation:
                self._calibrated = True
            return ClusterAssignment(speaker="calibrating", confidence=0.0, calibrating=True)

        idx = 0 if dist_to_a <= self.threshold else 1
        if self._centroids[idx] is None:
            self._centroids[idx] = emb.copy()
        else:
            self._update_centroid(idx, emb)

        if self._centroids[0] is not None and self._centroids[1] is not None:
            if self._centroid_distance() >= self.min_separation:
                self._calibrated = True
                sim_a = float(np.dot(emb, self._centroids[0]))
                sim_b = float(np.dot(emb, self._centroids[1]))
                if sim_a >= sim_b:
                    return ClusterAssignment(speaker=self._labels[0], confidence=sim_a, calibrating=False)
                return ClusterAssignment(speaker=self._labels[1], confidence=sim_b, calibrating=False)

        return ClusterAssignment(speaker="calibrating", confidence=0.0, calibrating=True)

    def _centroid_distance(self) -> float:
        if self._centroids[0] is None or self._centroids[1] is None:
            return 0.0
        return 1.0 - float(np.dot(self._centroids[0], self._centroids[1]))

    def _update_centroid(self, idx: int, emb: np.ndarray) -> None:
        current = self._centroids[idx]
        if current is None:
            self._centroids[idx] = emb.copy()
            return
        updated = (1.0 - self.ema_alpha) * current + self.ema_alpha * emb
        self._centroids[idx] = self._normalize(updated)

    @staticmethod
    def _normalize(vec: np.ndarray) -> np.ndarray:
        norm = np.linalg.norm(vec)
        if norm == 0:
            return vec.astype(np.float32)
        return (vec / norm).astype(np.float32)


def batch_cluster_two(embeddings: list[np.ndarray]) -> list[int]:
    """Assign N embeddings to cluster 0 or 1 (Speaker A / B)."""
    if not embeddings:
        return []
    if len(embeddings) == 1:
        return [0]

    embs = np.stack([TwoSpeakerClusterer._normalize(e) for e in embeddings])
    # Seed: first embedding + furthest from it
    sims = embs @ embs[0]
    seed_b = int(np.argmin(sims))
    centers = np.stack([embs[0], embs[seed_b]])

    for _ in range(5):
        labels = []
        for e in embs:
            labels.append(0 if float(centers[0] @ e) >= float(centers[1] @ e) else 1)
        labels_arr = np.array(labels)
        for k in (0, 1):
            members = embs[labels_arr == k]
            if len(members) > 0:
                c = members.mean(axis=0)
                norm = np.linalg.norm(c)
                if norm > 0:
                    centers[k] = c / norm

    return labels


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize/denoise.py
import sys
import threading
import types

import numpy as np
import torch

_DF_LOCK = threading.Lock()
_DF_MODEL_CACHE = None  # (model, df_state)

# ---- DeepFilterNet: giới hạn mức khử nhiễu tối đa ----
ATTEN_LIM_DB = 10.0

# ---- Envelope follower (theo dõi mức năng lượng) ----
FRAME_MS = 50            # kích thước khung để tính RMS

# ---- Mức mục tiêu chung cho toàn bộ file ----
TARGET_PERCENTILE = 50   

# ---- Ngưỡng phân biệt tiếng nói vs im lặng/nhiễu nền ----
NOISE_FLOOR_DB = -55.0  

# ---- Giới hạn gain 2 chiều ----
MAX_GAIN_DB = 24.0       
MIN_GAIN_DB = -10.0      

# ---- Làm mượt gain để tránh click/pumping ----
GAIN_SMOOTH_MS = 60      

# ---- Giới hạn biên độ cuối cùng để tránh clipping sau khi khuếch đại ----
OUTPUT_PEAK = 0.97


def _ensure_torchaudio_compat() -> None:
    """Provide torchaudio.backend.common shim for DeepFilterNet on torchaudio >= 2.1."""
    import torchaudio

    if not hasattr(torchaudio, "backend"):
        torchaudio.backend = types.ModuleType("torchaudio.backend")
        sys.modules["torchaudio.backend"] = torchaudio.backend
    if not hasattr(torchaudio.backend, "common"):
        torchaudio.backend.common = types.ModuleType("torchaudio.backend.common")
        sys.modules["torchaudio.backend.common"] = torchaudio.backend.common
        torchaudio.backend.common.AudioMetaData = getattr(
            torchaudio, "AudioMetaData", type("AudioMetaData", (), {})
        )


def _get_df_model():
    global _DF_MODEL_CACHE
    if _DF_MODEL_CACHE is not None:
        return _DF_MODEL_CACHE
    with _DF_LOCK:
        if _DF_MODEL_CACHE is None:
            import warnings

            _ensure_torchaudio_compat()
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore", message=".*AudioMetaData.*", category=UserWarning)
                from df.enhance import init_df

                model, df_state, _ = init_df()
            _DF_MODEL_CACHE = (model, df_state)
        return _DF_MODEL_CACHE


def denoise_with_deepfilternet(audio, sr):
    from df.enhance import enhance
    import torchaudio.transforms as T

    model, df_state = _get_df_model()

    wav = torch.from_numpy(audio.astype(np.float32)).unsqueeze(0)  # (1, samples)
    if sr != df_state.sr():
        wav = T.Resample(sr, df_state.sr())(wav)

    enhanced = enhance(model, df_state, wav, atten_lim_db=ATTEN_LIM_DB)

    if sr != df_state.sr():
        enhanced = T.Resample(df_state.sr(), sr)(enhanced)

    enhanced_arr = enhanced.detach().cpu().numpy().astype(np.float32)
    while enhanced_arr.ndim > 2:
        enhanced_arr = enhanced_arr[0]
    if enhanced_arr.ndim == 1:
        enhanced_np = enhanced_arr
    else:
        enhanced_np = enhanced_arr.mean(axis=0).astype(np.float32)

    return enhanced_np, sr


def db_to_lin(db):
    return 10.0 ** (db / 20.0)

def compute_rms_frames(audio, frame_len):
    n_frames = max(1, len(audio) // frame_len)
    trimmed = audio[: n_frames * frame_len]
    frames = trimmed.reshape(n_frames, frame_len)
    rms = np.sqrt(np.mean(frames ** 2, axis=1) + 1e-12)
    return rms, n_frames

def compute_target_level(rms, noise_floor_lin, percentile=TARGET_PERCENTILE):
    is_speech = rms > noise_floor_lin
    speech_rms = rms[is_speech]
    if len(speech_rms) == 0:
        return float(np.median(rms))
    return float(np.percentile(speech_rms, percentile))

def compute_gain_curve(rms, target, sr, frame_len):
    noise_floor_lin = db_to_lin(NOISE_FLOOR_DB)
    max_gain_lin = db_to_lin(MAX_GAIN_DB)
    min_gain_lin = db_to_lin(MIN_GAIN_DB)

    is_speech = rms > noise_floor_lin

    gain = np.ones_like(rms)
    desired_gain = target / (rms[is_speech] + 1e-9)
    gain[is_speech] = np.clip(desired_gain, min_gain_lin, max_gain_lin)

    frame_dur = frame_len / sr
    smooth_coef = np.exp(-frame_dur / (GAIN_SMOOTH_MS / 1000.0))
    smoothed = np.ones_like(gain)
    smoothed[0] = gain[0]
    for i in range(1, len(gain)):
        smoothed[i] = smooth_coef * smoothed[i - 1] + (1 - smooth_coef) * gain[i]

    return smoothed

def apply_gain_curve(audio, gain_per_frame, frame_len):
    n_frames = len(gain_per_frame)
    frame_centers = (np.arange(n_frames) + 0.5) * frame_len
    sample_positions = np.arange(len(audio))
    gain_per_sample = np.interp(
        sample_positions,
        frame_centers,
        gain_per_frame,
        left=gain_per_frame[0],
        right=gain_per_frame[-1],
    )
    return audio * gain_per_sample

def soft_limit(audio, peak=OUTPUT_PEAK):
    max_abs = np.max(np.abs(audio)) + 1e-9
    if max_abs > peak:
        audio = audio / max_abs * peak
    return audio

def level_audio_to_target(audio, sr):
    audio = np.asarray(audio, dtype=np.float32)
    while audio.ndim > 2:
        audio = audio[0]
    if audio.ndim == 2:
        audio = audio.mean(axis=0).astype(np.float32)

    frame_len = int(sr * FRAME_MS / 1000)
    rms, n_frames = compute_rms_frames(audio, frame_len)

    noise_floor_lin = db_to_lin(NOISE_FLOOR_DB)
    target = compute_target_level(rms, noise_floor_lin)

    gain_curve = compute_gain_curve(rms, target, sr, frame_len)
    leveled = apply_gain_curve(audio[: n_frames * frame_len], gain_curve, frame_len)

    remainder = audio[n_frames * frame_len:]
    if len(remainder) > 0:
        leveled = np.concatenate([leveled, remainder])

    return soft_limit(leveled.astype(np.float32))


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize/embedding.py
import os
import warnings

import numpy as np
import torch
import torchaudio

SAMPLE_RATE = 16000
HUB_MODEL_ID = "damo/speech_eres2net_large_200k_sv_zh-cn_16k-common"
DEFAULT_MODEL_ID = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "pretrained_models", "speech_eres2net_large_200k_sv_zh-cn_16k-common")

def resolve_eres2net_model(model_id: str | None = None) -> str:
    """
    Xác định đường dẫn checkpoint hợp lệ cho ERes2Net:
    1. Nếu model_id được truyền vào và thư mục chứa pretrained_eres2net.pt (> 10MB) -> Dùng model_id.
    2. Nếu thư mục mặc định trong speaker-diarize chứa pretrained_eres2net.pt (> 10MB) -> Dùng mặc định.
    3. Tìm kiếm trong /kaggle/working và /kaggle/input xem có thư mục chứa pretrained_eres2net.pt (> 10MB) -> Dùng thư mục đó.
    4. Nếu không có tại local, tự động tải/cache qua snapshot_download('damo/speech_eres2net_large_200k_sv_zh-cn_16k-common').
    """
    # 1. Kiểm tra model_id truyền vào
    if model_id and os.path.isdir(model_id):
        pt_path = os.path.join(model_id, "pretrained_eres2net.pt")
        cfg_path = os.path.join(model_id, "configuration.json")
        if os.path.isfile(pt_path) and os.path.getsize(pt_path) > 10_000_000 and os.path.isfile(cfg_path):
            return model_id

    # 2. Kiểm tra thư mục local mặc định
    if os.path.isdir(DEFAULT_MODEL_ID):
        pt_path = os.path.join(DEFAULT_MODEL_ID, "pretrained_eres2net.pt")
        cfg_path = os.path.join(DEFAULT_MODEL_ID, "configuration.json")
        if os.path.isfile(pt_path) and os.path.getsize(pt_path) > 10_000_000 and os.path.isfile(cfg_path):
            return DEFAULT_MODEL_ID

    # 3. Tim kiem trong Kaggle directories (/kaggle/working, /kaggle/input)
    for search_root in ["/kaggle/working", "/kaggle/input"]:
        if os.path.exists(search_root):
            for root, dirs, files in os.walk(search_root):
                if "pretrained_eres2net.pt" in files and "configuration.json" in files:
                    pt_path = os.path.join(root, "pretrained_eres2net.pt")
                    if os.path.getsize(pt_path) > 10_000_000:
                        print(f"[ERes2Net] Found valid checkpoint at: {root}")
                        return root

    # 4. Tai tu ModelScope Hub
    try:
        from modelscope.hub.snapshot_download import snapshot_download
        print(f"[ERes2Net] Local checkpoint missing or incomplete. Downloading {HUB_MODEL_ID} from ModelScope...")
        cache_dir = snapshot_download(HUB_MODEL_ID)
        print(f"[ERes2Net] Model downloaded successfully to: {cache_dir}")
        return cache_dir
    except Exception as e:
        print(f"[ERes2Net] Warning during download ({e}), using Hub ID directly: {HUB_MODEL_ID}")
        return HUB_MODEL_ID


def _compute_fbank(wav: np.ndarray, sample_rate: int = SAMPLE_RATE, n_mels: int = 80) -> torch.Tensor:
    """Tính log-Mel filterbank features trực tiếp trong RAM (không ghi file đĩa).
    
    ERes2Net sử dụng 80-dimensional fbank features tại 16kHz.
    Output shape: (1, n_mels, num_frames) — sẵn sàng cho model forward.
    """
    waveform = torch.from_numpy(wav).unsqueeze(0)  # (1, samples)
    
    # Tính Mel spectrogram giống cách ModelScope pipeline xử lý nội bộ
    fbank = torchaudio.compliance.kaldi.fbank(
        waveform,
        num_mel_bins=n_mels,
        sample_frequency=sample_rate,
        frame_length=25.0,
        frame_shift=10.0,
        window_type='povey',
        use_energy=False,
        dither=0.0,
    )  # (num_frames, n_mels)
    
    # Chuẩn hoá CMVN (Cepstral Mean and Variance Normalization)
    fbank = fbank - fbank.mean(dim=0, keepdim=True)
    
    return fbank.unsqueeze(0)  # (1, num_frames, n_mels)


class ERes2NetEmbedder:
    """Wrap ModelScope ERes2NetV2 for speaker embeddings.
    
    Hỗ trợ 3 chế độ inference:
    - embed_direct(): Truyền tensor trực tiếp vào model (NHANH NHẤT, không I/O đĩa)
    - embed_batch():  Gom nhiều audio windows thành 1 batch GPU forward (TỐI ƯU cho diarize)
    - embed():        Phương thức gốc qua ModelScope pipeline + file tạm (FALLBACK)
    """

    def __init__(self, device: str = "cuda", model_id: str | None = None) -> None:
        self.device = device if torch.cuda.is_available() else "cpu"
        resolved_model = resolve_eres2net_model(model_id)
        self.model_id = resolved_model
        from modelscope.pipelines import pipeline
        
        # Initialize pipeline
        name = resolved_model.replace("\\", "/").rstrip("/").split("/")[-1]
        print(f"[{name}] Loading ERes2Net model from '{resolved_model}'...")
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            self.sv_pipeline = pipeline(
                task='speaker-verification',
                model=resolved_model,
                device=self.device
            )
        print(f"[{name}] ERes2Net model loaded successfully on {self.device}!")

        # ── Trích xuất raw PyTorch model từ ModelScope pipeline để dùng embed_direct/embed_batch ──
        self._raw_model = None
        try:
            model_obj = getattr(self.sv_pipeline, 'model', None)
            if model_obj is not None:
                # SpeakerVerificationERes2Net wraps the actual nn.Module inside `embedding_model`
                inner = getattr(model_obj, 'embedding_model', None) or getattr(model_obj, 'model', None)
                if inner is not None and isinstance(inner, torch.nn.Module):
                    inner.to(self.device)
                    inner.eval()
                    self._raw_model = inner
                    print(f"[{name}] ✅ Raw PyTorch model extracted — embed_direct/embed_batch enabled (zero I/O)")
        except Exception as e:
            print(f"[{name}] ⚠️ Could not extract raw model ({e}), falling back to file-based embed()")

    @staticmethod
    def _to_wav(audio: np.ndarray | torch.Tensor) -> np.ndarray:
        """Chuyển đổi input thành mảng 1D float32."""
        if isinstance(audio, torch.Tensor):
            wav = audio.cpu().numpy().astype(np.float32)
        else:
            wav = audio.astype(np.float32)
        while wav.ndim > 1:
            wav = wav[0]
        return wav

    @staticmethod
    def _normalize_vec(vec: np.ndarray) -> np.ndarray:
        """L2-normalize embedding vector."""
        vec = np.squeeze(vec).astype(np.float32)
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec /= norm
        return vec

    @torch.inference_mode()
    def embed_direct(self, audio: np.ndarray | torch.Tensor, sample_rate: int = SAMPLE_RATE) -> np.ndarray:
        """Trích xuất speaker embedding KHÔNG qua file I/O.
        
        Tính fbank features trực tiếp trong RAM rồi forward qua raw model.
        Nhanh gấp ~10-20 lần so với embed() vì loại bỏ hoàn toàn disk I/O.
        Nếu raw model không khả dụng, tự động fallback về embed().
        """
        if self._raw_model is None:
            return self.embed(audio, sample_rate)

        wav = self._to_wav(audio)
        if len(wav) < int(0.2 * SAMPLE_RATE):
            raise ValueError("Audio too short for embedding (need >= 0.2s)")

        fbank = _compute_fbank(wav, sample_rate)  # (1, num_frames, n_mels)
        fbank = fbank.to(self.device)

        output = self._raw_model(fbank)
        
        # ERes2Net trả về embedding tensor — có thể là tuple hoặc tensor trực tiếp
        if isinstance(output, (tuple, list)):
            emb = output[0]
        elif isinstance(output, dict):
            emb = output.get('embs', output.get('embedding', list(output.values())[0]))
        else:
            emb = output
        
        if isinstance(emb, torch.Tensor):
            vec = emb.detach().cpu().numpy()
        else:
            vec = np.array(emb)
        
        return self._normalize_vec(vec)

    @torch.inference_mode()
    def embed_batch(self, audio_list: list[np.ndarray], sample_rate: int = SAMPLE_RATE) -> list[np.ndarray]:
        """Trích xuất embeddings cho NHIỀU audio windows cùng lúc (Batch GPU Inference).
        
        Gom tất cả windows thành 1 batch tensor, forward qua GPU 1 lần duy nhất.
        Hiệu quả gấp N lần so với gọi embed() N lần (giảm CUDA kernel launch overhead).
        
        Args:
            audio_list: Danh sách các audio numpy arrays (mỗi cái là 1 window).
            sample_rate: Sample rate (mặc định 16kHz).
        
        Returns:
            Danh sách các L2-normalized embedding vectors (512-D mỗi cái).
        """
        if not audio_list:
            return []
        
        # Nếu raw model không khả dụng, fallback gọi từng cái
        if self._raw_model is None:
            return [self.embed(a, sample_rate) for a in audio_list]

        # Tính fbank cho tất cả windows
        fbanks = []
        valid_indices = []
        for i, audio in enumerate(audio_list):
            wav = self._to_wav(audio)
            if len(wav) < int(0.2 * SAMPLE_RATE):
                continue
            fb = _compute_fbank(wav, sample_rate)  # (1, num_frames, n_mels)
            fbanks.append(fb)
            valid_indices.append(i)
        
        if not fbanks:
            return []

        # Chia thành các mini-batch để tránh tràn VRAM với file audio rất dài
        n_mels = fbanks[0].shape[2]
        BATCH_SIZE = 32
        all_embeddings = [None] * len(audio_list)
        
        for batch_start in range(0, len(fbanks), BATCH_SIZE):
            batch_fbanks = fbanks[batch_start:batch_start + BATCH_SIZE]
            batch_indices = valid_indices[batch_start:batch_start + BATCH_SIZE]
            
            batch_max_frames = max(fb.shape[1] for fb in batch_fbanks)
            padded = torch.zeros(len(batch_fbanks), batch_max_frames, n_mels)
            for j, fb in enumerate(batch_fbanks):
                padded[j, :fb.shape[1], :] = fb.squeeze(0)
            
            padded = padded.to(self.device)
            output = self._raw_model(padded)
            
            # Trích xuất embeddings từ output
            if isinstance(output, (tuple, list)):
                embs_tensor = output[0]
            elif isinstance(output, dict):
                embs_tensor = output.get('embs', output.get('embedding', list(output.values())[0]))
            else:
                embs_tensor = output
            
            if isinstance(embs_tensor, torch.Tensor):
                embs_np = embs_tensor.detach().cpu().numpy()
            else:
                embs_np = np.array(embs_tensor)
            
            # Gán kết quả vào đúng vị trí
            if embs_np.ndim == 1:
                # Single embedding returned for entire batch — fallback to sequential
                for idx in batch_indices:
                    all_embeddings[idx] = self.embed_direct(audio_list[idx], sample_rate)
            else:
                for j, idx in enumerate(batch_indices):
                    all_embeddings[idx] = self._normalize_vec(embs_np[j])
        
        # Lọc bỏ None (windows quá ngắn)
        return [e for e in all_embeddings if e is not None]

    def embed(self, audio: np.ndarray | torch.Tensor, sample_rate: int = SAMPLE_RATE) -> np.ndarray:
        """Phương thức gốc: trích xuất embedding qua ModelScope pipeline + file tạm.
        
        Chậm hơn embed_direct() do phải ghi/đọc file đĩa, nhưng đảm bảo tương thích 100%
        với mọi phiên bản ModelScope. Dùng làm fallback khi embed_direct() không khả dụng.
        """
        wav = self._to_wav(audio)
        if len(wav) < int(0.2 * SAMPLE_RATE):
            raise ValueError("Audio too short for embedding (need >= 0.2s)")

        # Thử embed_direct trước, fallback về file I/O nếu thất bại
        if self._raw_model is not None:
            try:
                return self.embed_direct(audio, sample_rate)
            except Exception:
                pass

        import soundfile as sf
        import tempfile
        
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
            temp_path = f.name
        try:
            sf.write(temp_path, wav, sample_rate)
            res = self.sv_pipeline([temp_path], output_emb=True)
            if isinstance(res, dict) and 'embs' in res:
                vec = res['embs']
            elif isinstance(res, dict) and 'text' in res:
                # Fallback if the pipeline returns text dict
                vec = res.get('embs', res)
            else:
                vec = res
                
            if isinstance(vec, list):
                vec = np.array(vec)
            return self._normalize_vec(vec)
        finally:
            if os.path.exists(temp_path):
                try:
                    os.remove(temp_path)
                except:
                    pass


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize/pipeline.py
"""Diarization pipeline using ERes2Net-Large.

Tối ưu hiệu năng:
- Batch GPU inference cho cả clustering pass và boundary refinement pass
- Sử dụng embed_direct() / embed_batch() thay vì file I/O
- Giảm số lần gọi model đáng kể nhờ gom batch
"""

from __future__ import annotations

import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np

from speaker_diarize.audio_io import load_audio, save_audio
from speaker_diarize.clustering import TwoSpeakerClusterer, batch_cluster_two
from speaker_diarize.embedding import ERes2NetEmbedder
from speaker_diarize.segmentation import RmsVad, SlidingWindowBuffer, SpeechWindow

LABELS = ("Speaker A", "Speaker B")
ROLE_TEACHER = "Teacher"
ROLE_STUDENT = "Student"


@dataclass
class DiarizationSegment:
    start: float
    end: float
    speaker: str
    confidence: float
    teacher_score: float | None = None
    student_score: float | None = None


@dataclass
class SplitResult:
    segments: list[DiarizationSegment]
    duration_sec: float
    teacher_cluster: int | None = None
    teacher_segments: list[DiarizationSegment] | None = None
    student_segments: list[DiarizationSegment] | None = None
    teacher_path: Path | None = None
    student_path: Path | None = None


class TwoSpeakerSplitter:
    """Diarize a 2-speaker recording and export separate audio tracks."""

    def __init__(
        self,
        device: str = "cpu",
        embedder: ERes2NetEmbedder | None = None,
        vad: RmsVad | None = None,
        vad_threshold_db: float = -42.0,
        cluster_window_sec: float = 1.5,
        boundary_window_sec: float = 0.5,
        min_speech_sec: float = 0.25,
        min_segment_sec: float = 0.3,
        merge_gap_sec: float = 0.5,
        consecutive_merge_gap_sec: float = 2.5,
        step_sec: float = 0.5,
        boundary_step_sec: float = 0.1,
        min_similarity_threshold: float = 0.40,
    ) -> None:
        self.embedder = embedder or ERes2NetEmbedder(device=device)
        self.vad = vad or RmsVad(threshold_db=vad_threshold_db)
        self.min_segment_sec = min_segment_sec
        self.merge_gap_sec = merge_gap_sec
        self.consecutive_merge_gap_sec = consecutive_merge_gap_sec
        self.min_similarity_threshold = min_similarity_threshold
        self.buffer = SlidingWindowBuffer(window_sec=cluster_window_sec, step_sec=step_sec)
        self.boundary_buffer = SlidingWindowBuffer(
            window_sec=boundary_window_sec, 
            step_sec=boundary_step_sec, 
            min_speech_sec=min_speech_sec
        )

    def split_file(
        self,
        input_path: str | Path,
        output_dir: str | Path | None = None,
        *,
        teacher_reference_path: str | Path | None = None,
        teacher_embedding: np.ndarray | None = None,
        student_embedding: np.ndarray | None = None,
        apply_denoise: bool = True,
    ) -> SplitResult:
        input_path = Path(input_path)

        teacher_emb = teacher_embedding
        if teacher_emb is None and teacher_reference_path:
            teacher_emb = self._embed_reference(teacher_reference_path)
        elif teacher_emb is not None:
            pass

        audio, sr = load_audio(input_path)

        diarize_audio = audio
        if apply_denoise:
            from speaker_diarize.denoise import denoise_with_deepfilternet, level_audio_to_target
            print("[Info] Applying DeepFilterNet denoise and leveling for accurate VAD segmentation...")
            diarize_audio, _ = denoise_with_deepfilternet(audio, sr)
            diarize_audio = level_audio_to_target(diarize_audio, sr)

        segments, teacher_cluster = self._diarize(
            diarize_audio, sr, teacher_emb=teacher_emb, student_emb=student_embedding
        )

        teacher_segments = [s for s in segments if s.speaker == ROLE_TEACHER] if teacher_cluster is not None else None
        student_segments = [s for s in segments if s.speaker == ROLE_STUDENT] if teacher_cluster is not None else None

        teacher_path = student_path = None
        if teacher_segments is not None and student_segments is not None:
            teacher_track = np.zeros_like(audio)
            student_track = np.zeros_like(audio)
            for s in teacher_segments:
                teacher_track[int(s.start * sr):int(s.end * sr)] = audio[int(s.start * sr):int(s.end * sr)]
            for s in student_segments:
                student_track[int(s.start * sr):int(s.end * sr)] = audio[int(s.start * sr):int(s.end * sr)]
            
            if output_dir is not None:
                out_dir = Path(output_dir)
            else:
                out_dir = input_path.parent / f"{input_path.stem}_split"
            out_dir.mkdir(parents=True, exist_ok=True)
            teacher_path = out_dir / f"{input_path.stem}_teacher.wav"
            student_path = out_dir / f"{input_path.stem}_student.wav"
            csv_path = out_dir / f"{input_path.stem}_cosine_scores.csv"
            
            save_audio(teacher_path, teacher_track, sr)
            save_audio(student_path, student_track, sr)
            
            # Lưu file CSV hiển thị điểm cosine
            with open(csv_path, "w", encoding="utf-8") as f:
                f.write("Start,End,Assigned_Speaker,Teacher_Score,Student_Score\n")
                for s in segments:
                    ts = f"{s.teacher_score:.4f}" if s.teacher_score is not None else "N/A"
                    ss = f"{s.student_score:.4f}" if s.student_score is not None else "N/A"
                    f.write(f"{s.start:.2f},{s.end:.2f},{s.speaker},{ts},{ss}\n")

        return SplitResult(
            segments=segments,
            duration_sec=len(audio) / sr,
            teacher_cluster=teacher_cluster,
            teacher_segments=teacher_segments,
            student_segments=student_segments,
            teacher_path=teacher_path,
            student_path=student_path,
        )

    def _embed_reference(self, reference_path: str | Path, apply_denoise: bool = True) -> np.ndarray:
        """Trích xuất embedding tham chiếu cho 1 giọng nói (dùng embed_batch cho tốc độ)."""
        audio, sr = load_audio(reference_path)
        
        if apply_denoise:
            from speaker_diarize.denoise import denoise_with_deepfilternet, level_audio_to_target
            audio, _ = denoise_with_deepfilternet(audio, sr)
            audio = level_audio_to_target(audio, sr)
            
        # Lọc windows có giọng nói
        speech_windows = [w for w in self.buffer.iter_windows(audio) if self.vad.is_speech(w.audio)]

        if speech_windows:
            # Batch inference: gom tất cả windows → 1 lần GPU forward
            audio_list = [w.audio for w in speech_windows]
            embs = self.embedder.embed_batch(audio_list)
        
        if speech_windows and embs:
            vec = np.mean(embs, axis=0).astype(np.float32)
        else:
            vec = self.embedder.embed_direct(audio)
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec /= norm
        return vec

    def _diarize(
        self,
        audio: np.ndarray,
        sample_rate: int,
        teacher_emb: np.ndarray | None = None,
        student_emb: np.ndarray | None = None,
    ) -> tuple[list[DiarizationSegment], int | None]:
        t0 = time.time()

        # ════════════════════════════════════════════════════════════════════
        # Pass 1: Clustering — Gom speech windows → Batch GPU Inference
        # ════════════════════════════════════════════════════════════════════
        all_cluster_windows = self.buffer.iter_windows(audio)
        speech_cluster_windows = [w for w in all_cluster_windows if self.vad.is_speech(w.audio)]

        if not speech_cluster_windows:
            raise ValueError("Không phát hiện giọng nói trong file audio.")

        # Batch inference: gom tất cả speech windows → 1 lần GPU forward
        cluster_audio_list = [w.audio for w in speech_cluster_windows]
        cluster_embeddings = self.embedder.embed_batch(cluster_audio_list)

        # Loại bỏ windows mà embedding extraction thất bại (quá ngắn)
        windows: list[SpeechWindow] = []
        embeddings: list[np.ndarray] = []
        for w, emb in zip(speech_cluster_windows, cluster_embeddings):
            if emb is not None:
                windows.append(w)
                embeddings.append(emb)

        if not windows:
            raise ValueError("Không phát hiện giọng nói trong file audio.")

        t1 = time.time()
        print(f"[Diarize] Pass 1 Clustering: {len(windows)} windows embedded in {t1 - t0:.2f}s (batch)")

        teacher_cluster: int | None = None
        if teacher_emb is not None and student_emb is not None:
            # Dual reference mode: directly use reference embeddings as centroids
            teacher_cluster = 0
            centers = [teacher_emb, student_emb]
            print("\n[Diarization: Dual Reference Mode]")
            print(f"- Giáo viên (Teacher) reference embedding loaded (norm={np.linalg.norm(teacher_emb):.2f})")
            print(f"- Học sinh (Student)  reference embedding loaded (norm={np.linalg.norm(student_emb):.2f})")
        else:
            labels = batch_cluster_two(embeddings)
            centers = self._cluster_centers(embeddings, labels)
            n_clusters = len(set(labels))

            if n_clusters == 1:
                teacher_cluster = 0 if (teacher_emb is not None or student_emb is not None) else None
            elif teacher_emb is not None or student_emb is not None:
                sim_t0 = float(np.dot(centers[0], teacher_emb)) if teacher_emb is not None else 0.0
                sim_t1 = float(np.dot(centers[1], teacher_emb)) if teacher_emb is not None else 0.0
                sim_s0 = float(np.dot(centers[0], student_emb)) if student_emb is not None else 0.0
                sim_s1 = float(np.dot(centers[1], student_emb)) if student_emb is not None else 0.0

                score_a = sim_t0 + sim_s1
                score_b = sim_t1 + sim_s0

                teacher_cluster = 0 if score_a >= score_b else 1
                print(f"\n[Mapping Similarity]")
                print(f"- Giao vien (Teacher) so voi Cluster 0: {sim_t0:.3f}, Cluster 1: {sim_t1:.3f}")
                print(f"- Hoc sinh (Student)  so voi Cluster 0: {sim_s0:.3f}, Cluster 1: {sim_s1:.3f}")
                print(f"=> Quyet dinh: Gan Giao vien = Cluster {teacher_cluster}, Hoc sinh = Cluster {1 - teacher_cluster}")
            else:
                teacher_cluster = None

        # ════════════════════════════════════════════════════════════════════
        # Pass 2: Boundary Refinement — Batch GPU Inference
        # ════════════════════════════════════════════════════════════════════
        t2 = time.time()
        all_boundary_windows = self.boundary_buffer.iter_windows(audio)
        speech_boundary_windows = [w for w in all_boundary_windows if self.vad.is_speech(w.audio, min_ratio=0.1)]

        n = len(audio)
        votes_a = np.zeros(n, dtype=np.float32)
        votes_b = np.zeros(n, dtype=np.float32)

        if speech_boundary_windows:
            # Batch inference: gom tất cả boundary windows → 1 lần GPU forward
            boundary_audio_list = [w.audio for w in speech_boundary_windows]
            boundary_embeddings = self.embedder.embed_batch(boundary_audio_list)

            for w, emb in zip(speech_boundary_windows, boundary_embeddings):
                if emb is None:
                    continue
                conf_a = float(np.clip(np.dot(emb, centers[0]), 0.0, 1.0))
                conf_b = float(np.clip(np.dot(emb, centers[1]), 0.0, 1.0))
                
                label = 0 if conf_a >= conf_b else 1
                
                s, e = w.start_sample, w.end_sample
                if label == 0:
                    votes_a[s:e] += conf_a
                else:
                    votes_b[s:e] += conf_b

        t3 = time.time()
        print(f"[Diarize] Pass 2 Boundary: {len(speech_boundary_windows)} windows embedded in {t3 - t2:.2f}s (batch)")

        stamps = self.vad.get_timestamps(audio, sample_rate=sample_rate)
        is_speech = np.zeros(n, dtype=bool)
        pad_samples = int(0.2 * sample_rate)
        for stamp in stamps:
            s_idx = max(0, stamp["start"] - pad_samples)
            e_idx = min(n, stamp["end"] + pad_samples)
            is_speech[s_idx:e_idx] = True
            
        pred_a = (votes_a >= votes_b) & (votes_a > 0) & is_speech
        pred_b = (votes_b > votes_a) & is_speech
        
        def get_blocks(mask):
            mask_int = np.concatenate(([0], mask.astype(int), [0]))
            diff = np.diff(mask_int)
            starts = np.where(diff == 1)[0]
            ends = np.where(diff == -1)[0]
            return starts, ends
            
        starts_a, ends_a = get_blocks(pred_a)
        starts_b, ends_b = get_blocks(pred_b)
        
        raw_segments = []
        for s, e in zip(starts_a, ends_a):
            raw_segments.append({
                "start": float(s) / sample_rate,
                "end": float(e) / sample_rate,
                "cluster": 0
            })
        for s, e in zip(starts_b, ends_b):
            raw_segments.append({
                "start": float(s) / sample_rate,
                "end": float(e) / sample_rate,
                "cluster": 1
            })
            
        raw_segments.sort(key=lambda x: x["start"])
        raw_segments = [s for s in raw_segments if (s["end"] - s["start"]) >= self.min_segment_sec]
        
        merged_segments = []
        for seg in raw_segments:
            if not merged_segments:
                merged_segments.append(seg)
                continue
                
            last = merged_segments[-1]
            if last["cluster"] == seg["cluster"] and (seg["start"] - last["end"] < self.merge_gap_sec):
                last["end"] = seg["end"]
            else:
                merged_segments.append(seg)

        # ════════════════════════════════════════════════════════════════════
        # Pass 3: 2nd-pass Refinement — Batch GPU Inference per segment
        # ════════════════════════════════════════════════════════════════════
        t4 = time.time()
        final_segments = []

        if teacher_emb is not None and student_emb is not None:
            # Gom tất cả segment chunks cần refinement → batch embed
            seg_chunks = []
            seg_chunk_window_map = []  # (seg_index, list_of_window_indices)

            for i, s in enumerate(merged_segments):
                chunk = audio[int(s["start"]*sample_rate) : int(s["end"]*sample_rate)]
                chunk_windows = [w for w in self.buffer.iter_windows(chunk) if self.vad.is_speech(w.audio)]
                
                if chunk_windows:
                    start_idx = len(seg_chunks)
                    for w in chunk_windows:
                        seg_chunks.append(w.audio)
                    seg_chunk_window_map.append((i, list(range(start_idx, len(seg_chunks)))))
                else:
                    # Không có speech windows → embed toàn bộ chunk
                    start_idx = len(seg_chunks)
                    seg_chunks.append(chunk)
                    seg_chunk_window_map.append((i, [start_idx]))

            # Batch inference cho tất cả chunks 1 lần
            all_chunk_embs = self.embedder.embed_batch(seg_chunks) if seg_chunks else []

            for seg_idx, window_indices in seg_chunk_window_map:
                s = merged_segments[seg_idx]
                t_score = None
                s_score = None

                try:
                    chunk_embs = [all_chunk_embs[j] for j in window_indices if j < len(all_chunk_embs) and all_chunk_embs[j] is not None]
                    
                    if chunk_embs:
                        seg_emb = np.mean(chunk_embs, axis=0)
                    else:
                        chunk = audio[int(s["start"]*sample_rate) : int(s["end"]*sample_rate)]
                        seg_emb = self.embedder.embed_direct(chunk)
                        
                    norm = np.linalg.norm(seg_emb)
                    if norm > 0: seg_emb /= norm
                    
                    t_score = float(np.dot(seg_emb, teacher_emb))
                    s_score = float(np.dot(seg_emb, student_emb))
                    
                    # Quyết định role dựa trên so sánh độ tương đồng giọng thực tế
                    role = ROLE_TEACHER if t_score >= s_score else ROLE_STUDENT
                    
                    # Lọc bỏ đoạn nhiễu/tạp âm không khớp với cả 2 người nói (< 0.40)
                    if max(t_score, s_score) < self.min_similarity_threshold:
                        continue
                        
                except (ValueError, IndexError):
                    role = ROLE_TEACHER if s["cluster"] == teacher_cluster else ROLE_STUDENT

                final_segments.append(DiarizationSegment(
                    start=s["start"], end=s["end"], speaker=role, confidence=1.0,
                    teacher_score=t_score, student_score=s_score
                ))
        else:
            # Không có dual reference → gán nhãn theo cluster trực tiếp (không cần embed thêm)
            for s in merged_segments:
                if teacher_cluster is not None:
                    role = ROLE_TEACHER if s["cluster"] == teacher_cluster else ROLE_STUDENT
                else:
                    role = LABELS[s["cluster"]]
                    
                final_segments.append(DiarizationSegment(
                    start=s["start"], end=s["end"], speaker=role, confidence=1.0,
                    teacher_score=None, student_score=None
                ))

        t5 = time.time()
        print(f"[Diarize] Pass 3 Refinement: {len(merged_segments)} segments refined in {t5 - t4:.2f}s (batch)")

        # Hậu xử lý: Gộp các lượt nói liền kề của cùng 1 người nếu người kia không nói xen vào
        merged_roles = []
        for seg in final_segments:
            if not merged_roles:
                merged_roles.append(seg)
                continue
            last = merged_roles[-1]
            gap = seg.start - last.end
            if last.speaker == seg.speaker and (gap <= self.consecutive_merge_gap_sec):
                d1 = max(0.01, last.end - last.start)
                d2 = max(0.01, seg.end - seg.start)
                last.end = seg.end
                if last.teacher_score is not None and seg.teacher_score is not None:
                    last.teacher_score = round((last.teacher_score * d1 + seg.teacher_score * d2) / (d1 + d2), 4)
                if last.student_score is not None and seg.student_score is not None:
                    last.student_score = round((last.student_score * d1 + seg.student_score * d2) / (d1 + d2), 4)
            else:
                merged_roles.append(seg)
        final_segments = merged_roles

        total_time = time.time() - t0
        print(f"[Diarize] ✅ Total diarization completed in {total_time:.2f}s ({len(final_segments)} segments)")
        return final_segments, teacher_cluster

    @staticmethod
    def _cluster_centers(embeddings: list[np.ndarray], labels: list[int]) -> list[np.ndarray]:
        centers = []
        for k in (0, 1):
            members = [embeddings[i] for i, lb in enumerate(labels) if lb == k]
            if not members:
                centers.append(TwoSpeakerClusterer._normalize(embeddings[0]))
            else:
                c = np.mean(members, axis=0)
                centers.append(TwoSpeakerClusterer._normalize(c))
        return centers


In [ ]:
import os; os.makedirs('/kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/SpeakAI-Eval/speaker-diarize/speaker_diarize/segmentation.py
"""Voice activity detection and speech window extraction."""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np

NOISE_FLOOR_DB = -42.0

SAMPLE_RATE = 16000


@dataclass
class SpeechWindow:
    audio: np.ndarray
    start_sample: int
    end_sample: int

    @property
    def start_sec(self) -> float:
        return self.start_sample / SAMPLE_RATE

    @property
    def end_sec(self) -> float:
        return self.end_sample / SAMPLE_RATE


class RmsVad:
    """VAD based on RMS noise floor."""

    def __init__(self, threshold_db: float = NOISE_FLOOR_DB) -> None:
        self.threshold_lin = 10.0 ** (threshold_db / 20.0)

    def is_speech(self, audio: np.ndarray, sample_rate: int = SAMPLE_RATE, min_ratio: float = 0.3) -> bool:
        if len(audio) == 0:
            return False
        frame_len = int(sample_rate * 0.05)
        if len(audio) < frame_len:
            rms = np.sqrt(np.mean(audio ** 2) + 1e-12)
            return bool(rms > self.threshold_lin)
            
        n_frames = len(audio) // frame_len
        trimmed = audio[: n_frames * frame_len]
        frames = trimmed.reshape(n_frames, frame_len)
        rms = np.sqrt(np.mean(frames ** 2, axis=1) + 1e-12)
        speech_frames = np.sum(rms > self.threshold_lin)
        return (speech_frames / n_frames) >= min_ratio

    def get_timestamps(self, audio: np.ndarray, sample_rate: int = SAMPLE_RATE) -> list[dict]:
        if len(audio) == 0:
            return []
        frame_len = int(sample_rate * 0.05)
        n_frames = max(1, len(audio) // frame_len)
        trimmed = audio[: n_frames * frame_len]
        frames = trimmed.reshape(n_frames, frame_len)
        rms = np.sqrt(np.mean(frames ** 2, axis=1) + 1e-12)
        
        is_speech = rms > self.threshold_lin
        
        stamps = []
        in_speech = False
        start_frame = 0
        for i, speech in enumerate(is_speech):
            if speech and not in_speech:
                in_speech = True
                start_frame = i
            elif not speech and in_speech:
                in_speech = False
                stamps.append({
                    "start": start_frame * frame_len,
                    "end": i * frame_len
                })
        if in_speech:
            stamps.append({
                "start": start_frame * frame_len,
                "end": len(audio)
            })
        return stamps


class SlidingWindowBuffer:
    """Accumulate mic chunks and emit overlapping speech windows."""

    def __init__(
        self,
        window_sec: float = 1.5,
        step_sec: float = 0.5,
        min_speech_sec: float = 0.5,
        sample_rate: int = SAMPLE_RATE,
    ) -> None:
        self.window_samples = int(window_sec * sample_rate)
        self.step_samples = int(step_sec * sample_rate)
        self.min_speech_samples = int(min_speech_sec * sample_rate)
        self._buffer = np.array([], dtype=np.float32)
        self._total_samples = 0
        self._next_emit = 0

    def push(self, chunk: np.ndarray) -> list[SpeechWindow]:
        if chunk.size == 0:
            return []
        self._buffer = np.concatenate([self._buffer, chunk.astype(np.float32)])
        windows: list[SpeechWindow] = []

        while self._total_samples + len(self._buffer) - self._next_emit >= self.window_samples:
            start = self._next_emit
            end = start + self.window_samples
            if end > self._total_samples + len(self._buffer):
                break
            rel_start = start - self._total_samples
            audio = self._buffer[rel_start : rel_start + self.window_samples].copy()
            windows.append(SpeechWindow(audio=audio, start_sample=start, end_sample=end))
            self._next_emit += self.step_samples

        consumed = self._next_emit - self._total_samples
        if consumed > 0:
            self._buffer = self._buffer[consumed:]
            self._total_samples = self._next_emit

        return [w for w in windows if len(w.audio) >= self.min_speech_samples]

    def iter_windows(self, audio: np.ndarray) -> list[SpeechWindow]:
        """Generate sliding windows over a full waveform."""
        windows: list[SpeechWindow] = []
        offset = 0
        while offset + self.window_samples <= len(audio):
            chunk = audio[offset : offset + self.window_samples]
            windows.append(
                SpeechWindow(
                    audio=chunk.copy(),
                    start_sample=offset,
                    end_sample=offset + self.window_samples,
                )
            )
            offset += self.step_samples
        return [w for w in windows if len(w.audio) >= self.min_speech_samples]


---
### Tải model Speaker Diarization (ERes2Net-Large)

In [ ]:
import subprocess, sys, os
try:
    import numpy
    np_ver = numpy.__version__
except: np_ver = '2.0.2'
print('Installing Rust (required for DeepFilterNet on Python 3.12)...')
os.system("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y")
os.environ['PATH'] = f"/root/.cargo/bin:{os.environ.get('PATH', '')}"
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'addict', 'modelscope', 'deepfilternet'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', '-q', f'numpy=={np_ver}'], check=True)
import os, re, shutil
from modelscope.hub.snapshot_download import snapshot_download

TARGET_DIR = '/kaggle/working/SpeakAI-Eval'
ERES_MODEL_DIR = f'{TARGET_DIR}/speaker-diarize/pretrained_models/speech_eres2net_large_200k_sv_zh-cn_16k-common'
pt_file = os.path.join(ERES_MODEL_DIR, 'pretrained_eres2net.pt')

if not os.path.exists(pt_file) or os.path.getsize(pt_file) < 10000000:
    print('Downloading ERes2Net-Large via ModelScope snapshot_download...')
    os.makedirs(ERES_MODEL_DIR, exist_ok=True)
    snapshot_download('damo/speech_eres2net_large_200k_sv_zh-cn_16k-common', local_dir=ERES_MODEL_DIR)
    print('ERes2Net-Large download done!')
else:
    print('ERes2Net-Large model already exists and is valid.')

print('ALL SETUP COMPLETED!')

---
### Tải Qwen3-ASR-1.7B, WavLM, Lang-ID & DeepFilterNet vào Kaggle

In [ ]:
import os
import torch
import shutil
from huggingface_hub import snapshot_download
os.environ['HF_HUB_CACHE'] = '/tmp/hf_cache'

TARGET_DIR = '/kaggle/working/SpeakAI-Eval'
QWEN3_DIR = f'{TARGET_DIR}/pretrained_models/Qwen3-ASR-1.7B-hf'
WAVLM_DIR = f'{TARGET_DIR}/pretrained_models/wavlm-large'
LANG_ID_DIR = f'{TARGET_DIR}/pretrained_models/lang-id-voxlingua107-ecapa'

def download_model(repo_id, local_dir, ignore_patterns):
    import inspect
    if not os.path.exists(local_dir):
        dl_kw = {'repo_id': repo_id, 'local_dir': local_dir, 'ignore_patterns': ignore_patterns}
        if 'local_dir_use_symlinks' in inspect.signature(snapshot_download).parameters:
            dl_kw['local_dir_use_symlinks'] = False
        snapshot_download(**dl_kw)
        shutil.rmtree('/tmp/hf_cache', ignore_errors=True)
        shutil.rmtree('/root/.cache/huggingface', ignore_errors=True)
    else:
        print(f'{repo_id} already exists.')

print('Downloading DeepFilterNet model (offline caching)...')
import torchaudio
import sys, types
if not hasattr(torchaudio, 'backend'):
    torchaudio.backend = types.ModuleType('torchaudio.backend')
    sys.modules['torchaudio.backend'] = torchaudio.backend
if not hasattr(torchaudio.backend, 'common'):
    torchaudio.backend.common = types.ModuleType('torchaudio.backend.common')
    sys.modules['torchaudio.backend.common'] = torchaudio.backend.common
    torchaudio.backend.common.AudioMetaData = getattr(torchaudio, 'AudioMetaData', type('AudioMetaData', (), {}))
from df.enhance import init_df
init_df()
shutil.rmtree('/tmp/hf_cache', ignore_errors=True)
shutil.rmtree('/root/.cache/huggingface', ignore_errors=True)

print('Downloading Qwen3-ASR-1.7B model (Qwen/Qwen3-ASR-1.7B-hf, offline caching)...')
download_model('Qwen/Qwen3-ASR-1.7B-hf', QWEN3_DIR, ['*.h5', '*.ot', '*.msgpack', '*.onnx', '*.mlmodel', '*coreml*', '*.tflite', '*.pb', '*.ckpt'])

print('Downloading WavLM model (offline caching)...')
download_model('microsoft/wavlm-large', WAVLM_DIR, ['*.h5', '*.ot', '*.msgpack', '*.onnx', '*.mlmodel', '*coreml*', '*.tflite', '*.pb', '*.ckpt'])

print('Downloading Lang-ID model (offline caching)...')
download_model('speechbrain/lang-id-voxlingua107-ecapa', LANG_ID_DIR, ['*.h5', '*.ot', '*.msgpack', '*.onnx', '*.mlmodel', '*coreml*', '*.tflite', '*.pb'])

print('Offline models ready!')
